# Simulation figures: s01, s02, s03, s031, s032

Roadrunner v7 (`halo685x095_v7`, `vanilla/phase`; `RUN_DIR` in the first cell), host 1509. All data is computed **in memory on jotun**; nothing is written but the PNGs you choose to save.

**Running it**
- on jotun: `cd ~/roadrunner/notebooks && ~/miniconda3/envs/lv-jeans/bin/jupyter lab --no-browser --port 8888`
- on your laptop: `ssh -N -L 8888:localhost:8888 asier@jotun.astro.lu.se`, then open the URL jupyter printed

**Cells**
- `# Loading`, `# Processing`: Claude's. Their logic lives in `replay/simdata.py`; autoreload picks up changes to it without a restart, so the data in memory survives.
- `# Plotting`: yours. Re-running one only redraws (seconds), and the image shown is exactly the PNG that would be saved.
- Processing cells compute once per kernel and return at once afterwards (`REDO = True` recomputes). Restarting the kernel means computing again.
- `SAVE = True` (first cell): plotting cells also write their PNG, under the scripts' file names, to `outputs/figures/sim/` next to the notebooks.
- `FRET` (first cell): how a satellite's f_ret is defined, for every figure with satellites (s032, Comp01, Comp01b): `'list'` among the stars in its infall list that it binds (the stars roadrunner counts as its M*), `'owner'` among the stars it owns (the last galaxy to claim each star) and binds. Both are computed together, so switching is instant; re-run the processing and plotting cells after a change.

In [ ]:
# Loading

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from types import SimpleNamespace

# the project root on the path (the kernel starts in notebooks/)
HERE = Path.cwd().resolve()            # replay/ (the notebooks' module) sits next to this notebook
if HERE.name != 'notebooks' and (HERE / 'notebooks' / 'replay').is_dir():
    HERE = HERE / 'notebooks'
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize, TwoSlopeNorm, to_rgba
from matplotlib.patches import Circle
import cmasher  # noqa: F401   (registers the cmr.* colormaps)
import cmyt     # noqa: F401   (registers the cmyt.* colormaps)

from replay import plotstyle as S    # rcParams, ink colours, colourbar helper
from replay import simmap as SM      # map colours, colourbar ends, titles, footnotes
from replay import simdata as SD     # loading and processing of every figure


SAVE = False              # True: every plotting cell also writes its figure to outputs/figures/sim (on jotun) ...
SAVE_FORMATS = ('pdf',)   # ... in these formats, e.g. ('pdf',), ('png',) or ('pdf', 'png') ...
SAVE_DPI = 700            # ... at this resolution (pdf: of its rasterized parts, e.g. the map, background orbits)
SD.SAVE_FORMATS, SD.SAVE_DPI = SAVE_FORMATS, SAVE_DPI     # (the image shown in the notebook stays a 300 dpi PNG)

FRET = 'list'             # a satellite's f_ret = 1 - M(taken by the host) / M(its stars), its stars being:
                          #   'list':  those in its infall list that it binds (roadrunner's own M*: catalogue Mtot)
                          #   'owner': those it owns (the last galaxy to claim each star) and binds
                          # (s032, Comp01, Comp01b; after a change re-run this cell, then processing and plotting cells)

RUN_DIR = '/home/asier/roadrunner_test/halo685x095_v7'   # the roadrunner run: its catalogue in vanilla/phase, its
                                                          # particles in particle_data (v6: halo685x095_v6). Runs other
                                                          # than v6 add their name to the saved files, e.g. _halo685x095_v7

if 'R' not in globals() or R.rr_out != f'{RUN_DIR}/vanilla/phase':   # re-running this cell keeps the data in memory
    R = SD.Run(rr_out=f'{RUN_DIR}/vanilla/phase', particle_dir=f'{RUN_DIR}/particle_data', workers=42)
R.fret = FRET

In [ ]:
# Loading

R.load()    # merger tree, host, and the ownership replay to snapshot 177 (shared by everything below)

print(f'host {R.host}  |  snapshots {R.first}-{R.last}  |  run tag: {R.tag or "(none)"}')

## s01 / s02: f_ret or log10(E_H/E_S) at one snapshot

Square map (s01) and polar map (s02) of the host-bound stars, E_S from the galaxy each star should belong to (its owner).

In [ ]:
# Processing

REDO = False

# star tables kept in memory (any other snapshot loads on demand, ~1 min)
SNAPSHOTS = [68, 77, 87, 146]

_ = R.starmaps(SNAPSHOTS, redo=REDO)

In [ ]:
# Plotting  ·  s01: square map

# ---- what to draw -----------------------------------------------------------
SNAP = 68

MAP = dict(
    quantity = 'logratio',    # 'fret' or 'logratio'
    stars    = 'all',     # 'all' host-bound stars, or 'contested' only
    smooth   = True,      # meshoid kernels (False: plain bins)
    joint    = False,     # True: host-only and contested stars averaged together
    view     = 'face',    # 'face' or 'edge'
    box      = 0.9,       # half-width of the map, R_vir
    sigma_min = 1e4,      # smooth: blank where the deposited surface density is below this (Msun/kpc^2)
)

# ---- colours and size -------------------------------------------------------
CMAP       = None          # None: magma for fret, cgbreak_warm_v2 for logratio
ZMIN, ZMAX = -0.5, 3       # logratio colour range (grey at 0)
FIGSIZE    = (1.5*4.2, 1.5*3.7)    # inches


# ---- data -------------------------------------------------------------------
M   = SD.square_map(R, SNAP, **MAP)
img = M['img']
e   = M['edges']                                   # pixel edges, R_vir

a = SD.colour_args(M['args'], cmap=CMAP, zmin=ZMIN, zmax=ZMAX)
cmap, norm, clabel = SM.colours(a, img)


# ---- figure -----------------------------------------------------------------
S.use()
fig, ax = plt.subplots(figsize=FIGSIZE)

im = ax.imshow(
    np.ma.masked_invalid(img),
    origin='lower',
    extent=(e[0], e[-1], e[0], e[-1]),
    cmap=cmap,
    norm=norm,
    interpolation='none' if a.smooth else 'nearest',
)
ax.set_facecolor('white')

# R_vir circle and the host centre
ax.add_patch(Circle((0, 0), 1.0, fill=False, ls=(0, (3, 2)), lw=0.8, ec=S.INK2))
ax.plot([0], [0], marker='+', ms=5, mew=1, color=S.INK)


# ---- axes -------------------------------------------------------------------
ax.set_xlim(e[0], e[-1])
ax.set_ylim(e[0], e[-1])
ax.set_aspect('equal')

_, _, xl, yl = SM.VIEWS[a.view]
ax.set_xlabel(rf'${xl}/R_{{\rm vir}}$')
ax.set_ylabel(rf'${yl}/R_{{\rm vir}}$')


# ---- colourbar, title, footnote ---------------------------------------------
SM.finish_colorbar(fig, im, a, clabel, S, ax=ax, fraction=0.046, pad=0.01)
#SM.title(ax, M['d'])
#SM.footnote(fig, M['P'], S)


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, M['stem'], SAVE)

In [ ]:
# Plotting  ·  s02: polar map

# ---- what to draw -----------------------------------------------------------
SNAP = 68

MAP = dict(
    quantity = 'logratio',    # 'fret' or 'logratio'
    stars    = 'all',     # 'all' host-bound stars, or 'contested' only
    smooth   = True,      # meshoid kernels, re-binned into the polar cells (False: plain bins)
    joint    = False,     # True: host-only and contested stars averaged together
    view     = 'face',    # 'face' or 'edge'
    box      = 1,       # outer radius, R_vir
    sigma_min = 1e4,      # smooth: blank where the deposited surface density is below this (Msun/kpc^2)
)

POLAR = dict(
    nr     = 2*72,          # radial cells
    nphi   = 5*72,          # angular cells (5 deg)
    rscale = 'linear',    # 'linear', or 'sqrt' (opens up the inner halo)
    phi0   = 0.0,         # rotate phi = 0, degrees
)

# ---- colours and size -------------------------------------------------------
CMAP       = None          # None: magma for fret, cgbreak_warm_v2 for logratio
ZMIN, ZMAX = -0.5, 3       # logratio colour range (grey at 0)
FIGSIZE    = (1.5*4.4, 1.5*3.9)    # inches


# ---- data -------------------------------------------------------------------
M     = SD.polar_map(R, SNAP, **MAP, **POLAR)
img   = M['img']                                   # (radial cell, angular cell)
r_e   = M['r_edges']                               # R_vir
phi_e = M['phi_edges']                             # radians

a = SD.colour_args(M['args'], cmap=CMAP, zmin=ZMIN, zmax=ZMAX)
cmap, norm, clabel = SM.colours(a, img)

def rad(r):
    # radius as drawn (sqrt-spaced or linear)
    return np.sqrt(r) if M['sqrt'] else r


# ---- figure -----------------------------------------------------------------
S.use()
fig, ax = plt.subplots(figsize=FIGSIZE, subplot_kw=dict(projection='polar'))

im = ax.pcolormesh(
    phi_e, rad(r_e), np.ma.masked_invalid(img),
    cmap=cmap,
    norm=norm,
    shading='flat',
    rasterized=True,
)
ax.set_facecolor('white')

# R_vir circle
th = np.linspace(0, 2 * np.pi, 361)
ax.plot(th, np.full_like(th, rad(1.0)), ls=(0, (3, 2)), lw=0.8, color=S.INK2)


# ---- axes -------------------------------------------------------------------
ax.set_ylim(0, rad(a.box))

r_ticks = [r for r in (0.1, 0.25, 0.5, 0.75, 1.0) if r <= a.box]
ax.set_yticks([rad(r) for r in r_ticks])
ax.set_yticklabels([f'{r:g}' for r in r_ticks], fontsize=6, color=S.INK2, fontweight="bold")
ax.set_rlabel_position(45)

ax.tick_params(axis='x', labelsize=6.5)
ax.grid(color=S.HAIR, lw=0.4)
ax.text(np.radians(22.5), rad(a.box) * 1.08, r'$R/R_{\rm vir}$',
        fontsize=6.5, color=S.INK2, ha='left', va='center')


# ---- colourbar, title, footnote ---------------------------------------------
SM.finish_colorbar(fig, im, a, clabel, S, ax=ax, fraction=0.05, pad=0.1, shrink=1, )
#SM.title(ax, M['d'], pad=14)
#SM.footnote(fig, M['P'], S)


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, M['stem'], SAVE)

## s03: f_ret of newborn stars over cosmic time and r/R_vir

Per snapshot, every newborn is a 3D Gaussian integrated over spherical shells; each radial profile is drawn as a vertical band over the interval its newborns formed in.

In [ ]:
# Processing

REDO = False

D03 = R.newborns(redo=REDO)      # every newborn star of snapshots 1-177 (~5 min the first time)

ck = D03['checks']
print(f"newborns {ck['newborns']:,}  |  contested {ck['contested_owner']:,}  |  "
      f"retained {ck['retained']:,}  |  satellite-born within R_vir {ck['sat_born_in_rvir']:,}")

In [ ]:
# Plotting  ·  s03: newborn f_ret over time and radius

# ---- what to draw -----------------------------------------------------------
BORN = 'all'                # newborns born in: 'all', 'satellite' or 'host'

BANDS = dict(
    only_contested = False,   # True: only newborns bound to the host and to their owner
    hsml_from      = 'all',   # smoothing lengths from 'all' stars, or from the 'newborn' ones
    kernel_frac    = 0.5,     # Gaussian sigma = this x distance to the 32nd neighbour
    sigma_floor    = 0.01,    # minimum sigma, R_vir
    nr_bands       = 200,     # shells on [0, 1] R_vir
    min_n_band     = 3.0,     # blank shells holding fewer (kernel-weighted) newborns
)

TRACKS = 0                  # thin grey orbits of the galaxies forming the most stars inside R_vir
MARKS  = []                 # snapshots ticked with an arrow on top

# ---- colours and size -------------------------------------------------------
CMAP        = 'magma'
SHADE       = True          # fade sparse shells towards white ...
ALPHA_MIN   = 0.4           # ... down to this opacity,
ALPHA_GAMMA = 0.5           # ... as ALPHA_MIN + (1 - ALPHA_MIN) x^ALPHA_GAMMA (x: log newborn density)
TIME_AXIS   = 'cosmic'      # 'cosmic' or 'lookback'
FIGSIZE     = (5.4, 3.2)    # inches


# ---- data -------------------------------------------------------------------
B     = SD.newborn_bands(R, BORN, **BANDS)
img   = B['img']                                   # (time pixel, shell): f_ret
n_k   = B['n_k']                                   # (time pixel, shell): newborns in the kernel
te    = B['te']                                    # time edges, Gyr
r_e   = B['re']                                    # shell edges, R_vir
t_now = B['t_now']

def tx(t):
    # time as drawn on the x axis
    t = np.asarray(t)
    return t_now - t if TIME_AXIS == 'lookback' else t


# ---- colours: f_ret -> RGB, faded where few newborns form -------------------
norm = TwoSlopeNorm(0.5, 0.0, 1.0)
cmap = matplotlib.colormaps[CMAP]
rgb  = cmap(norm(np.nan_to_num(img.T)))[..., :3]

min_n  = BANDS['min_n_band']
enough = n_k >= min_n
if SHADE:
    # opacity grows with log newborn density, from min_n to the 99th percentile
    hi    = np.nanpercentile(n_k[enough], 99) if enough.any() else min_n * 10
    x     = np.log(np.maximum(n_k.T, 1e-30) / min_n) / np.log(hi / min_n)
    alpha = ALPHA_MIN + (1.0 - ALPHA_MIN) * np.clip(x, 0, 1) ** ALPHA_GAMMA
else:
    alpha = np.ones(img.T.shape)

alpha = np.where(np.isfinite(img.T), alpha, 0.0)
rgb   = 1.0 - alpha[..., None] * (1.0 - rgb)       # blend onto white


# ---- figure -----------------------------------------------------------------
S.use()
fig, ax = plt.subplots(figsize=FIGSIZE)

ax.imshow(
    rgb,
    origin='lower',
    aspect='auto',
    interpolation='nearest',
    extent=(float(tx(te[0])), float(tx(te[-1])), float(r_e[0]), float(r_e[-1])),
)
ax.set_facecolor('white')


# ---- optional: galaxy orbits and snapshot marks -----------------------------
if TRACKS > 0:
    ranked = list(dict.fromkeys(D03['track_id'].tolist()))[:TRACKS]     # stored in rank order
    for g in ranked:
        k     = D03['track_id'] == g
        snaps = D03['track_snap'][k]
        ts    = np.interp(snaps, D03['snap_list'], D03['snap_time'])
        rr    = D03['track_r'][k].astype(float)

        o = np.argsort(ts)
        ts, rr, snaps = ts[o], rr[o], snaps[o]
        rr = np.where(rr <= 1.0, rr, np.nan)               # inside R_vir only
        rr[np.r_[False, np.diff(snaps) > 1]] = np.nan      # break the line over missing snapshots

        ax.plot(tx(ts), rr, lw=0.5, color=S.INK2, alpha=0.6, zorder=3)

t_of = dict(zip(D03['snap_list'].tolist(), D03['snap_time'].tolist()))
for s in MARKS:
    ax.annotate(
        '', xy=(tx(t_of[s]), 1.0), xytext=(tx(t_of[s]), 1.06), annotation_clip=False,
        arrowprops=dict(arrowstyle='-|>', lw=0.6, color=S.INK2, mutation_scale=5),
    )


# ---- axes -------------------------------------------------------------------
ax.set_xlim(*sorted((float(tx(0.0)), float(tx(t_now)))))
if TIME_AXIS == 'lookback':
    ax.invert_xaxis()
ax.set_ylim(r_e[0], 1.0)

r_ticks = [0, 0.1, 0.25, 0.5, 0.75, 1.0]
ax.set_yticks(r_ticks)
ax.set_yticklabels([f'{r:g}' for r in r_ticks])

ax.set_xlabel('lookback time [Gyr]' if TIME_AXIS == 'lookback' else 'cosmic time [Gyr]')
ax.set_ylabel(r'$r/R_{\rm vir}$')

# redshift ticks on top
zz, tt  = D03['snap_z'], D03['snap_time']
o       = np.argsort(zz)
z_ticks = [z for z in (6, 3, 2, 1, 0.5, 0.2) if zz.min() - 1e-3 <= z <= zz.max()]

top = ax.secondary_xaxis('top')
top.set_xticks(tx(np.interp(z_ticks, zz[o], tt[o])))
top.set_xticklabels([f'{z:g}' for z in z_ticks])
top.tick_params(labelsize=6.5)
top.set_xlabel(r'$z$', fontsize=7, labelpad=2)


# ---- colourbar --------------------------------------------------------------
cb = S.colorbar(
    fig, ScalarMappable(norm=norm, cmap=cmap), ax=ax,
    label=r'$f_{\rm retention}$', ends=False, fraction=0.035, pad=0.02,
)
cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cb.set_ticklabels(['0', '0.25', '0.5', '0.75', '1'])

end_style = dict(transform=cb.ax.transAxes, ha='center', fontsize=6.5, color=S.INK2)
cb.ax.text(0.5, 1.035, 'birth host', va='bottom', **end_style)
cb.ax.text(0.5, -0.035, 'relabelled', va='top', **end_style)


# ---- footnote ---------------------------------------------------------------
m, ret = B['m'], B['ret']                          # newborns within R_vir, and whether retained
who = {'all': 'all newborns', 'satellite': 'born in a satellite', 'host': 'born in the host'}[BORN]
if BANDS['only_contested']:
    who += ', contested'
retained = 100 * (m * ret).sum() / m.sum()

fig.text(0.5, 0.01, rf'{who}: {m.size:,} within $R_{{\rm vir}}$, {retained:.1f}% retained',
         ha='center', va='bottom', fontsize=6.3, color=S.INK2)
fig.subplots_adjust(left=0.1, right=0.97, bottom=0.17, top=0.88)


# ---- show (and save if SAVE) ------------------------------------------------
stem = B['stem'] + ('' if SHADE else '_flat') + ('_lb' if TIME_AXIS == 'lookback' else '')
SD.show(fig, R.stem(stem), SAVE)

## s031: the whole host-bound population, averaged over spherical shells

Per snapshot, every host-bound star is a 3D Gaussian (smoothing lengths from all stars) integrated over spherical shells. Drawn twice: with the satellite orbits, and without them (`_nosat`).

In [ ]:
# Processing

REDO = False

D031 = R.population(redo=REDO)   # shell sums of every snapshot (~5 min the first time)

ck = D031['checks']
print(f"host-bound star-snapshots {ck['host_bound_star_snapshots']:,}  |  "
      f"contested {ck['contested_star_snapshots']:,}  |  "
      f"owner keeps {100 * ck['contested_retained_mass_frac']:.2f}% of the contested mass")
print('orbits available (most newborn mass inside R_vir first):', ck['tracks'])

In [ ]:
# Plotting  ·  s031: population shells, with and without the orbits

# ---- what to draw -----------------------------------------------------------
QUANTITY   = 'fret'         # 'fret' or 'logratio'
SMOOTHING  = 'joint'        # 'joint' (all stars together) or 'layered' (contested over host-only)
STAT       = 'mean'         # 'mean' (mass-weighted) or 'dmean' (density-weighted: rho alone)
MIN_N_BAND = 3.0            # blank shells holding fewer (kernel-weighted) stars

# ---- orbits -----------------------------------------------------------------
TRACKS   = 8                # satellites with the most newborn mass inside R_vir
TRACK_LW = 1.3
SUNK_LW  = 0.6              # line width once the satellite has sunk, i.e. ...
SINK_R   = 0.25             # ... never again rises above this r/R_vir

# ---- colours and size -------------------------------------------------------
CMAPS = dict(fret='magma', logratio='cgbreak_warm_v2')     # f_ret: magma or inferno
ZMIN, ZMAX  = -0.5, 3       # log X colour range (grey at 0)
SHADE       = False         # True: fade sparse shells towards white ...
ALPHA_MIN   = 0.4           # ... down to this opacity,
ALPHA_GAMMA = 0.5           # ... as ALPHA_MIN + (1 - ALPHA_MIN) x^ALPHA_GAMMA (x: log star count)
FIGSIZE     = (5.4, 3.2)    # inches


# ---- data -------------------------------------------------------------------
P     = D031
t     = P['snap_time']                             # snapshot times, Gyr
r_e   = P['edges']                                 # shell edges, R_vir
t_now = float(t.max())

img, N = SD.shell_value(P, QUANTITY, STAT, SMOOTHING, MIN_N_BAND)    # (snapshot, shell)


# ---- colours: value -> RGB per snapshot band, spread over a fine time grid ----
v = SimpleNamespace(quantity=QUANTITY, es='owner', project='logmean',
                    cmap=CMAPS[QUANTITY], zmin=ZMIN, zmax=ZMAX, vlim=None)
cmap_name, norm, clabel = SM.colours(v, img)
cmap = matplotlib.colormaps[cmap_name]

drawn = np.isfinite(img)
if SHADE:
    hi    = np.nanpercentile(N[drawn], 99) if drawn.any() else MIN_N_BAND * 10
    x     = np.log(np.maximum(N, 1e-30) / MIN_N_BAND) / np.log(max(hi, MIN_N_BAND * 1.0001) / MIN_N_BAND)
    alpha = ALPHA_MIN + (1 - ALPHA_MIN) * np.clip(x, 0, 1) ** ALPHA_GAMMA
else:
    alpha = np.ones_like(img)
alpha = np.where(drawn, alpha, 0.0)

base_val = 1.0 if QUANTITY == 'fret' else 0.0
rgb = cmap(norm(np.nan_to_num(img, nan=base_val)))[..., :3]
rgb = 1.0 - alpha[..., None] * (1.0 - rgb)         # blend onto white

te, col, valid = SD.fine_columns(t, t_now)         # each time pixel -> the snapshot whose band covers it
pix = np.where(valid[:, None, None], rgb[col], 1.0)  # (time pixel, shell, RGB)
pix = np.transpose(pix, (1, 0, 2))                 # rows = shells, for imshow


# ---- footnote and file name -------------------------------------------------
ck        = P['checks']
contested = 100 * ck['contested_star_snapshots'] / max(ck['host_bound_star_snapshots'], 1)
kept      = ck['contested_retained_mass_frac']
weighting = {'mean': 'mass-weighted', 'dmean': 'density-weighted'}[STAT]

note = f'{SMOOTHING} smoothing, {weighting} mean over shells; '
if kept is not None:
    note += (rf'host-bound stars within $R_{{\rm vir}}$, all snapshots: contested (host and owner bind) '
             rf'{contested:.1f}%, owner keeps {100 * kept:.2f}% of their mass')
else:
    note += 'host-bound stars'

stem = 's031_' + ('fret' if QUANTITY == 'fret' else 'logx') + f'_{SMOOTHING}'
if STAT != 'mean':
    stem += f'_{STAT}'
if QUANTITY == 'fret':
    stem += '_' + CMAPS['fret'].replace('.', '-')


# ---- figure -----------------------------------------------------------------
def draw(with_orbits):
    S.use()
    fig, ax = plt.subplots(figsize=FIGSIZE)

    ax.imshow(
        pix,
        origin='lower',
        aspect='auto',
        interpolation='nearest',
        extent=(float(te[0]), float(te[-1]), float(r_e[0]), float(r_e[-1])),
    )
    ax.set_facecolor('white')

    # ---- orbits: full width until the sinking time, thinner after it --------
    for g, tt, rr in (P['tracks'][:TRACKS] if with_orbits else []):
        t_sink = SD.sinking_time(tt, rr, SINK_R)
        late   = tt >= t_sink if t_sink is not None else np.zeros(tt.size, bool)
        early  = ~late | np.r_[False, ~late[:-1]]           # overlap one sample: no gap

        ax.plot(tt, np.where(early, rr, np.nan), color='black', lw=TRACK_LW, zorder=3, solid_capstyle='round')
        if late.any():
            ax.plot(tt, np.where(late, rr, np.nan), color='black', lw=SUNK_LW, zorder=3, solid_capstyle='round')

    # ---- axes ---------------------------------------------------------------
    ax.set_xlim(0.0, t_now)
    ax.set_ylim(r_e[0], 1.0)

    r_ticks = [0, 0.1, 0.25, 0.5, 0.75, 1.0]
    ax.set_yticks(r_ticks)
    ax.set_yticklabels([f'{r:g}' for r in r_ticks])

    ax.set_xlabel('cosmic time [Gyr]')
    ax.set_ylabel(r'$r/R_{\rm vir}$')

    # redshift ticks on top
    zz      = P['snap_z']
    o       = np.argsort(zz)
    z_ticks = [z for z in (6, 3, 2, 1, 0.5, 0.2) if zz.min() - 1e-3 <= z <= zz.max()]

    top = ax.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], t[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.tick_params(labelsize=6.5)
    top.set_xlabel(r'$z$', fontsize=7, labelpad=2)

    # ---- colourbar and footnote ---------------------------------------------
    cb = SM.finish_colorbar(fig, ScalarMappable(norm=norm, cmap=cmap), v, clabel, S,
                            ax=ax, fraction=0.035, pad=0.02)
    if QUANTITY == 'fret':
        cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
        cb.set_ticklabels(['0', '0.25', '0.5', '0.75', '1'])

    fig.text(0.5, 0.01, note, ha='center', va='bottom', fontsize=6.0, color=S.INK2)
    fig.subplots_adjust(left=0.1, right=0.97, bottom=0.17, top=0.88)
    return fig


# ---- show (and save if SAVE): with the orbits, then without -----------------
SD.show(draw(with_orbits=True), R.stem(stem), SAVE)
SD.show(draw(with_orbits=False), R.stem(stem + '_nosat'), SAVE)

## s032: satellite orbits coloured by their own f_ret

Each satellite's orbit r/R_vir(t) (cubic splines of the merger tree), coloured by the fraction of the stars it owns and binds that greedy assignment leaves with it. The processing cell lists every candidate (ids for `IDS`).

In [ ]:
# Processing

REDO = False

D032 = R.satellites(redo=REDO)   # per-satellite f_ret and newborns, all snapshots (~5 min the first time)

# every candidate (enters R_vir, >= 100 star particles at its peak), by infall time
SD.candidate_table(SD.candidates(R, min_npart=100))

In [ ]:
# Plotting  ·  s032 helpers: orbit lines, panel frame, labels


def draw_orbit(ax, tt, rr, sat_data, st, cmap, norm, tx):
    # One satellite's orbit r/R_vir(t), coloured by its f_ret; thinner after its sinking time.

    # f_ret along the line (known per snapshot, interpolated in time)
    f       = SD.fret_along(sat_data, tt)
    f_seg   = 0.5 * (f[:-1] + f[1:])
    defined = np.isfinite(f_seg)

    # segments between consecutive samples
    pts  = np.stack([tx(tt), rr], axis=1)
    seg  = np.stack([pts[:-1], pts[1:]], axis=1)
    good = np.all(np.isfinite(seg), axis=(1, 2))
    if st['drop_undefined']:
        good &= defined

    # widths: thinner once the satellite has sunk
    t_sink = SD.sinking_time(tt, rr, st['sink_r'])
    lw = np.full(tt.size - 1, st['lw'])
    if t_sink is not None:
        lw[tt[:-1] >= t_sink] = st['sunk_lw']

    # colours: f_ret, grey where undefined
    colours = cmap(norm(np.nan_to_num(f_seg)))
    colours[~defined] = to_rgba(st['undefined_colour'])

    ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good],
                                     capstyle='round', zorder=3))
    return t_sink


def frame(ax, st, tx, t_now, show_x=True, show_y=True, z_axis=True):
    # R_vir line, axis ranges and labels, redshift ticks on top.

    ax.axhline(1.0, zorder=1, **st['rvir_line'])

    t_lo = st['tmin']
    t_hi = st['tmax'] if st['tmax'] is not None else t_now
    ax.set_xlim(*sorted((float(tx(t_lo)), float(tx(t_hi)))))
    if st['time_axis'] == 'lookback':
        ax.invert_xaxis()
    ax.set_ylim(0.0, st['rmax'])

    if show_x:
        ax.set_xlabel('lookback time [Gyr]' if st['time_axis'] == 'lookback' else 'cosmic time [Gyr]')
    else:
        ax.tick_params(labelbottom=False)

    if show_y:
        ax.set_ylabel(r'$r/R_{\rm vir}$')
    else:
        ax.tick_params(labelleft=False)

    if z_axis and st['z_axis']:
        host    = D032['tree'][D032['tree']['Sub_tree_id'] == D032['host']]
        zz, tt  = host['Redshift'].astype(float), host['Time'].astype(float)
        o       = np.argsort(zz)
        z_ticks = [z for z in (6, 3, 2, 1, 0.5, 0.2) if zz.min() <= z <= zz.max()]

        top = ax.secondary_xaxis('top')
        top.set_xticks(tx(np.interp(z_ticks, zz[o], tt[o])))
        top.set_xticklabels([f'{z:g}' for z in z_ticks])
        top.tick_params(labelsize=6.5)
        top.set_xlabel(r'$z$', fontsize=7, labelpad=2)


def label_positions(sats, st, orbits, tx):
    # (id, x, y) of each label: where the satellite crosses R_vir ('infall') or where its line
    # ends ('end'); labels closer than 0.45 Gyr are lifted in steps of 0.07 R_vir.

    pos = []
    for sat in sats:
        g = int(sat['g'])
        if st['labels'] == 'infall':
            pos.append((g, tx(float(sat['t'])), 1.0))
        elif st['labels'] == 'end' and g in orbits:
            tt, rr = orbits[g]
            k = np.flatnonzero(np.isfinite(rr))
            if k.size:
                pos.append((g, tx(tt[k[-1]]), float(rr[k[-1]])))

    if not st['label_stagger']:
        return pos

    pos.sort(key=lambda p: p[1])
    lifted, last_x, level = [], None, 0
    for g, x, y in pos:
        close = last_x is not None and abs(x - last_x) < 0.45
        level = level + 1 if close else 0
        lifted.append((g, x, y + 0.07 * level))
        last_x = x
    return lifted

In [ ]:
# Plotting  ·  s032: satellite orbits

# ---- which satellites -------------------------------------------------------
IDS    = None               # e.g. [3674, 7465, 10779, 26681]: exactly these (overrides SELECT)
SELECT = 'sf'               # 'sf'     most newborn mass born in them inside R_vir
                            # 'ratio'  uniform in log M*/M_dm at infall x infall time
                            # 'grid' / 'even'  log M* x infall-time cells, most massive per cell
                            # 'mass'   most massive;  'infall'  spread evenly in infall time
NSAT   = 10

MIN_NPART = 100             # candidates own at least this many star particles at their peak
LOGM      = None            # (lo, hi): peak log M* range
INFALL    = None            # (lo, hi): infall time range, Gyr

# ---- panels and file name ---------------------------------------------------
LAYOUT    = 'single'        # 'single', 'mass' (a panel per log M* bin), 'each' (a panel per satellite)
MASS_BINS = 3               # LAYOUT 'mass'
NAME      = 'sf10'          # file name s032_satellites_<NAME>.png

# ---- style ------------------------------------------------------------------
STYLE = dict(
    figsize_single   = (5.4, 3.2),      # inches, LAYOUT 'single'
    panel_size       = (2.6, 1.7),      # inches per panel, LAYOUT 'mass' / 'each'
    ncols            = 3,               # panels per row

    cmap             = 'cmr.cosmic',    # f_ret colours (0: to the host, 1: kept)
    vmin             = 0.0,
    vmax             = 1.0,
    undefined_colour = '#c8c8c8',       # where the satellite binds none of its stars
    drop_undefined   = False,           # True: leave those stretches out instead of grey

    lw               = 1.3,
    sunk_lw          = 0.6,             # after the sinking time ...
    sink_r           = 0.25,            # ... from which it never again rises above this r/R_vir

    rmax             = 1.5,             # y axis top, R_vir
    tmin             = 0.0,             # x axis range, Gyr ...
    tmax             = None,            # ... None: the last snapshot
    time_axis        = 'cosmic',        # 'cosmic' or 'lookback'
    z_axis           = True,            # redshift ticks on top
    rvir_line        = dict(color='#52514e', lw=0.6, ls=(0, (3, 2))),

    labels           = 'infall',        # 'infall' (where it crosses R_vir), 'end', 'none'
    label_text       = '{logm:.1f}',    # fields: logm (peak M*), ratio (log M*/M_dm at infall), id, tinf
    label_fontsize   = 5.5,
    label_stagger    = True,            # lift labels whose infalls are close

    colorbar_label   = r'$f_{\rm retention}$ (own stars)',
    colorbar_ends    = ('kept', 'to host'),
    footnote         = True,
    dpi              = 300,
)
st = STYLE


# ---- data -------------------------------------------------------------------
cands = SD.candidates(R, MIN_NPART, LOGM, INFALL)
chosen, missing = SD.select(cands, SELECT, NSAT, IDS)
if missing:
    print(f'not candidates (no R_vir entry or too few particles): {missing}')

info   = {int(sat['g']): sat for sat in chosen}    # id -> candidate row
orbits = SD.orbits(R, list(info))                  # id -> (t, r/R_vir)

cmap  = matplotlib.colormaps[st['cmap']]
norm  = Normalize(st['vmin'], st['vmax'])
t_now = float(D032['snap_time'].max())

def tx(t):
    # time as drawn on the x axis
    t = np.asarray(t)
    return t_now - t if st['time_axis'] == 'lookback' else t


# ---- panels: (title, satellite ids) each ------------------------------------
if LAYOUT == 'single':
    groups = [('', list(info))]

elif LAYOUT == 'each':
    groups = [(rf'{g}  ($\log M_\star$ {np.log10(info[g]["m"]):.1f})', [g]) for g in info]

else:   # 'mass': one panel per log M* bin
    logm  = {g: np.log10(info[g]['m']) for g in info}
    edges = np.linspace(min(logm.values()), max(logm.values()) + 1e-9, MASS_BINS + 1)
    groups = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        members = [g for g in info if lo <= logm[g] < hi]
        if members:
            groups.append((rf'$\log M_\star$ {lo:.1f}-{hi:.1f}', members))


# ---- figure: one panel, or a grid of panels ---------------------------------
S.use()
if LAYOUT == 'single':
    fig, ax = plt.subplots(figsize=st['figsize_single'])
    axes  = [ax]
    ncols = 1
else:
    ncols = min(st['ncols'], len(groups))
    nrows = int(np.ceil(len(groups) / ncols))
    w, h  = st['panel_size']
    fig, grid = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows + 0.6),
                             squeeze=False, sharex=True, sharey=True)
    axes = list(grid.flat)
    for ax in axes[len(groups):]:
        ax.set_visible(False)


# ---- each panel: orbits, frame, labels --------------------------------------
sinks = {}
for i, ((title, members), ax) in enumerate(zip(groups, axes)):
    row, col = divmod(i, ncols)

    for g in members:
        if g in orbits:
            sinks[g] = draw_orbit(ax, *orbits[g], D032['sats'].get(g), st, cmap, norm, tx)

    lowest_in_column = LAYOUT == 'single' or i + ncols >= len(groups)
    frame(ax, st, tx, t_now, show_x=lowest_in_column, show_y=(col == 0), z_axis=(row == 0))

    if title:
        ax.set_title(title, fontsize=6.5, pad=12 if (row == 0 and st['z_axis']) else 3)

    if st['labels'] != 'none':
        for g, x, y in label_positions([info[g] for g in members], st, orbits, tx):
            sat  = info[g]
            text = st['label_text'].format(logm=np.log10(sat['m']), id=g, tinf=sat['t'], ratio=sat['ratio'])
            ax.annotate(text, (x, y), xytext=(2, 3), textcoords='offset points',
                        fontsize=st['label_fontsize'], color=S.INK2, zorder=4)


# ---- colourbar --------------------------------------------------------------
cb = S.colorbar(
    fig, ScalarMappable(norm=norm, cmap=cmap),
    ax=axes[0] if LAYOUT == 'single' else axes[:len(groups)],
    label=st['colorbar_label'], ends=False,
    fraction=0.035 if LAYOUT == 'single' else 0.02, pad=0.02,
)
cb.set_ticks(np.linspace(st['vmin'], st['vmax'], 5))

top_end, bottom_end = st['colorbar_ends']
end_style = dict(transform=cb.ax.transAxes, ha='center', fontsize=6.5, color=S.INK2)
cb.ax.text(0.5, 1.035, top_end, va='bottom', **end_style)
cb.ax.text(0.5, -0.035, bottom_end, va='top', **end_style)


# ---- footnote and margins ---------------------------------------------------
if st['footnote']:
    fig.text(0.5, 0.005,
             f'{len(chosen)} of {cands.size} satellites entering R_vir with >= {MIN_NPART} star particles; '
             'f_ret = 1 outside R_vir by construction (host energies stored inside R_vir only)',
             ha='center', va='bottom', fontsize=5.5, color=S.INK2)
if LAYOUT == 'single':
    fig.subplots_adjust(left=0.1, right=0.97, bottom=0.17, top=0.88 if st['z_axis'] else 0.95)


# ---- the chosen satellites, then show (and save if SAVE) --------------------
table = SD.candidate_table(np.array(chosen, dtype=cands.dtype))
table['sinking [Gyr]'] = [None if sinks.get(g) is None else round(sinks[g], 2) for g in table['id']]
display(table)

SD.show(fig, R.stem('s032_satellites' + (f'_{NAME}' if NAME else '')), SAVE, dpi=st['dpi'])

## Comp01b and Comp01: parameters and data

The three figures share their settings: the **Comp01b parameters** (colour scale and colourbar by stellar mass, the zoom inset INSET, and the beads MBEADS shared with Comp01c), the **Comp01c parameters** (colour scale by dark-matter mass) and the **Comp01 parameters** (layout, panels, orbit set, styles, fonts; Comp01b and Comp01c take everything but their colours from them). One processing cell computes the data of all three. Then the figures: Comp01b, Comp01c, Comp01d (the three orbit panels stacked; its own few options: Comp01d parameters), then Comp01 (and its mirrored layout).

The zoom inset (INSET) is a close-up in Comp01b's orbit panel (`INSET['show']`); Comp01 shows only its mark (`INSET['comp01']`) with a note under it (`INSET['comp01_text']`), to point the reader to Comp01b.

In [ ]:
# Plotting  ·  Comp01b parameters: colour scale, colourbar, line widths, zoom inset (all the rest: the Comp01 parameters, below)

MSTAR = dict(
    mode       = 'mstar',         # 'mstar':  colour = M* (log colour scale), line widths as in Comp01
                                  # 'fbound': colour = f_bound = M*(t) / M*_peak, line width growing with M*_peak
    source     = 'catalogue',     # 'catalogue': roadrunner's Mtot, the stars it holds bound to the galaxy;
                                  # 'owned': the stars the galaxy owns (birth map + infall lists), bound or not;
                                  # 'bound': owned and bound (the f_ret denominator)
    cmap       = 'octarine_tuned',   # both modes: any matplotlib, cmr.* or cmyt.* name, octarine_tuned, or 'name-T'
    name       = 'comp01b',       # file name <name>_<mode>_<ORBIT_SET>.png
)

# ---- mode 'mstar': M* on a log colour scale ---------------------------------
MSTAR_SCALE = dict(
    vmin       = 5.0,             # log10 M*/Msun at the bottom of the colour scale ...
    vmax       = 8.0,             # ... and at the top (lines outside the range take the end colours)
    ticks      = None,            # colourbar ticks in Msun, e.g. (1e5, 1e6, 1e7, 1e8, 1e9); None: automatic
    cbar_label = r'$M_\star\ [{\rm M}_\odot]$',
    cbar_ends  = None,            # notes beyond the colourbar's ends, (top, bottom); None: none
)

# ---- mode 'fbound': the share of its peak stellar mass a satellite still holds; line width from that peak ----
FBOUND = dict(
    peak       = 'global',        # 'global': its maximum over its whole life (the colour shows growth, then loss);
                                  # 'running': its maximum so far (loss only)
    log        = False,           # log colour scale ...
    vmin       = 0.0,             # ... from vmin (with log, e.g. 1e-2) ...
    vmax       = 1.0,             # ... to vmax
    cbar_label = r'$f_{\rm bound} = M_\star / M_{\star,{\rm peak}}$',
    cbar_ends  = ('peak', 'stripped'),   # notes beyond the colourbar's ends, (top, bottom); None: none
    width_mass = (1e5, 1e9),      # M*_peak range [Msun] mapped onto ...
    width_lw   = (0.5, 3.0),      # ... these line widths (sinking thins a line by ORBIT_STYLE's sunk_lw / lw)
    width_scale = 'log',          # 'log': width grows with log M*_peak; 'linear': with M*_peak itself
    width_background = True,      # background lines scale too (times BACKGROUND['lw'] / ORBIT_STYLE['lw'])
    legend     = dict(show=True, loc='lower left', masses=(1e6, 1e7, 1e8)),   # line-width key
)

# ---- zoom inset: a square close-up of one region of the orbit panel, with its own colour range ----
INSET = dict(
    show       = True,            # the inset in Comp01b ...
    comp01     = True,            # ... and its mark (only the mark, no inset) in Comp01's orbit panels: 'look here in Comp01b'
    comp01_text = 'Fig A.2',   # Comp01: note just under that mark (FONTS['notes']); '' or None: none
    snaps      = (34, 43),        # the region: snapshots s1..s2 (27155 stripped at pericentres, regaining a little
    r          = (0.04, 0.21),    # at apocentres) and r/R_vir r1..r2; marked on the orbit panel with Comp01's MARK style
    pos        = (-0.045, -0.4),    # lower-left corner of the inset, fractions of the orbit panel (x, y)
    size       = 1.4,             # width of the inset, inches (its height follows the mark's aspect ratio)
    stems      = 'down',          # 'down': the inset is below its mark (bottom corners of the mark -> top corners of
                                  # the inset); 'up': above it (top corners of the mark -> bottom corners of the inset)
    vrange     = dict(mstar=(3.85, 5.5), fbound=(0.0, 1.0)),   # its colour range per mode: log10 M*, f_bound
    background = False,            # the faint background orbits in the inset too
    cbar       = dict(show=True, width=0.08, pad=0.00, length=1.0, label=True),   # its own colourbar, right of it (label: the main one's)
    fontsize   = None,            # tick labels in the inset and its colourbar (None: FONTS['ticks'], as the rest)
)

# ---- beads on the M* orbits (and Comp01c's): one per BEADS['dt'] bin with star formation; size by its SFR, fill by
#      the line's own value where the bead lands (M* or f_bound here, M_DM in Comp01c) ----
MBEADS = dict(
    show       = True,
    cmap       = None,            # None ('none', 'None'): the panel's colour map and range (a bead takes its line's
                                  # colour); or a colour map name, on the range vrange, keyed by colour_key
    vrange     = dict(mstar=None, fbound=None, dm=None),   # that map's range per panel: (lo, hi) in log10 M*, f_bound,
                                  # log10 M_DM (Comp01c); None: the panel's colourbar range
    colour_key = dict(show=True, loc='upper left', values=None, n=4, title=None),   # (own cmap only) beads coloured
                                  # at these values (Msun for the masses); None: n evenly spread over the range;
                                  # title None: the panel's colourbar label
    legend     = dict(show=True, loc='upper right'),   # size key, at BEADS['legend']['values'] (or BEADS' classes)
)
# which births, sizes (show_min, sfr_min..sfr_max, d_min..d_max or classes), edges, fit_dt, zorder and
# undefined_colour: Comp01's BEADS

In [ ]:
# Plotting  ·  Comp01c parameters: colour scale and colourbar by dark-matter mass (all the rest: the Comp01 parameters; beads: MBEADS)

DMASS = dict(
    source     = 'mass',          # merger-tree column: 'mass' (virial mass), 'bound_mass' or 'peak_mass'
    vmin       = 7.0,             # log10 M_DM/Msun at the bottom of the colour scale ...
    vmax       = 11.0,            # ... and at the top (lines outside the range take the end colours)
    ticks      = None,            # colourbar ticks in Msun, e.g. (1e8, 1e9, 1e10, 1e11); None: automatic
    cmap       = 'ghostlight_tuned',   # any matplotlib, cmr.* or cmyt.* name, octarine_tuned, ghostlight_tuned, or 'name-T'
    cbar_label = r'$M_{\rm DM}\ [{\rm M}_\odot]$',
    cbar_ends  = None,            # notes beyond the colourbar's ends, (top, bottom); None: none
    name       = 'comp01c',       # file name <name>_<ORBIT_SET>
)

In [ ]:
# Plotting  ·  Comp01d parameters: the three orbit panels stacked (all the rest: the Comp01, Comp01b and Comp01c parameters)

COMP01D = dict(
    gap        = 0.25,            # inches between the panels (each panel and its colourbar: SIZE, CBAR['orbit'], as in Comp01)
    share_x    = True,            # True: time tick labels and 'cosmic time' on the bottom panel only, redshifts (ORBIT_STYLE
                                  # ['z_axis']) on the top one only; False: every panel exactly as in its own figure (raise gap)
    bead_keys  = 'all',           # bead keys (size, and colour with an own map): 'all' (each panel as in its own figure),
                                  # 'top' (the top panel only), 'none'
    name       = 'comp01d',       # file name <name>_<MSTAR['mode']>_<ORBIT_SET>
)

In [ ]:
# Plotting  ·  Comp01 parameters: layout, what each panel shows

# ---- panel sizes, inches ----------------------------------------------------
SIZE = dict(
    orbit_w  = None,        # orbit panel width; None: as wide as the bottom row (zooms + gap + map)
    orbit_h  = 2.6,         # orbit panel height
    map_size = 3.5,         # side of the square map
    zoom_w   = 3.2,         # width of each zoom panel
    zoom_h   = 1.2,        # height of each zoom panel
)

# ---- gaps between panels, inches (panel sizes stay fixed; the figure grows or shrinks) ----
GAP = dict(
    rows  = 0.75,           # vertical: orbit panel -> bottom row
    zooms = 1.0,           # vertical: between the two zoom panels
    cols  = 0.50,           # horizontal: zoom panels -> map
)
ALIGN = 'center'               # in the bottom row, the shorter of map / zoom stack sits at the 'top', 'center' or 'bottom'

# ---- nudges: move single panels off the grid, inches ------------------------
# (dx, dy): dx > 0 right, dy > 0 up; a colourbar moves with its panel; the figure grows to fit
OFFSET = {
    'orbit':  (0.0, 0.0),
    'map':    (0.70, 0.0),
    'zoom A': (-0.70, -0.15),
    'zoom B': (-0.325, 0.15),
}

# ---- colourbars, right of their panel ---------------------------------------
CBAR = dict(
    orbit = dict(           # f_ret
        width  = 0.2,      # thickness, inches
        pad    = 0.05,      # gap from the panel, inches
        length = 1.0,       # vertical extension, fraction of the panel height
        align  = 'center',  # 'center', 'top' or 'bottom' of the panel
    ),
    map = dict(             # log X
        width  = 0.2,
        pad    = 0.07,
        length = 1.0,
        align  = 'center',
    ),
)

# ---- figure margins, inches -------------------------------------------------
MARGIN = dict(left=0.45, right=0.55, top=0.35, bottom=0.40)

SHOW_NAMES = False           # write each panel's name inside it

# ---- what each bottom panel shows (also places its mark on the orbit panel) ----
MAP_SNAP = 68               # snapshot of the map
MAP_BOX  = 0.9              # half-width of the map, R_vir: its mark spans r = 0 .. MAP_BOX

ZOOM = {                    # one panel each: the stars satellites own and bind at snapshots s1..s2, within
                            # r/R_vir r1..r2 (the box marked on the orbit panel); sats: whose stars count:
                            # None = the satellites drawn (ORBIT_SET), 'all' = every satellite, or a list of ids
    'zoom B': dict(snaps=(126, 127),   r=(0.25, 0.60), sats=None),
    'zoom A': dict(snaps=(75, 76), r=(0.25, 0.45), sats=None),
}

# ---- zoom panels: log10(E_H/E_S) of those stars, all and newborn ------------
KDE = dict(
    histtype = 'kde',       # 'kde' (smooth) or 'step' (histogram)
    norm     = 'density',   # 'density' or 'counts' (per dex: number of stars, or Msun with weight 'mass')
    per      = 'population',  # density over 'population' (all the stars in the box, so the curve's area is the
                            # host-bound share; the rest, not host-bound, is kept at log X = -inf) or 'curve' (area 1)
    weight   = 'number',    # 'number' or 'mass'
    bw       = 0.05,        # kde bandwidth, dex
    bins     = 60,          # step: bins across xrange
    xrange   = (-1.0, 3.0),
    all      = dict(color='#0b0b0b', lw=1.0, ls='-',          fill=1.0),    # fill: opacity of the log X colours
    newborn  = dict(color='#0b0b0b', lw=1.0, ls=(0, (3, 1.5)), fill=0.45),  #       under the curve (0: none)
    zero     = dict(color='#52514e', lw=0.6, ls='-'),                       # the line at log X = 0
    notes    = dict(          # per curve: 'all N, R%' / 'newborn N, R%' (stars in the box, their retention)
        show      = True,
        where     = 'legend',    # 'upper left', 'upper right', 'lower left', 'lower right', 'title' or 'legend'
        inset     = (0.02, 0.04),    # corners: distance from the panel's edges, fractions of the panel (x, y)
        title_pad = 4,               # 'title': gap between the text and the panel, points
    ),
    legend   = True,
)

# ---- orbit panel ranges -----------------------------------------------------
T_RANGE = (0.0, None)       # Gyr; None: the last snapshot
R_MAX   = 6                # top of the r/R_vir axis
R_LOG   = True              # logarithmic r/R_vir axis ...
R_MIN   = 1E-2             # ... from here (marks starting at r = 0 start here instead)
R_TICKS = (1E-2, 1E-1, 1)         # labelled r/R_vir values (log axis)
Z_TICKS = (10, 5, 4, 3, 2, 1, 0)              # redshifts labelled on the top axis

# ---- marks on the orbit panel and their links to the panels -----------------
MARK = dict(
    min_width = 0.25,       # Gyr: a mark narrower than this (e.g. a single snapshot) is widened to it, centred
    label     = False,      # the panel's name above its mark
    box  = dict(ec='#52514e', lw=0.7, ls='-', fc='none', alpha=1.0, zorder=5),            # the marks
    link = dict(color='#52514e', lw=0.5, ls=(0, (2, 2)), alpha=1.0, zorder=4),            # their lines to the panels
)
MARK_PANEL = {
        'inset': dict(box=dict(lw=1.0, zorder=101)),   # was lw 0.7, zorder 5
        'zoom A': dict(box=dict(lw=1.0), link=dict(lw=0.6)),   # was lw 0.7, zorder 5
        'zoom B': dict(box=dict(lw=1.0), link=dict(lw=0.6)),   # was lw 0.7, zorder 5
        'map': dict(box=dict(lw=1.0), link=dict(lw=0.6)),   # was lw 0.7, zorder 5

}             # per-panel changes, e.g. {'zoom A': dict(box=dict(ec='tab:red'), link=dict(color='tab:red'))}


# ---- orbits: which satellites, and how they are drawn -----------------------
ORBIT_SETS = {               # named sets of satellite ids; ORBIT_SET picks the one drawn
    'showcase':     [26681, 10326, 3674, 411773, 10779, 11168, 6871, 34253],   # groups + colours + backsplash
    'star_formers': [26681, 10326, 3674, 10779, 3567, 20813, 11508, 34253],    # strongest SFR inside R_vir
    'groups':       [26681, 10326, 10779, 11168, 34253, 25093],                # three infall groups, duos
    'groups_alt':   [18557, 5283, 2574, 2858, 11508, 48825],                   # three other infall groups
    'groups_3x3':   [26681, 10326, 17674, 10779, 11168, 78376, 34253, 25093],  # three groups, up to 3 each
    'sf_ladder':    [754, 3567, 11091, 10302, 520, 11508, 20813],              # peak SFR: max, min, log-even between
    'uniform':      [27155, 10326, 28819, 17907, 12006, 30241, 815932],        # infall time x log M*, mass alternating
    'mass_ladder':  [846062, 26681, 399201, 559042, 6871, 20988, 34253],       # log M*: max, min, even between
    'retention':    [17157, 4369, 855279, 399201, 3674, 34253, 2574],          # mean f_ret in R_vir, evenly spaced
    'epochs':       [26681, 20389, 3674, 411773, 34253, 846062],               # most / least massive per epoch
    'sf10':         [3567, 20813, 30272, 10779, 26681, 18557, 11168, 16069, 2574, 3674],   # the earlier default,
    'final':        [27155, 26681, 3674, 18557, 10779, 3567, 20813, 11508, 34253]
}
ORBIT_SET = 'final'          
                                    # favs, in decreasing order:
                                    #
                                    #   1. star formers -> good visibility of satellites overall
                                    #   2. gorups -> same
                                    #   3. 
                                    #
                                    #
                                    #
                                    #


if ORBIT_SET not in ORBIT_SETS:
    raise KeyError(f'ORBIT_SET {ORBIT_SET!r} is not one of {list(ORBIT_SETS)}')
ORBIT_IDS = ORBIT_SETS[ORBIT_SET]

ORBIT_ZORDER = {27155: 100}           
                            # drawing order of single satellites (Comp01 and Comp01b), e.g. {3567: 3.5, 26681: 2.5}:
                            # higher on top. Unlisted satellites: 3, the later in ORBIT_IDS on top. For reference:
                            # background 2, beads one above their own line, marks 4-5, ORBIT_STYLE['rvir_line'] zorder

ORBIT_STYLE = dict(
    cmap             = 'cmyt.arbre-0.9',    # f_ret colours (0: to the host, 1: kept); any matplotlib, cmr.* or cmyt.*
                                        # name; 'name-T' cuts it at T (T -> 1): arbre-0.8 ends in green, not yellow
    fret_log         = True,          # True: logarithmic f_ret colours and colourbar ...
    fret_min         = 1e-2,           # ... from here (lower f_ret, 0 included, gets the lowest colour)
    lw               = 1.4,
    sunk_lw          = 0.2,             # after the sinking time ...
    sink_r           = 0.1,            # ... from which it never again rises above this r/R_vir
    undefined_colour = '#c8c8c8',       # where the satellite binds none of its stars
    drop_undefined   = False,           # True: leave those stretches out instead of grey
    rvir_line        = dict(color='#52514e', lw=1, ls=(0, (3, 2)), zorder=1000),
    z_axis           = False,            # redshift ticks on top
    cbar_label       = r'$f_{\rm retention}$ (own stars)',
    cbar_ends        = ('kept', 'to host'),
)

# ---- star-formation beads on the orbits (exact birth times and initial masses, particle_data/birth_times.hdf5) ----
BEADS = dict(
    show        = True,
    dt          = 0.2,              # Gyr: SFR bin width, one bead per bin (bin centre, on the orbit);
                                    #   with fit_dt the largest bead is ~35 pt/Gyr x dt (0.1: 3.6 pt, 0.2: 7.1 pt)
    since       = 'inside',         # which births: 'inside' (only while the satellite is within r_max R_vir: no bead
                                    # beyond it; a bin crossing r_max is split there), 'infall' (from its first entry
                                    # within r_max on, also when it goes back out), 'all'
    r_max       = None,             # r/R_vir of that cut; None: R_MAX (the orbit panel's top)
    show_min    = 1e-3,             # Msun/yr: no bead below this
    sfr_min     = 1e-3,             # Msun/yr: SFR of the smallest bead (lower SFRs: that size) ...
    sfr_max     = 1.0,              # ... and of the largest (higher SFRs: that size); continuous sizes
    sizes       = 'continuous',     # 'continuous' (log SFR from sfr_min to sfr_max) or 'classes'
    classes     = (1e-3, 1e-2, 1e-1, 1.0),   # 'classes': one size per SFR threshold, Msun/yr
    d_min       = 2.5,              # bead diameter, points: smallest ...
    d_max       = 7.5,              # ... and largest
    fit_dt      = True,             # cap d_max to the beads' spacing (dt on the time axis): neighbours never touch
    edge        = dict(color='white', lw=0.5),
    undefined_colour = '#c8c8c8',   # bins where the satellite binds none of its newborns
    cmap        = None,             # fill: None ('none', 'None'): the orbits' colour map and range; or a colour map
                                    # name (as ORBIT_STYLE['cmap']) on the range vmin..vmax, keyed by colour_key
    vmin        = None,             # that map's range, in newborn f_ret; None: the orbits' ends
    vmax        = None,
    colour_key  = dict(show=True, loc='upper left', values=None, n=4, title=r'newborn $f_{\rm ret}$'),
                                    # (own cmap only) beads coloured at these values; None: n evenly spread over the range
    zorder      = 4,
    legend      = dict(show=True, loc='upper right', values=(1e-3, 1e-2, 1e-1, 1.0)),   # values: continuous sizes
)
# fill: the newborns' own f_ret (1 - those greedy gives to the host / those the satellite binds, initial-mass
# weighted, at each star's first snapshot), on the orbits' colour scale. Outside R_vir the host energies are
# not stored, so beads there (after a backsplash) are 'kept' by construction

# ---- background: every other satellite's orbit, faint, under the selected ones (no beads) ----
BACKGROUND = dict(
    show       = True,
    which      = 'candidates',      # 'candidates': satellites entering R_vir with >= min_npart star particles at
                                    # their peak (as s032); 'all': every galaxy owning >= min_npart star particles
    min_npart  = 50,
    colour     = 'fret',            # 'fret': their own f_ret, on the orbits' colour scale; or one colour, e.g. '#9a9a9a'
    alpha      = 0.55,
    lw         = 0.2,
    sunk_lw    = 0.0,               # after the sinking time (ORBIT_STYLE['sink_r'])
    zorder     = 2,                 # under the selected orbits (3) and the beads (BEADS['zorder'])
    rasterized = True,              # many lines: drawn as an image inside a vector (pdf/svg) file
)

# ---- map: how it is drawn (snapshot and half-width: MAP_SNAP, MAP_BOX above) ----
MAP_OPTS = dict(
    stars     = 'all',      # 'all' host-bound stars, or 'contested' only
    smooth    = True,       # meshoid kernels (False: plain bins)
    joint     = False,      # True: host-only and contested stars averaged together
    view      = 'face',     # 'face' or 'edge'
    sigma_min = 1e4,        # blank where the deposited surface density is below this (Msun/kpc^2)
)
MAP_CMAP   = 'cgbreak_warm_v2'   # any matplotlib, cmr.* or cmyt.* name, or 'name-T' (also fills the zoom panels)
MAP_ZRANGE = (-0.5, 3)      # log X colour range (grey at 0)

# ---- text sizes, points (common to every panel and colourbar) ---------------
FONTS = dict(
    label = 14,              # axis labels (x, y, colourbars, redshift)
    ticks = 9.5,            # tick labels (all axes and colourbars)
    notes = 10,            # the notes at both ends of the colourbars, the map's time
    marks = 8,              # the names above the marks on the orbit panel
)
NOTE_GAP = 0.04             # inches between a colourbar's end and its note


In [ ]:
# Processing  ·  Comp01b, Comp01c and Comp01

import cmyt  # noqa: F401   (registers the cmyt.* colormaps)

R.load()

host_rows = R.tree[R.tree['Sub_tree_id'] == R.host]
SNAP_TIME = dict(zip(host_rows['Snapshot'].astype(int).tolist(), host_rows['Time'].astype(float).tolist()))
SNAP_Z    = dict(zip(host_rows['Snapshot'].astype(int).tolist(), host_rows['Redshift'].astype(float).tolist()))
T_NOW     = SNAP_TIME[max(s for s in SNAP_TIME if s <= R.last)]

print(f'map     snapshot {MAP_SNAP:3d}  t = {SNAP_TIME[MAP_SNAP]:.2f} Gyr')
for name, z in ZOOM.items():
    s1, s2 = SD.zoom_range(z)
    print(f'{name}  snapshots {s1}-{s2}  t = {SNAP_TIME[s1]:.2f}-{SNAP_TIME[s2]:.2f} Gyr  r/R_vir {z["r"][0]}-{z["r"][1]}')

# where the sf satellites form most stars inside R_vir (once the s032 data is in memory)
if 'satellites' in R.data:
    import pandas as pd
    _sf, _ = SD.select(SD.candidates(R, 100), 'sf', 10)
    SF_PEAKS = SD.sf_peaks(R, [int(x['g']) for x in _sf])
    display(pd.DataFrame(SF_PEAKS).rename(columns={'g': 'id', 'm_in': 'newborn mass in R_vir [Msun]'}).round(3))

# ---- orbits: per-satellite f_ret over time (FRET, first cell) and the spline orbits ----
R.set_r_keep(max(2.0, R_MAX + 0.5))         # star tables reach a bit beyond the orbit panel's top
D032   = R.satellites()                     # ~5 min the first time (shared with s032; both f_ret definitions)
print(f'f_ret definition: {R.fret_def()!r}')
ORBITS = SD.orbits(R, ORBIT_IDS)            # id -> (t, r/R_vir)
missing = [g for g in ORBIT_IDS if g not in ORBITS or g not in D032['sats']]
if missing:
    print(f'no orbit or no f_ret for: {missing}')

# ---- map at MAP_SNAP --------------------------------------------------------
MAP_DATA = SD.square_map(R, MAP_SNAP, quantity='logratio', box=MAP_BOX, **MAP_OPTS)

# ---- zooms: every satellite's stars in each box (~1 min per new range; then in memory) ---
for name, z in ZOOM.items():
    SD.zoom_sample(R, SD.zoom_range(z), z['r'])        # the figure keeps those of the chosen satellites

# ---- star-formation beads: birth table, newborn fates (~3 min the first time), SFR bins ----
if BEADS['show']:
    SD.sf_beads(R, ORBIT_IDS, BEADS['dt'], since=BEADS['since'], r_max=R_MAX if BEADS['r_max'] is None else BEADS['r_max'])

# ---- background orbits: which galaxies, and their spline orbits -------------
if BACKGROUND['show']:
    SD.orbits(R, SD.background_ids(R, BACKGROUND['which'], BACKGROUND['min_npart'], exclude=ORBIT_IDS))

# ---- Comp01b: stellar mass of every galaxy over time (catalogue: a few seconds; owned / bound: from the satellite data) ----
MSTAR_DATA = SD.mstar_series(R, MSTAR['source'])

for g in ORBIT_IDS:
    e = MSTAR_DATA.get(g)
    print(f'{g:>8}: ' + (f'log M* {e["logm"].min():.2f}-{e["logm"].max():.2f} at {e["time"].size} snapshots '
                         f'({e["time"][0]:.2f}-{e["time"][-1]:.2f} Gyr)' if e is not None else 'no stellar mass'))

# ---- Comp01b / Comp01c beads (MBEADS; bins: BEADS) --------------------------
if MBEADS['show']:
    SD.sf_beads(R, ORBIT_IDS, BEADS['dt'], since=BEADS['since'], r_max=R_MAX if BEADS['r_max'] is None else BEADS['r_max'])

# ---- Comp01c: dark-matter mass of every galaxy over time (merger tree: instant) ----
DM_DATA = SD.dm_series(R, DMASS['source'])
for g in ORBIT_IDS:
    e = DM_DATA.get(g)
    print(f'{g:>8}: ' + (f'log M_DM {e["logm"].min():.2f}-{e["logm"].max():.2f} ({e["time"][0]:.2f}-{e["time"][-1]:.2f} Gyr)'
                         if e is not None else 'no dark-matter mass'))

## Comp01b: the Comp01 orbit panel coloured by stellar mass

Same satellites (ORBIT_SET), orbits, background, axes, line widths, sizes and fonts as Comp01: everything comes from the Comp01 parameters, so the two figures match. Only the colour scale, its colourbar and the zoom inset have their own options (Comp01b parameters, above). Beads (`MBEADS`): one per `BEADS['dt']` bin with star formation, size by its SFR, fill by the line's own value where it lands (or MBEADS' own colour map, with a colour key); the only mark is the inset's. Needs the s032 helpers cell, the two parameter cells and the processing cell above.

In [ ]:
# Plotting  ·  Comp01b figure: the orbit panel coloured by stellar mass (MSTAR['mode']; drawing order: ORBIT_ZORDER; zoom: INSET; beads: MBEADS)

# ---- geometry: the Comp01 orbit panel and its colourbar (SIZE, CBAR['orbit'], MARGIN) ----
orbit_w = SIZE['orbit_w'] if SIZE['orbit_w'] is not None else SIZE['zoom_w'] + GAP['cols'] + SIZE['map_size']
orbit_h = SIZE['orbit_h']
cbo     = CBAR['orbit']
cb_len  = cbo['length'] * orbit_h
cb_dy   = {'bottom': 0.0, 'center': (orbit_h - cb_len) / 2, 'top': orbit_h - cb_len}[cbo['align']]

boxes = {
    'orbit':   (MARGIN['left'], MARGIN['bottom'], orbit_w, orbit_h),
    'cbar M*': (MARGIN['left'] + orbit_w + cbo['pad'], MARGIN['bottom'] + cb_dy, cbo['width'], cb_len),
}
W = MARGIN['left'] + orbit_w + cbo['pad'] + cbo['width'] + MARGIN['right']
H = MARGIN['bottom'] + orbit_h + MARGIN['top']


# ---- figure -----------------------------------------------------------------
S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}


# ---- colour and line width, per mode ----------------------------------------
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch, Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, LogLocator, NullFormatter

MSTAR_DATA = SD.mstar_series(R, MSTAR['source'])          # in memory after the first time
mode = MSTAR['mode']
cmap_m = SD.colormap(MSTAR['cmap'])
log_peak = {g: float(e['logm'].max()) for g, e in MSTAR_DATA.items()}     # log10 M*_peak of every galaxy

if mode == 'mstar':
    sc = MSTAR_SCALE
    norm_m   = Normalize(sc['vmin'], sc['vmax'])                  # lines: coloured and interpolated in log10 M* ...
    norm_bar = LogNorm(10 ** sc['vmin'], 10 ** sc['vmax'])        # ... the same mapping, on a log colourbar in Msun
    cbar_label, cbar_ends = sc['cbar_label'], sc['cbar_ends']

    def colour_values(g):
        # draw_orbit (s032 helpers) colours a line by the 'fret' entry it is given: here log10 M*
        e = MSTAR_DATA.get(g)
        return None if e is None else dict(time=e['time'], fret=e['logm'])

elif mode == 'fbound':
    fb = FBOUND
    norm_m = LogNorm(max(fb['vmin'], 1e-6), fb['vmax'], clip=True) if fb['log'] else Normalize(fb['vmin'], fb['vmax'])
    norm_bar = norm_m
    cbar_label, cbar_ends = fb['cbar_label'], fb['cbar_ends']

    def colour_values(g):
        # f_bound = M*(t) / M*_peak (global peak, or the peak so far)
        e = MSTAR_DATA.get(g)
        if e is None:
            return None
        m = 10 ** e['logm']
        peak = np.maximum.accumulate(m) if fb['peak'] == 'running' else m.max()
        return dict(time=e['time'], fret=m / peak)

else:
    raise ValueError(f"MSTAR['mode'] must be 'mstar' or 'fbound', not {mode!r}")


def peak_lw(log_m):
    # mode 'fbound': the line width for a peak stellar mass log10 M*_peak = log_m
    lo, hi = FBOUND['width_mass']
    w0, w1 = FBOUND['width_lw']
    if FBOUND['width_scale'] == 'log':
        x = (log_m - np.log10(lo)) / (np.log10(hi) - np.log10(lo))
    else:
        x = (10 ** log_m - lo) / (hi - lo)
    return w0 + (w1 - w0) * float(np.clip(x, 0, 1))


def line_widths(g, lw, sunk_lw):
    # (lw, sunk_lw) of galaxy g: from its M*_peak in mode 'fbound' (sinking keeps the ratio), else as given
    if mode != 'fbound' or g not in log_peak:
        return lw, sunk_lw
    w = peak_lw(log_peak[g]) * lw / ORBIT_STYLE['lw']
    return w, w * sunk_lw / lw


# ---- drawing, shared by the orbit panel and the zoom inset ------------------
def draw_background(ax, norm):
    # every other satellite's orbit, faint (BACKGROUND; 'fret' colour = this panel's quantity)
    bg = BACKGROUND
    bg_ids = SD.background_ids(R, bg['which'], bg['min_npart'], exclude=ORBIT_IDS)
    bg_orbits = SD.orbits(R, bg_ids)
    for g in bg_ids:
        if g not in bg_orbits:
            continue
        tt, rr = bg_orbits[g]
        pts  = np.stack([tt, rr], axis=1)
        seg  = np.stack([pts[:-1], pts[1:]], axis=1)
        good = np.all(np.isfinite(seg), axis=(1, 2))

        lw_g, sunk_g = (line_widths(g, bg['lw'], bg['sunk_lw']) if FBOUND['width_background']
                        else (bg['lw'], bg['sunk_lw']))
        lw = np.full(tt.size - 1, lw_g)
        t_sink = SD.sinking_time(tt, rr, ORBIT_STYLE['sink_r'])
        if t_sink is not None:
            lw[tt[:-1] >= t_sink] = sunk_g

        if bg['colour'] == 'fret':
            v = SD.fret_along(colour_values(g), tt)
            v_seg = 0.5 * (v[:-1] + v[1:])
            defined = np.isfinite(v_seg)
            colours = cmap_m(norm(np.nan_to_num(v_seg)))
            colours[~defined] = to_rgba(ORBIT_STYLE['undefined_colour'])
            if ORBIT_STYLE['drop_undefined']:
                good &= defined
        else:
            colours = np.tile(to_rgba(bg['colour']), (tt.size - 1, 1))
        colours[:, 3] = bg['alpha']

        ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good], capstyle='butt',
                                         zorder=bg['zorder'], rasterized=bg['rasterized']))


def draw_selected(ax, norm):
    # the selected satellites' orbits, with their drawing order (ORBIT_ZORDER), and the R_vir line
    for g in ORBIT_IDS:
        if g in ORBITS:
            lw_g, sunk_g = line_widths(g, ORBIT_STYLE['lw'], ORBIT_STYLE['sunk_lw'])
            style_g = {**ORBIT_STYLE, 'lw': lw_g, 'sunk_lw': sunk_g}
            draw_orbit(ax, *ORBITS[g], colour_values(g), style_g, cmap_m, norm, np.asarray)
            if g in ORBIT_ZORDER:                               # this satellite's own drawing order
                ax.collections[-1].set_zorder(ORBIT_ZORDER[g])
    ax.axhline(1.0, **ORBIT_STYLE['rvir_line'])


ORBITS = SD.orbits(R, ORBIT_IDS)


# ---- orbit panel: axes as in Comp01 -----------------------------------------
ax_orb = axes['orbit']
t_hi = T_RANGE[1] if T_RANGE[1] is not None else T_NOW
ax_orb.set_xlim(T_RANGE[0], t_hi)
if R_LOG:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(R_MIN, R_MAX)
    ax_orb.yaxis.set_major_locator(FixedLocator(R_TICKS))
    ax_orb.set_yticklabels([f'{r:g}' for r in R_TICKS])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, R_MAX)
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')

if BACKGROUND['show']:
    draw_background(ax_orb, norm_m)
draw_selected(ax_orb, norm_m)

# line-width key (mode 'fbound')
if mode == 'fbound' and FBOUND['legend']['show']:
    masses = FBOUND['legend']['masses']
    handles = [Line2D([], [], color=S.INK2, lw=peak_lw(np.log10(m)), solid_capstyle='butt') for m in masses]
    labels = [rf'$10^{{{np.log10(m):g}}}$' for m in masses]
    ax_orb.legend(handles, labels, title=r'$M_{\star,{\rm peak}}$ [M$_\odot$]', loc=FBOUND['legend']['loc'],
                  ncol=len(masses), fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                  handlelength=2.2, columnspacing=1.2)

# ---- star-formation beads (MBEADS; which births and sizes: BEADS): size by the bin's SFR, fill by the line's value there ----
if MBEADS['show']:
    b, mb = BEADS, MBEADS
    beads = SD.sf_beads(R, ORBIT_IDS, b['dt'], since=b['since'], r_max=R_MAX if b['r_max'] is None else b['r_max'])     # in memory after the first time

    # bead size: the largest no wider than the spacing of neighbouring beads (fit_dt)
    t_axis = ax_orb.get_xlim()
    pt_per_gyr = boxes['orbit'][2] * 72 / (t_axis[1] - t_axis[0])
    d_max = min(b['d_max'], 0.95 * pt_per_gyr * b['dt']) if b['fit_dt'] else b['d_max']
    d_min = min(b['d_min'], d_max)

    def bead_diameter(sfr):
        sfr = np.atleast_1d(np.asarray(sfr, float))
        if b['sizes'] == 'classes':
            cl = np.asarray(b['classes'], float)
            k = np.clip(np.searchsorted(cl, sfr, side='right') - 1, 0, cl.size - 1)
            return d_min + (d_max - d_min) * k / max(cl.size - 1, 1)
        x = np.log10(sfr / b['sfr_min']) / np.log10(b['sfr_max'] / b['sfr_min'])
        return d_min + (d_max - d_min) * np.clip(x, 0, 1)

    # fill: the panel's map and range, or MBEADS' own
    lo_hi = mb['vrange'][mode] or (None, None)
    cmap_b, norm_b, own_b = SD.bead_colours(cmap_m, norm_m, mb['cmap'], *lo_hi)

    for g in ORBIT_IDS:
        if g not in beads or g not in ORBITS:
            continue
        e = beads[g]
        on = e['sfr'] >= b['show_min']
        if not on.any():
            continue
        tt, rr = ORBITS[g]
        ok = np.isfinite(tt) & np.isfinite(rr)
        t_b = e['t'][on]
        r_b = np.interp(t_b, tt[ok], rr[ok], left=np.nan, right=np.nan)    # on the orbit at the bead's time
        vis = np.isfinite(r_b)
        v = SD.fret_along(colour_values(g), t_b[vis])                      # the line's value there
        colours = cmap_b(norm_b(np.nan_to_num(v)))
        colours[~np.isfinite(v)] = to_rgba(b['undefined_colour'])
        ax_orb.scatter(t_b[vis], r_b[vis], s=bead_diameter(e['sfr'][on][vis]) ** 2, c=colours,
                       edgecolors=b['edge']['color'], linewidths=b['edge']['lw'], zorder=ORBIT_ZORDER.get(g, 3) + b['zorder'] - 3)

    def add_key(handles, labels, title, loc):
        # one more legend on the orbit panel, keeping those already there (e.g. fbound's line widths)
        previous = ax_orb.get_legend()
        ax_orb.legend(handles, labels, title=title, loc=loc, ncol=len(handles),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
        if previous is not None:
            ax_orb.add_artist(previous)

    # size key: grey beads at BEADS' SFR values
    if mb['legend']['show']:
        values = b['classes'] if b['sizes'] == 'classes' else b['legend']['values']
        handles = [ax_orb.scatter([], [], s=bead_diameter(v)[0] ** 2, c='#9a9a9a',
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in values]
        add_key(handles, [f'{v:g}' for v in values], r'SFR [M$_\odot$/yr]', mb['legend']['loc'])

    # colour key (own map only): middle-sized beads coloured at these values
    ck = mb['colour_key']
    if own_b and ck['show']:
        if ck['values'] is not None:
            x = np.log10(np.asarray(ck['values'], float)) if mode == 'mstar' else np.asarray(ck['values'], float)
        else:
            x = np.asarray(norm_b.inverse(np.linspace(0, 1, ck['n'])), float)
        labels = [rf'$10^{{{v:.2g}}}$' for v in x] if mode == 'mstar' else [f'{v:.2g}' for v in x]
        d_key = 0.5 * (d_min + d_max)
        handles = [ax_orb.scatter([], [], s=d_key ** 2, color=cmap_b(norm_b(v)),
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in x]
        add_key(handles, labels, ck['title'] or cbar_label, ck['loc'])

# redshift ticks on top
if ORBIT_STYLE['z_axis']:
    snaps   = sorted(SNAP_Z)
    zz      = np.array([SNAP_Z[s_] for s_ in snaps])
    tt      = np.array([SNAP_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in Z_TICKS if zz.min() <= z <= zz.max()]

    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=FONTS['label'], labelpad=2)


# ---- colourbar --------------------------------------------------------------
cax = axes['cbar M*']
cb_m = S.colorbar(fig, ScalarMappable(norm=norm_bar, cmap=cmap_m), cax=cax, label=cbar_label, ends=False)
if mode == 'mstar' and MSTAR_SCALE['ticks'] is not None:
    cb_m.set_ticks(list(MSTAR_SCALE['ticks']))
if cbar_ends:
    gap = NOTE_GAP / boxes['cbar M*'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, cbar_ends[0], va='bottom', **end_style)
    cax.text(0.5, -gap, cbar_ends[1], va='top', **end_style)


# ---- zoom inset (INSET): a close-up of one region (the mark's aspect ratio), with its own colour range ----
inset_axes = []
if INSET['show']:
    ins = INSET
    s1, s2 = ins['snaps']
    r_lo, r_hi = ins['r']
    t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]

    # where: ins['size'] inches wide, as tall as the mark's aspect ratio asks (the mark's height / width on the
    # orbit panel, inches), its lower-left corner at ins['pos'] (fractions of the orbit panel)
    ox, oy, ow, oh = boxes['orbit']
    (t0_ax, t1_ax), (y0_ax, y1_ax) = ax_orb.get_xlim(), ax_orb.get_ylim()
    w_mark = (t_hi_ - t_lo) / (t1_ax - t0_ax) * ow
    if R_LOG:
        h_mark = np.log10(r_hi / r_lo) / np.log10(y1_ax / y0_ax) * oh
    else:
        h_mark = (r_hi - r_lo) / (y1_ax - y0_ax) * oh
    x_in, y_in = ox + ins['pos'][0] * ow, oy + ins['pos'][1] * oh
    w_in = ins['size']
    h_in = w_in * h_mark / w_mark
    ax_in = fig.add_axes([x_in / W, y_in / H, w_in / W, h_in / H])
    ax_in.set_facecolor('white')
    inset_axes.append(ax_in)

    # its own colour range, in the mode's units (log10 M* or f_bound)
    v0, v1 = ins['vrange'][mode]
    norm_in = Normalize(v0, v1)
    norm_in_bar = LogNorm(10 ** v0, 10 ** v1) if mode == 'mstar' else norm_in

    # the same lines as the orbit panel, with the inset's colours
    if BACKGROUND['show'] and ins['background']:
        draw_background(ax_in, norm_in)
    draw_selected(ax_in, norm_in)
    ax_in.set_xlim(t_lo, t_hi_)
    if R_LOG:
        ax_in.set_yscale('log')
        ax_in.yaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
        ax_in.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
        ax_in.yaxis.set_minor_formatter(NullFormatter())
    ax_in.set_ylim(r_lo, r_hi)

    # its colourbar, right of it
    cb_in = ins['cbar']
    if cb_in['show']:
        L = cb_in['length'] * h_in
        cax_in = fig.add_axes([(x_in + w_in + cb_in['pad']) / W, (y_in + (h_in - L) / 2) / H, cb_in['width'] / W, L / H])
        S.colorbar(fig, ScalarMappable(norm=norm_in_bar, cmap=cmap_m), cax=cax_in,
                   label=cbar_label if cb_in['label'] else '', ends=False)
        inset_axes.append(cax_in)

    # its mark on the orbit panel, and the stems (INSET['stems']): 'down', bottom corners of the mark -> top corners
    # of the inset; 'up', top corners of the mark -> bottom corners of the inset
    box_style  = {**MARK['box'], **MARK_PANEL.get('inset', {}).get('box', {})}
    link_style = {**MARK['link'], **MARK_PANEL.get('inset', {}).get('link', {})}
    ax_orb.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
    stems = ins.get('stems', 'up')
    if stems not in ('up', 'down'):
        raise ValueError(f"INSET['stems'] must be 'up' or 'down', not {stems!r}")
    r_mark, y_inset = (r_lo, 1.0) if stems == 'down' else (r_hi, 0.0)
    for x_corner, t_corner in ((0.0, t_lo), (1.0, t_hi_)):
        fig.add_artist(ConnectionPatch(xyA=(x_corner, y_inset), coordsA=ax_in.transAxes,
                                       xyB=(t_corner, r_mark), coordsB=ax_orb.transData, **link_style))


# ---- common text sizes: axis labels and tick labels everywhere (the inset: INSET['fontsize'] if set) ----
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=FONTS['ticks'])
    ax.xaxis.label.set_size(FONTS['label'])
    ax.yaxis.label.set_size(FONTS['label'])
if INSET['fontsize'] is not None:
    for ax in inset_axes:
        ax.tick_params(which='both', labelsize=INSET['fontsize'])


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, R.stem(f"{MSTAR['name']}_{mode}_{ORBIT_SET}"), SAVE)

## Comp01c: the Comp01 orbit panel coloured by dark-matter mass

As Comp01b, with each orbit coloured by its dark-matter mass from the merger tree (`DMASS['source']`: virial mass by default), on a log scale. Beads as in Comp01b (`MBEADS`): one per `BEADS['dt']` bin with star formation, size by its SFR, fill by the line's M_DM where it lands. Everything else from the Comp01 parameters; needs the s032 helpers cell, the parameter cells and the processing cell above.

In [ ]:
# Plotting  ·  Comp01c figure: the orbit panel coloured by dark-matter mass (DMASS; beads: MBEADS; drawing order: ORBIT_ZORDER)

# ---- geometry: the Comp01 orbit panel and its colourbar (SIZE, CBAR['orbit'], MARGIN) ----
orbit_w = SIZE['orbit_w'] if SIZE['orbit_w'] is not None else SIZE['zoom_w'] + GAP['cols'] + SIZE['map_size']
orbit_h = SIZE['orbit_h']
cbo     = CBAR['orbit']
cb_len  = cbo['length'] * orbit_h
cb_dy   = {'bottom': 0.0, 'center': (orbit_h - cb_len) / 2, 'top': orbit_h - cb_len}[cbo['align']]

boxes = {
    'orbit':    (MARGIN['left'], MARGIN['bottom'], orbit_w, orbit_h),
    'cbar DM':  (MARGIN['left'] + orbit_w + cbo['pad'], MARGIN['bottom'] + cb_dy, cbo['width'], cb_len),
}
W = MARGIN['left'] + orbit_w + cbo['pad'] + cbo['width'] + MARGIN['right']
H = MARGIN['bottom'] + orbit_h + MARGIN['top']


# ---- figure -----------------------------------------------------------------
S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}

from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator, NullFormatter


# ---- colour: log10 M_DM -----------------------------------------------------
DM_DATA = SD.dm_series(R, DMASS['source'])                  # in memory after the first time
cmap_d   = SD.colormap(DMASS['cmap'])
norm_d   = Normalize(DMASS['vmin'], DMASS['vmax'])                    # lines: coloured and interpolated in log10 M_DM ...
norm_bar = LogNorm(10 ** DMASS['vmin'], 10 ** DMASS['vmax'])          # ... the same mapping, on a log colourbar in Msun


def colour_values(g):
    # draw_orbit (s032 helpers) colours a line by the 'fret' entry it is given: here log10 M_DM
    e = DM_DATA.get(g)
    return None if e is None else dict(time=e['time'], fret=e['logm'])


def draw_background(ax, norm):
    # every other satellite's orbit, faint (BACKGROUND; 'fret' colour = its own dark-matter mass)
    bg = BACKGROUND
    bg_ids = SD.background_ids(R, bg['which'], bg['min_npart'], exclude=ORBIT_IDS)
    bg_orbits = SD.orbits(R, bg_ids)
    for g in bg_ids:
        if g not in bg_orbits:
            continue
        tt, rr = bg_orbits[g]
        pts  = np.stack([tt, rr], axis=1)
        seg  = np.stack([pts[:-1], pts[1:]], axis=1)
        good = np.all(np.isfinite(seg), axis=(1, 2))

        lw = np.full(tt.size - 1, bg['lw'])
        t_sink = SD.sinking_time(tt, rr, ORBIT_STYLE['sink_r'])
        if t_sink is not None:
            lw[tt[:-1] >= t_sink] = bg['sunk_lw']

        if bg['colour'] == 'fret':
            v = SD.fret_along(colour_values(g), tt)
            v_seg = 0.5 * (v[:-1] + v[1:])
            defined = np.isfinite(v_seg)
            colours = cmap_d(norm(np.nan_to_num(v_seg)))
            colours[~defined] = to_rgba(ORBIT_STYLE['undefined_colour'])
            if ORBIT_STYLE['drop_undefined']:
                good &= defined
        else:
            colours = np.tile(to_rgba(bg['colour']), (tt.size - 1, 1))
        colours[:, 3] = bg['alpha']

        ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good], capstyle='butt',
                                         zorder=bg['zorder'], rasterized=bg['rasterized']))


def draw_selected(ax, norm):
    # the selected satellites' orbits, with their drawing order (ORBIT_ZORDER), and the R_vir line
    for g in ORBIT_IDS:
        if g in ORBITS:
            draw_orbit(ax, *ORBITS[g], colour_values(g), ORBIT_STYLE, cmap_d, norm, np.asarray)
            if g in ORBIT_ZORDER:                               # this satellite's own drawing order
                ax.collections[-1].set_zorder(ORBIT_ZORDER[g])
    ax.axhline(1.0, **ORBIT_STYLE['rvir_line'])


ORBITS = SD.orbits(R, ORBIT_IDS)


# ---- orbit panel: axes as in Comp01 -----------------------------------------
ax_orb = axes['orbit']
t_hi = T_RANGE[1] if T_RANGE[1] is not None else T_NOW
ax_orb.set_xlim(T_RANGE[0], t_hi)
if R_LOG:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(R_MIN, R_MAX)
    ax_orb.yaxis.set_major_locator(FixedLocator(R_TICKS))
    ax_orb.set_yticklabels([f'{r:g}' for r in R_TICKS])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, R_MAX)
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')

if BACKGROUND['show']:
    draw_background(ax_orb, norm_d)
draw_selected(ax_orb, norm_d)


# ---- star-formation beads (MBEADS; which births and sizes: BEADS): size by the bin's SFR, fill by the line's value there ----
if MBEADS['show']:
    b, mb = BEADS, MBEADS
    beads = SD.sf_beads(R, ORBIT_IDS, b['dt'], since=b['since'], r_max=R_MAX if b['r_max'] is None else b['r_max'])     # in memory after the first time

    # bead size: the largest no wider than the spacing of neighbouring beads (fit_dt)
    t_axis = ax_orb.get_xlim()
    pt_per_gyr = boxes['orbit'][2] * 72 / (t_axis[1] - t_axis[0])
    d_max = min(b['d_max'], 0.95 * pt_per_gyr * b['dt']) if b['fit_dt'] else b['d_max']
    d_min = min(b['d_min'], d_max)

    def bead_diameter(sfr):
        sfr = np.atleast_1d(np.asarray(sfr, float))
        if b['sizes'] == 'classes':
            cl = np.asarray(b['classes'], float)
            k = np.clip(np.searchsorted(cl, sfr, side='right') - 1, 0, cl.size - 1)
            return d_min + (d_max - d_min) * k / max(cl.size - 1, 1)
        x = np.log10(sfr / b['sfr_min']) / np.log10(b['sfr_max'] / b['sfr_min'])
        return d_min + (d_max - d_min) * np.clip(x, 0, 1)

    # fill: the panel's map and range, or MBEADS' own
    lo_hi = mb['vrange']['dm'] or (None, None)
    cmap_b, norm_b, own_b = SD.bead_colours(cmap_d, norm_d, mb['cmap'], *lo_hi)

    for g in ORBIT_IDS:
        if g not in beads or g not in ORBITS:
            continue
        e = beads[g]
        on = e['sfr'] >= b['show_min']
        if not on.any():
            continue
        tt, rr = ORBITS[g]
        ok = np.isfinite(tt) & np.isfinite(rr)
        t_b = e['t'][on]
        r_b = np.interp(t_b, tt[ok], rr[ok], left=np.nan, right=np.nan)    # on the orbit at the bead's time
        vis = np.isfinite(r_b)
        v = SD.fret_along(colour_values(g), t_b[vis])                      # the line's value there
        colours = cmap_b(norm_b(np.nan_to_num(v)))
        colours[~np.isfinite(v)] = to_rgba(b['undefined_colour'])
        ax_orb.scatter(t_b[vis], r_b[vis], s=bead_diameter(e['sfr'][on][vis]) ** 2, c=colours,
                       edgecolors=b['edge']['color'], linewidths=b['edge']['lw'], zorder=ORBIT_ZORDER.get(g, 3) + b['zorder'] - 3)

    def add_key(handles, labels, title, loc):
        # one more legend on the orbit panel, keeping those already there (e.g. fbound's line widths)
        previous = ax_orb.get_legend()
        ax_orb.legend(handles, labels, title=title, loc=loc, ncol=len(handles),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
        if previous is not None:
            ax_orb.add_artist(previous)

    # size key: grey beads at BEADS' SFR values
    if mb['legend']['show']:
        values = b['classes'] if b['sizes'] == 'classes' else b['legend']['values']
        handles = [ax_orb.scatter([], [], s=bead_diameter(v)[0] ** 2, c='#9a9a9a',
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in values]
        add_key(handles, [f'{v:g}' for v in values], r'SFR [M$_\odot$/yr]', mb['legend']['loc'])

    # colour key (own map only): middle-sized beads coloured at these values
    ck = mb['colour_key']
    if own_b and ck['show']:
        if ck['values'] is not None:
            x = np.log10(np.asarray(ck['values'], float)) if True else np.asarray(ck['values'], float)
        else:
            x = np.asarray(norm_b.inverse(np.linspace(0, 1, ck['n'])), float)
        labels = [rf'$10^{{{v:.2g}}}$' for v in x] if True else [f'{v:.2g}' for v in x]
        d_key = 0.5 * (d_min + d_max)
        handles = [ax_orb.scatter([], [], s=d_key ** 2, color=cmap_b(norm_b(v)),
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in x]
        add_key(handles, labels, ck['title'] or DMASS['cbar_label'], ck['loc'])

# redshift ticks on top
if ORBIT_STYLE['z_axis']:
    snaps   = sorted(SNAP_Z)
    zz      = np.array([SNAP_Z[s_] for s_ in snaps])
    tt      = np.array([SNAP_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in Z_TICKS if zz.min() <= z <= zz.max()]

    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=FONTS['label'], labelpad=2)


# ---- colourbar --------------------------------------------------------------
cax = axes['cbar DM']
cb_d = S.colorbar(fig, ScalarMappable(norm=norm_bar, cmap=cmap_d), cax=cax, label=DMASS['cbar_label'], ends=False)
if DMASS['ticks'] is not None:
    cb_d.set_ticks(list(DMASS['ticks']))
if DMASS['cbar_ends']:
    gap = NOTE_GAP / boxes['cbar DM'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, DMASS['cbar_ends'][0], va='bottom', **end_style)
    cax.text(0.5, -gap, DMASS['cbar_ends'][1], va='top', **end_style)


# ---- common text sizes: axis labels and tick labels everywhere --------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=FONTS['ticks'])
    ax.xaxis.label.set_size(FONTS['label'])
    ax.yaxis.label.set_size(FONTS['label'])


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, R.stem(f"{DMASS['name']}_{ORBIT_SET}"), SAVE)

## Comp01d: the three orbit panels stacked (f_ret, M*, M_DM)

Comp01's orbit panel (f_ret) on top, Comp01b's (stellar mass, `MSTAR['mode']`) in the middle, Comp01c's (dark-matter mass) at the bottom: each panel, its colourbar, lines, background, beads and keys exactly as in its own figure (same parameters), stacked `COMP01D['gap']` apart. No zoom inset and none of Comp01's map / zoom marks; Comp01b's zoom region is marked in all three panels when `INSET['comp01']` is on, with the note `INSET['comp01_text']` under it. Needs the s032 helpers cell, the parameter cells and the processing cell above.

In [ ]:
# Plotting  ·  Comp01d figure: Comp01's, Comp01b's and Comp01c's orbit panels stacked (f_ret, M*, M_DM; COMP01D)

from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, NullFormatter

# ---- geometry: three Comp01 orbit panels and their colourbars (SIZE, CBAR['orbit'], MARGIN), top to bottom ----
ROWS    = ('f_ret', 'M*', 'M_DM')
orbit_w = SIZE['orbit_w'] if SIZE['orbit_w'] is not None else SIZE['zoom_w'] + GAP['cols'] + SIZE['map_size']
orbit_h = SIZE['orbit_h']
cbo     = CBAR['orbit']
cb_len  = cbo['length'] * orbit_h
cb_dy   = {'bottom': 0.0, 'center': (orbit_h - cb_len) / 2, 'top': orbit_h - cb_len}[cbo['align']]

boxes = {}
for k, name in enumerate(ROWS):
    y = MARGIN['bottom'] + (len(ROWS) - 1 - k) * (orbit_h + COMP01D['gap'])
    boxes[name] = (MARGIN['left'], y, orbit_w, orbit_h)
    boxes[f'cbar {name}'] = (MARGIN['left'] + orbit_w + cbo['pad'], y + cb_dy, cbo['width'], cb_len)
W = MARGIN['left'] + orbit_w + cbo['pad'] + cbo['width'] + MARGIN['right']
H = MARGIN['bottom'] + len(ROWS) * orbit_h + (len(ROWS) - 1) * COMP01D['gap'] + MARGIN['top']

S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}
ORBITS = SD.orbits(R, ORBIT_IDS)


# ---- what each panel shows: its values per satellite ('fret' entry: draw_orbit colours by it), colours, line
#      widths, colourbar and beads, each as in its own figure ----
def same_widths(g, lw, sunk_lw):
    return lw, sunk_lw


# top: f_ret (Comp01; ORBIT_STYLE)
cmap_f = SD.colormap(ORBIT_STYLE['cmap'])
fret_log = ORBIT_STYLE.get('fret_log', False)
norm_f = LogNorm(ORBIT_STYLE['fret_min'], 1.0, clip=True) if fret_log else Normalize(0.0, 1.0)
PANELS = {'f_ret': dict(
    values=lambda g: D032['sats'].get(g), cmap=cmap_f, norm=norm_f, norm_bar=norm_f,
    widths=same_widths, widths_background=False,
    cbar_label=ORBIT_STYLE['cbar_label'], ticks=None if fret_log else [0, 0.25, 0.5, 0.75, 1.0],
    cbar_ends=ORBIT_STYLE['cbar_ends'],
    beads=BEADS['show'], bead_fill='newborn', bead_colours=SD.bead_colours(cmap_f, norm_f, BEADS['cmap'], BEADS['vmin'], BEADS['vmax']),
    size_key=BEADS['legend'], colour_key=BEADS['colour_key'], key_log=False, key_title=BEADS['colour_key']['title'],
)}

# middle: stellar mass (Comp01b; MSTAR['mode'], MSTAR_SCALE or FBOUND)
MSTAR_DATA = SD.mstar_series(R, MSTAR['source'])          # in memory after the first time
mode = MSTAR['mode']
cmap_m = SD.colormap(MSTAR['cmap'])
log_peak = {g: float(e['logm'].max()) for g, e in MSTAR_DATA.items()}     # log10 M*_peak of every galaxy

if mode == 'mstar':
    sc = MSTAR_SCALE
    norm_m, norm_m_bar = Normalize(sc['vmin'], sc['vmax']), LogNorm(10 ** sc['vmin'], 10 ** sc['vmax'])
    label_m, ends_m, ticks_m = sc['cbar_label'], sc['cbar_ends'], sc['ticks']

    def mstar_values(g):
        # log10 M*
        e = MSTAR_DATA.get(g)
        return None if e is None else dict(time=e['time'], fret=e['logm'])

elif mode == 'fbound':
    fb = FBOUND
    norm_m = LogNorm(max(fb['vmin'], 1e-6), fb['vmax'], clip=True) if fb['log'] else Normalize(fb['vmin'], fb['vmax'])
    norm_m_bar = norm_m
    label_m, ends_m, ticks_m = fb['cbar_label'], fb['cbar_ends'], None

    def mstar_values(g):
        # f_bound = M*(t) / M*_peak (global peak, or the peak so far)
        e = MSTAR_DATA.get(g)
        if e is None:
            return None
        m = 10 ** e['logm']
        peak = np.maximum.accumulate(m) if fb['peak'] == 'running' else m.max()
        return dict(time=e['time'], fret=m / peak)

else:
    raise ValueError(f"MSTAR['mode'] must be 'mstar' or 'fbound', not {mode!r}")


def peak_lw(log_m):
    # mode 'fbound': the line width for a peak stellar mass log10 M*_peak = log_m
    lo, hi = FBOUND['width_mass']
    w0, w1 = FBOUND['width_lw']
    if FBOUND['width_scale'] == 'log':
        x = (log_m - np.log10(lo)) / (np.log10(hi) - np.log10(lo))
    else:
        x = (10 ** log_m - lo) / (hi - lo)
    return w0 + (w1 - w0) * float(np.clip(x, 0, 1))


def mstar_widths(g, lw, sunk_lw):
    # (lw, sunk_lw) of galaxy g: from its M*_peak in mode 'fbound' (sinking keeps the ratio), else as given
    if mode != 'fbound' or g not in log_peak:
        return lw, sunk_lw
    w = peak_lw(log_peak[g]) * lw / ORBIT_STYLE['lw']
    return w, w * sunk_lw / lw


PANELS['M*'] = dict(
    values=mstar_values, cmap=cmap_m, norm=norm_m, norm_bar=norm_m_bar,
    widths=mstar_widths, widths_background=FBOUND['width_background'],
    cbar_label=label_m, ticks=ticks_m, cbar_ends=ends_m,
    beads=MBEADS['show'], bead_fill='line',
    bead_colours=SD.bead_colours(cmap_m, norm_m, MBEADS['cmap'], *(MBEADS['vrange'][mode] or (None, None))),
    size_key=MBEADS['legend'], colour_key=MBEADS['colour_key'], key_log=mode == 'mstar',
    key_title=MBEADS['colour_key']['title'] or label_m,
)

# bottom: dark-matter mass (Comp01c; DMASS)
DM_DATA = SD.dm_series(R, DMASS['source'])                  # in memory after the first time
cmap_d = SD.colormap(DMASS['cmap'])
norm_d = Normalize(DMASS['vmin'], DMASS['vmax'])
PANELS['M_DM'] = dict(
    values=lambda g: (None if DM_DATA.get(g) is None else dict(time=DM_DATA[g]['time'], fret=DM_DATA[g]['logm'])),
    cmap=cmap_d, norm=norm_d, norm_bar=LogNorm(10 ** DMASS['vmin'], 10 ** DMASS['vmax']),
    widths=same_widths, widths_background=False,
    cbar_label=DMASS['cbar_label'], ticks=DMASS['ticks'], cbar_ends=DMASS['cbar_ends'],
    beads=MBEADS['show'], bead_fill='line',
    bead_colours=SD.bead_colours(cmap_d, norm_d, MBEADS['cmap'], *(MBEADS['vrange']['dm'] or (None, None))),
    size_key=MBEADS['legend'], colour_key=MBEADS['colour_key'], key_log=True,
    key_title=MBEADS['colour_key']['title'] or DMASS['cbar_label'],
)


# ---- drawing, shared by the three panels ---------------------------------------
def draw_background(ax, p):
    # every other satellite's orbit, faint (BACKGROUND; 'fret' colour = this panel's quantity)
    bg = BACKGROUND
    bg_ids = SD.background_ids(R, bg['which'], bg['min_npart'], exclude=ORBIT_IDS)
    bg_orbits = SD.orbits(R, bg_ids)
    for g in bg_ids:
        if g not in bg_orbits:
            continue
        tt, rr = bg_orbits[g]
        pts  = np.stack([tt, rr], axis=1)
        seg  = np.stack([pts[:-1], pts[1:]], axis=1)
        good = np.all(np.isfinite(seg), axis=(1, 2))

        lw_g, sunk_g = (p['widths'](g, bg['lw'], bg['sunk_lw']) if p['widths_background']
                        else (bg['lw'], bg['sunk_lw']))
        lw = np.full(tt.size - 1, lw_g)
        t_sink = SD.sinking_time(tt, rr, ORBIT_STYLE['sink_r'])
        if t_sink is not None:
            lw[tt[:-1] >= t_sink] = sunk_g

        if bg['colour'] == 'fret':
            v = SD.fret_along(p['values'](g), tt)
            v_seg = 0.5 * (v[:-1] + v[1:])
            defined = np.isfinite(v_seg)
            colours = p['cmap'](p['norm'](np.nan_to_num(v_seg)))
            colours[~defined] = to_rgba(ORBIT_STYLE['undefined_colour'])
            if ORBIT_STYLE['drop_undefined']:
                good &= defined
        else:
            colours = np.tile(to_rgba(bg['colour']), (tt.size - 1, 1))
        colours[:, 3] = bg['alpha']

        ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good], capstyle='butt',
                                         zorder=bg['zorder'], rasterized=bg['rasterized']))


def draw_selected(ax, p):
    # the selected satellites' orbits, with their drawing order (ORBIT_ZORDER), and the R_vir line
    for g in ORBIT_IDS:
        if g in ORBITS:
            lw_g, sunk_g = p['widths'](g, ORBIT_STYLE['lw'], ORBIT_STYLE['sunk_lw'])
            draw_orbit(ax, *ORBITS[g], p['values'](g), {**ORBIT_STYLE, 'lw': lw_g, 'sunk_lw': sunk_g},
                       p['cmap'], p['norm'], np.asarray)
            if g in ORBIT_ZORDER:                               # this satellite's own drawing order
                ax.collections[-1].set_zorder(ORBIT_ZORDER[g])
    ax.axhline(1.0, **ORBIT_STYLE['rvir_line'])


def add_key(ax, handles, labels, title, loc):
    # one more legend on a panel, keeping those already there (e.g. fbound's line widths)
    previous = ax.get_legend()
    ax.legend(handles, labels, title=title, loc=loc, ncol=len(handles),
              fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
              handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
    if previous is not None:
        ax.add_artist(previous)


# beads (BEADS: which births, sizes; fill: the newborns' f_ret on top, the line's value where it lands below)
b = BEADS
beads = SD.sf_beads(R, ORBIT_IDS, b['dt'], since=b['since'], r_max=R_MAX if b['r_max'] is None else b['r_max'])     # in memory after the first time
t_lim = (T_RANGE[0], T_RANGE[1] if T_RANGE[1] is not None else T_NOW)
pt_per_gyr = orbit_w * 72 / (t_lim[1] - t_lim[0])
d_max = min(b['d_max'], 0.95 * pt_per_gyr * b['dt']) if b['fit_dt'] else b['d_max']   # no wider than their spacing
d_min = min(b['d_min'], d_max)


def bead_diameter(sfr):
    sfr = np.atleast_1d(np.asarray(sfr, float))
    if b['sizes'] == 'classes':
        cl = np.asarray(b['classes'], float)
        k = np.clip(np.searchsorted(cl, sfr, side='right') - 1, 0, cl.size - 1)
        return d_min + (d_max - d_min) * k / max(cl.size - 1, 1)
    x = np.log10(sfr / b['sfr_min']) / np.log10(b['sfr_max'] / b['sfr_min'])
    return d_min + (d_max - d_min) * np.clip(x, 0, 1)


def draw_beads(ax, p, keys):
    cmap_b, norm_b, own_b = p['bead_colours']
    for g in ORBIT_IDS:
        if g not in beads or g not in ORBITS:
            continue
        e = beads[g]
        on = e['sfr'] >= b['show_min']
        if not on.any():
            continue
        tt, rr = ORBITS[g]
        ok = np.isfinite(tt) & np.isfinite(rr)
        t_b = e['t'][on]
        r_b = np.interp(t_b, tt[ok], rr[ok], left=np.nan, right=np.nan)    # on the orbit at the bead's time
        vis = np.isfinite(r_b)
        v = e['f_new'][on][vis] if p['bead_fill'] == 'newborn' else SD.fret_along(p['values'](g), t_b[vis])
        colours = cmap_b(norm_b(np.nan_to_num(v)))
        colours[~np.isfinite(v)] = to_rgba(b['undefined_colour'])
        ax.scatter(t_b[vis], r_b[vis], s=bead_diameter(e['sfr'][on][vis]) ** 2, c=colours,
                   edgecolors=b['edge']['color'], linewidths=b['edge']['lw'], zorder=ORBIT_ZORDER.get(g, 3) + b['zorder'] - 3)
    if not keys:
        return

    # size key: grey beads at BEADS' SFR values
    if p['size_key']['show']:
        values = b['classes'] if b['sizes'] == 'classes' else b['legend']['values']
        handles = [ax.scatter([], [], s=bead_diameter(v)[0] ** 2, c='#9a9a9a',
                              edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in values]
        add_key(ax, handles, [f'{v:g}' for v in values], r'SFR [M$_\odot$/yr]', p['size_key']['loc'])

    # colour key (own map only): middle-sized beads coloured at these values
    ck = p['colour_key']
    if own_b and ck['show']:
        if ck['values'] is not None:
            x = np.log10(np.asarray(ck['values'], float)) if p['key_log'] else np.asarray(ck['values'], float)
        else:
            x = np.asarray(norm_b.inverse(np.linspace(0, 1, ck['n'])), float)
        labels = [rf'$10^{{{v:.2g}}}$' for v in x] if p['key_log'] else [f'{v:.2g}' for v in x]
        d_key = 0.5 * (d_min + d_max)
        handles = [ax.scatter([], [], s=d_key ** 2, color=cmap_b(norm_b(v)),
                              edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in x]
        add_key(ax, handles, labels, p['key_title'], ck['loc'])


# ---- the three panels ------------------------------------------------------------
share = COMP01D['share_x']
if COMP01D['bead_keys'] not in ('all', 'top', 'none'):
    raise ValueError(f"COMP01D['bead_keys'] must be 'all', 'top' or 'none', not {COMP01D['bead_keys']!r}")

for k, name in enumerate(ROWS):
    p, ax = PANELS[name], axes[name]
    first, last = k == 0, k == len(ROWS) - 1

    # axes as in Comp01
    ax.set_xlim(*t_lim)
    if R_LOG:
        ax.set_yscale('log')
        ax.set_ylim(R_MIN, R_MAX)
        ax.yaxis.set_major_locator(FixedLocator(R_TICKS))
        ax.set_yticklabels([f'{r:g}' for r in R_TICKS])
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.set_ylim(0.0, R_MAX)
    ax.set_ylabel(r'$r/R_{\rm vir}$')
    if last or not share:
        ax.set_xlabel('cosmic time [Gyr]')
    else:
        ax.tick_params(axis='x', which='both', labelbottom=False)

    # orbits, background, beads
    if BACKGROUND['show']:
        draw_background(ax, p)
    draw_selected(ax, p)
    if name == 'M*' and mode == 'fbound' and FBOUND['legend']['show']:      # line-width key, as in Comp01b
        masses = FBOUND['legend']['masses']
        handles = [Line2D([], [], color=S.INK2, lw=peak_lw(np.log10(m_)), solid_capstyle='butt') for m_ in masses]
        ax.legend(handles, [rf'$10^{{{np.log10(m_):g}}}$' for m_ in masses], title=r'$M_{\star,{\rm peak}}$ [M$_\odot$]',
                  loc=FBOUND['legend']['loc'], ncol=len(masses), fontsize=FONTS['notes'], title_fontsize=FONTS['notes'],
                  frameon=False, handlelength=2.2, columnspacing=1.2)
    if p['beads']:
        draw_beads(ax, p, keys=COMP01D['bead_keys'] == 'all' or (COMP01D['bead_keys'] == 'top' and first))

    # Comp01b's zoom region (INSET['comp01']): its mark, and the note under it
    if INSET['comp01']:
        s1, s2 = INSET['snaps']
        r_lo, r_hi = INSET['r']
        t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]
        box_style = {**MARK['box'], **MARK_PANEL.get('inset', {}).get('box', {})}
        ax.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
        if INSET.get('comp01_text'):
            ax.annotate(INSET['comp01_text'], xy=(0.5 * (t_lo + t_hi_), r_lo), xycoords='data',
                        xytext=(0, -1), textcoords='offset points', ha='center', va='top',
                        fontsize=FONTS['notes'], color=box_style.get('ec', S.INK2), zorder=box_style.get('zorder', 5))

    # redshift ticks on top (shared time axis: the top panel only)
    if ORBIT_STYLE['z_axis'] and (first or not share):
        snaps   = sorted(SNAP_Z)
        zz      = np.array([SNAP_Z[s_] for s_ in snaps])
        tt      = np.array([SNAP_TIME[s_] for s_ in snaps])
        o       = np.argsort(zz)
        z_ticks = [z for z in Z_TICKS if zz.min() <= z <= zz.max()]
        ax.tick_params(axis='x', which='both', top=False)
        top = ax.secondary_xaxis('top')
        top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
        top.set_xticklabels([f'{z:g}' for z in z_ticks])
        top.minorticks_off()
        top.tick_params(labelsize=FONTS['ticks'])
        top.set_xlabel(r'$z$', fontsize=FONTS['label'], labelpad=2)

    # its colourbar, with the notes beyond its ends
    cax = axes[f'cbar {name}']
    cb_ = S.colorbar(fig, ScalarMappable(norm=p['norm_bar'], cmap=p['cmap']), cax=cax, label=p['cbar_label'], ends=False)
    if p['ticks'] is not None:
        cb_.set_ticks(list(p['ticks']))
    if p['cbar_ends']:
        gap = NOTE_GAP / boxes[f'cbar {name}'][3]
        end_style = dict(transform=cax.transAxes, ha='center', fontsize=FONTS['notes'], color=S.INK2)
        cax.text(0.5, 1 + gap, p['cbar_ends'][0], va='bottom', **end_style)
        cax.text(0.5, -gap, p['cbar_ends'][1], va='top', **end_style)


# ---- common text sizes: axis labels and tick labels everywhere --------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=FONTS['ticks'])
    ax.xaxis.label.set_size(FONTS['label'])
    ax.yaxis.label.set_size(FONTS['label'])


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, R.stem(f"{COMP01D['name']}_{mode}_{ORBIT_SET}"), SAVE)

## Comp01: satellite orbits (s032) + map (s01) + two satellite zooms

Parameters and data: the cells above, shared with Comp01b (the Comp01 parameters: layout, and what each bottom panel shows: the map's snapshot, each zoom's snapshot and r/R_vir range, which also place the marks on the orbit panel). Here the figure, then the same figure mirrored (map on the left). The orbit set is picked by name (ORBIT_SET); each zoom panel shows the KDE (or step histogram) of log10(E_H/E_S) of the satellites' stars in its box (snapshot range x r/R_vir range), all and newborn: by `FRET` (first cell), those in a satellite's infall list and bound to it, or those it owns and binds.

Star-formation beads on the orbits (BEADS): one per `dt` bin where a satellite forms stars, from the exact birth times and initial masses (`particle_data/birth_times.hdf5`); size ~ SFR, fill = the newborns' own f_ret.

Background (BACKGROUND): every other satellite's orbit, faint, under the selected ones (no beads).

Comp01b's zoom region (`INSET['comp01']`, Comp01b parameters): its mark on the orbit panel and a note under it (`INSET['comp01_text']`), no inset.

In [ ]:
# Plotting  ·  Comp01 figure (orbits, map and zooms)

# ---- geometry: every box in inches from the figure's bottom-left corner -----
s, g, m = SIZE, GAP, MARGIN

bottom_w = s['zoom_w'] + g['cols'] + s['map_size']
orbit_w  = s['orbit_w'] if s['orbit_w'] is not None else bottom_w
stack_h  = 2 * s['zoom_h'] + g['zooms']
bottom_h = max(s['map_size'], stack_h)

def place(h):
    # bottom edge of a block of height h inside the bottom row
    offset = {'bottom': 0.0, 'center': (bottom_h - h) / 2, 'top': bottom_h - h}[ALIGN]
    return m['bottom'] + offset

x0 = m['left']
boxes = {
    'orbit':  (x0, m['bottom'] + bottom_h + g['rows'], orbit_w, s['orbit_h']),
    'map':    (x0 + s['zoom_w'] + g['cols'], place(s['map_size']), s['map_size'], s['map_size']),
    'zoom A': (x0, place(stack_h) + s['zoom_h'] + g['zooms'], s['zoom_w'], s['zoom_h']),
    'zoom B': (x0, place(stack_h), s['zoom_w'], s['zoom_h']),
}

def cbar_box(panel, c):
    px, py, pw, ph = panel
    length = c['length'] * ph
    shift = {'bottom': 0.0, 'center': (ph - length) / 2, 'top': ph - length}[c['align']]
    return (px + pw + c['pad'], py + shift, c['width'], length)

# nudges, then the colourbars beside their (moved) panels
for name, (dx, dy) in OFFSET.items():
    x, y, w, h = boxes[name]
    boxes[name] = (x + dx, y + dy, w, h)

boxes['cbar f_ret'] = cbar_box(boxes['orbit'], CBAR['orbit'])
boxes['cbar log X'] = cbar_box(boxes['map'], CBAR['map'])

# figure: the bounding box of every box, plus the margins
x_lo = min(x for x, y, w, h in boxes.values())
y_lo = min(y for x, y, w, h in boxes.values())
x_hi = max(x + w for x, y, w, h in boxes.values())
y_hi = max(y + h for x, y, w, h in boxes.values())

boxes = {name: (x - x_lo + m['left'], y - y_lo + m['bottom'], w, h) for name, (x, y, w, h) in boxes.items()}
W = m['left'] + (x_hi - x_lo) + m['right']
H = m['bottom'] + (y_hi - y_lo) + m['top']


# ---- figure -----------------------------------------------------------------
S.use()
fig = plt.figure(figsize=(W, H))

DRAWN = ('orbit', 'map', 'zoom A', 'zoom B', 'cbar f_ret', 'cbar log X')     # every panel is filled below

axes = {}
for name, (x, y, w, h) in boxes.items():
    axes[name] = ax = fig.add_axes([x / W, y / H, w / W, h / H])
    if name in DRAWN:
        continue
    if SHOW_NAMES:
        ax.text(0.5, 0.5, name, transform=ax.transAxes, ha='center', va='center', fontsize=7,
                color=S.INK2, rotation=90 if name.startswith('cbar') else 0)

print(f'figure {W:.2f} x {H:.2f} in')
for name, (x, y, w, h) in boxes.items():
    print(f'  {name:<11s} x {x:5.2f}  y {y:5.2f}  w {w:5.2f}  h {h:5.2f}')


# ---- helpers: colourbar end notes -------------------------------------------
def cbar_notes(cax, top_text, bottom_text):
    # notes just beyond both ends of a colourbar, NOTE_GAP inches away
    gap = NOTE_GAP / boxes[[n for n, a_ in axes.items() if a_ is cax][0]][3]
    style = dict(transform=cax.transAxes, ha='center', fontsize=FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, top_text, va='bottom', **style)
    cax.text(0.5, -gap, bottom_text, va='top', **style)


# ---- orbit panel: the axes the marks live in --------------------------------
from matplotlib.patches import ConnectionPatch, Rectangle
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator, MaxNLocator, NullFormatter

ax_orb = axes['orbit']
t_hi = T_RANGE[1] if T_RANGE[1] is not None else T_NOW
ax_orb.set_xlim(T_RANGE[0], t_hi)
if R_LOG:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(R_MIN, R_MAX)
    ax_orb.yaxis.set_major_locator(FixedLocator(R_TICKS))
    ax_orb.set_yticklabels([f'{r:g}' for r in R_TICKS])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, R_MAX)
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- orbits, coloured by each satellite's f_ret -----------------------------
ORBITS = SD.orbits(R, ORBIT_IDS)                   # in memory after the first time
cmap_f = SD.colormap(ORBIT_STYLE['cmap'])          # a name, or 'name-T' (cut at T)
fret_log = ORBIT_STYLE.get('fret_log', False)
if fret_log:
    norm_f = LogNorm(ORBIT_STYLE['fret_min'], 1.0, clip=True)   # below fret_min (0 included): the lowest colour
else:
    norm_f = Normalize(0.0, 1.0)


def draw_selected(ax, norm):
    # the selected satellites' orbits, with their drawing order (ORBIT_ZORDER), and the R_vir line
    for g in ORBIT_IDS:
        if g in ORBITS:
            draw_orbit(ax, *ORBITS[g], D032['sats'].get(g), ORBIT_STYLE, cmap_f, norm, np.asarray)
            if g in ORBIT_ZORDER:                               # this satellite's own drawing order
                ax.collections[-1].set_zorder(ORBIT_ZORDER[g])
    ax.axhline(1.0, **ORBIT_STYLE['rvir_line'])


# ---- background: every other satellite's orbit, faint (square line ends: no darker joints) ----
def draw_background(ax, norm):
    bg = BACKGROUND
    bg_ids = SD.background_ids(R, bg['which'], bg['min_npart'], exclude=ORBIT_IDS)
    bg_orbits = SD.orbits(R, bg_ids)                    # in memory after the first time
    for g in bg_ids:
        if g not in bg_orbits:
            continue
        tt, rr = bg_orbits[g]
        pts  = np.stack([tt, rr], axis=1)
        seg  = np.stack([pts[:-1], pts[1:]], axis=1)
        good = np.all(np.isfinite(seg), axis=(1, 2))

        lw = np.full(tt.size - 1, bg['lw'])
        t_sink = SD.sinking_time(tt, rr, ORBIT_STYLE['sink_r'])
        if t_sink is not None:
            lw[tt[:-1] >= t_sink] = bg['sunk_lw']

        if bg['colour'] == 'fret':
            f = SD.fret_along(D032['sats'].get(g), tt)
            f_seg = 0.5 * (f[:-1] + f[1:])
            defined = np.isfinite(f_seg)
            colours = cmap_f(norm(np.nan_to_num(f_seg)))
            colours[~defined] = to_rgba(ORBIT_STYLE['undefined_colour'])
            if ORBIT_STYLE['drop_undefined']:
                good &= defined
        else:
            colours = np.tile(to_rgba(bg['colour']), (tt.size - 1, 1))
        colours[:, 3] = bg['alpha']

        ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good], capstyle='butt',
                                         zorder=bg['zorder'], rasterized=bg['rasterized']))


# ---- star-formation beads: one per BEADS['dt'] bin with star formation (size ~ SFR, fill = newborn f_ret) ----
if BEADS['show']:
    b = BEADS
    beads = SD.sf_beads(R, ORBIT_IDS, b['dt'], since=b['since'], r_max=R_MAX if b['r_max'] is None else b['r_max'])     # in memory after the first time

    # bead size: the largest no wider than the spacing of neighbouring beads (fit_dt)
    t_axis = ax_orb.get_xlim()
    pt_per_gyr = boxes['orbit'][2] * 72 / (t_axis[1] - t_axis[0])
    d_max = min(b['d_max'], 0.95 * pt_per_gyr * b['dt']) if b['fit_dt'] else b['d_max']
    d_min = min(b['d_min'], d_max)

    def bead_diameter(sfr):
        sfr = np.atleast_1d(np.asarray(sfr, float))
        if b['sizes'] == 'classes':
            cl = np.asarray(b['classes'], float)
            k = np.clip(np.searchsorted(cl, sfr, side='right') - 1, 0, cl.size - 1)
            return d_min + (d_max - d_min) * k / max(cl.size - 1, 1)
        x = np.log10(sfr / b['sfr_min']) / np.log10(b['sfr_max'] / b['sfr_min'])
        return d_min + (d_max - d_min) * np.clip(x, 0, 1)

    # fill: the orbits' map and range, or BEADS' own
    cmap_b, norm_b, own_b = SD.bead_colours(cmap_f, norm_f, b['cmap'], b['vmin'], b['vmax'])


def draw_beads(ax, norm):
    # the selected satellites' beads, each just above its own orbit
    b = BEADS
    for g in ORBIT_IDS:
        if g not in beads or g not in ORBITS:
            continue
        e = beads[g]
        on = e['sfr'] >= b['show_min']
        if not on.any():
            continue
        tt, rr = ORBITS[g]
        ok = np.isfinite(tt) & np.isfinite(rr)
        t_b = e['t'][on]
        r_b = np.interp(t_b, tt[ok], rr[ok], left=np.nan, right=np.nan)    # on the orbit at the bin centre
        vis = np.isfinite(r_b)
        f_new = e['f_new'][on][vis]
        cm, nm = (cmap_b, norm_b) if own_b else (cmap_f, norm)
        colours = cm(nm(np.nan_to_num(f_new)))
        colours[~np.isfinite(f_new)] = to_rgba(b['undefined_colour'])
        ax.scatter(t_b[vis], r_b[vis], s=bead_diameter(e['sfr'][on][vis]) ** 2, c=colours,
                   edgecolors=b['edge']['color'], linewidths=b['edge']['lw'], zorder=ORBIT_ZORDER.get(g, 3) + b['zorder'] - 3)


# ---- the orbit panel: orbits, background, beads and their size key ---------
draw_selected(ax_orb, norm_f)
if BACKGROUND['show']:
    draw_background(ax_orb, norm_f)
if BEADS['show']:
    draw_beads(ax_orb, norm_f)
    if b['legend']['show']:
        classes = b['sizes'] == 'classes'
        values = b['classes'] if classes else b['legend']['values']
        handles = [ax_orb.scatter([], [], s=bead_diameter(v)[0] ** 2, c='#9a9a9a',
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in values]
        labels = [(r'' if classes else '') + f'{v:g}' for v in values]
        ax_orb.legend(handles, labels, title=r'SFR [M$_\odot$/yr]', loc=b['legend']['loc'], ncol=len(values),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
    ck = b['colour_key']
    if own_b and ck['show']:                       # colour key (own map only): middle-sized beads at these values
        x = (np.asarray(ck['values'], float) if ck['values'] is not None
             else np.asarray(norm_b.inverse(np.linspace(0, 1, ck['n'])), float))
        d_key = 0.5 * (d_min + d_max)
        handles = [ax_orb.scatter([], [], s=d_key ** 2, color=cmap_b(norm_b(v)),
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in x]
        previous = ax_orb.get_legend()
        ax_orb.legend(handles, [f'{v:.2g}' for v in x], title=ck['title'], loc=ck['loc'], ncol=len(x),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
        if previous is not None:
            ax_orb.add_artist(previous)

# redshift ticks on top
if ORBIT_STYLE['z_axis']:
    snaps   = sorted(SNAP_Z)
    zz      = np.array([SNAP_Z[s_] for s_ in snaps])
    tt      = np.array([SNAP_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in Z_TICKS if zz.min() <= z <= zz.max()]

    ax_orb.tick_params(axis='x', which='both', top=False)     # the redshift ticks replace them
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=FONTS['label'], labelpad=2)

# f_ret colourbar
cax_f = axes['cbar f_ret']
cb_f = S.colorbar(fig, ScalarMappable(norm=norm_f, cmap=cmap_f), cax=cax_f,
                  label=ORBIT_STYLE['cbar_label'], ends=False)
if not fret_log:
    cb_f.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar_notes(cax_f, *ORBIT_STYLE['cbar_ends'])


# ---- marks: a box per bottom panel (its snapshots x its radii), linked to that panel ----
targets = {'map': ((MAP_SNAP, MAP_SNAP), (0.0, MAP_BOX))}
for name, z in ZOOM.items():
    targets[name] = (SD.zoom_range(z), z['r'])

for name, ((s1, s2), (r_lo, r_hi)) in targets.items():
    if R_LOG:
        r_lo = max(r_lo, R_MIN)
    t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]
    if t_hi_ - t_lo < MARK['min_width']:                 # e.g. one snapshot: a box of the minimum width
        t_mid = 0.5 * (t_lo + t_hi_)
        t_lo, t_hi_ = t_mid - MARK['min_width'] / 2, t_mid + MARK['min_width'] / 2

    box_style  = {**MARK['box'], **MARK_PANEL.get(name, {}).get('box', {})}
    link_style = {**MARK['link'], **MARK_PANEL.get(name, {}).get('link', {})}

    ax_orb.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
    if MARK['label']:
        ax_orb.text(0.5 * (t_lo + t_hi_), r_hi, name, ha='center', va='bottom', fontsize=FONTS['marks'],
                    color=box_style.get('ec', S.INK2))

    # lower corners of the mark -> upper corners of the panel
    for t_corner, x_panel in ((t_lo, 0.0), (t_hi_, 1.0)):
        fig.add_artist(ConnectionPatch(xyA=(t_corner, r_lo), coordsA=ax_orb.transData,
                                       xyB=(x_panel, 1.0), coordsB=axes[name].transAxes, **link_style))


# ---- Comp01b's zoom region (INSET['comp01']): its mark only, 'look here in Comp01b', and a note under it ----
if INSET['comp01']:
    ins = INSET
    s1, s2 = ins['snaps']
    r_lo, r_hi = ins['r']
    t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]
    box_style = {**MARK['box'], **MARK_PANEL.get('inset', {}).get('box', {})}     # line width etc.: MARK, MARK_PANEL['inset']
    ax_orb.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
    if ins.get('comp01_text'):
        ax_orb.annotate(ins['comp01_text'], xy=(0.5 * (t_lo + t_hi_), r_lo), xycoords='data',
                        xytext=(0, -1), textcoords='offset points', ha='center', va='top',
                        fontsize=FONTS['notes'], color=box_style.get('ec', S.INK2), zorder=box_style.get('zorder', 5))


# ---- map: log X at MAP_SNAP -------------------------------------------------
ax_map = axes['map']
MAP_DATA = SD.square_map(R, MAP_SNAP, quantity='logratio', box=MAP_BOX, **MAP_OPTS)   # in memory after the first time
a_map  = SD.colour_args(MAP_DATA['args'], cmap=MAP_CMAP, zmin=MAP_ZRANGE[0], zmax=MAP_ZRANGE[1])
SD.colormap(MAP_CMAP)                              # registers a 'name-T' cut under its name
cmap_x, norm_x, clabel_x = SM.colours(a_map, MAP_DATA['img'])
e = MAP_DATA['edges']                              # pixel edges, R_vir

im = ax_map.imshow(
    np.ma.masked_invalid(MAP_DATA['img']),
    origin='lower',
    extent=(e[0], e[-1], e[0], e[-1]),
    cmap=cmap_x,
    norm=norm_x,
    interpolation='none' if MAP_OPTS['smooth'] else 'nearest',
)
ax_map.set_facecolor('white')

# R_vir circle and the host centre
ax_map.add_patch(Circle((0, 0), 1.0, fill=False, ls=(0, (3, 2)), lw=0.8, ec=S.INK2))
ax_map.plot([0], [0], marker='+', ms=5, mew=1.0, color=S.INK)

ax_map.set_xlim(e[0], e[-1])
ax_map.set_ylim(e[0], e[-1])
ax_map.set_aspect('equal')
_, _, xl, yl = SM.VIEWS[MAP_OPTS['view']]
ax_map.set_xlabel(rf'${xl}/R_{{\rm vir}}$')
ax_map.set_ylabel(rf'${yl}/R_{{\rm vir}}$', labelpad=1)
ax_map.text(0.5, 0.93, f'$s = {int(MAP_SNAP)}$', transform=ax_map.transAxes,
            ha='center', va='top', fontsize=FONTS['notes'], color=S.INK2)

# log X colourbar: linear value axis (colours follow the two-leg norm), 'host' / 'owner' at its ends
cax_x = axes['cbar log X']
S.colorbar(fig, im, cax=cax_x, label=clabel_x, ends=False)
cax_x.set_yscale('linear')
cax_x.yaxis.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]))
cax_x.minorticks_off()
cbar_notes(cax_x, 'host', 'owner')


# ---- zooms: log10(E_H/E_S) of the satellites' stars in each box, all and newborn ----
k = KDE
x_lo, x_hi = k['xrange']
grid  = np.linspace(x_lo, x_hi, 801)
edges = np.linspace(x_lo, x_hi, k['bins'] + 1)
step  = k['histtype'] == 'step'

def curve(x, w):
    # per dex on the grid (kde) or per bin (step), normalised as KDE asks; x may hold -inf (not host-bound)
    finite = np.isfinite(x)
    if step:
        xs, y = edges, np.histogram(x[finite], bins=edges, weights=w[finite])[0] / np.diff(edges)
    else:
        xs, y = grid, SD.kde(x[finite], w[finite], grid, k['bw']) * w[finite].sum()
    if k['norm'] == 'density':
        y = y / max(w.sum() if k['per'] == 'population' else w[finite].sum(), 1e-300)
    return xs, y

def gradient_fill(ax, xs, y, alpha):
    # area under the curve filled with the map's log X colours
    px, py = (np.repeat(xs, 2)[1:-1], np.repeat(y, 2)) if step else (xs, y)
    poly = ax.fill_between(px, 0, py, facecolor='none', edgecolor='none')
    img = ax.imshow(grid[None, :], extent=(x_lo, x_hi, 0, py.max()), aspect='auto', origin='lower',
                    cmap=cmap_x, norm=norm_x, alpha=alpha, zorder=1)
    img.set_clip_path(poly.get_paths()[0], transform=ax.transData)

for name, z in ZOOM.items():
    ax = axes[name]
    Z = SD.zoom_sample(R, SD.zoom_range(z), z['r'])     # every satellite's stars in the box (in memory)

    # keep the stars of the chosen satellites: by default those drawn on the orbit panel
    sats = z.get('sats')
    if sats is None:
        sats = ORBIT_IDS
    if not (isinstance(sats, str) and sats == 'all'):
        keep = np.isin(Z['owner'], np.asarray(list(sats), np.int64))
        Z = {key: val[keep] for key, val in Z.items()}
    owners, counts = np.unique(Z['owner'], return_counts=True)
    print(f'{name}: {Z["m"].size:,} stars ({Z["new"].sum():,} newborn) from '
          + (', '.join(f'{o} ({c:,})' for o, c in zip(owners, counts)) if owners.size else 'none of the chosen satellites'))
    weights = Z['m'] if k['weight'] == 'mass' else np.ones(Z['m'].size)

    y_top, note_lines = 0.0, []
    in_legend = k['notes']['show'] and k['notes']['where'] == 'legend'
    for part, sel in (('all', np.ones(Z['m'].size, bool)), ('newborn', Z['new'])):
        x, w = Z['logx'][sel], weights[sel]
        n = int(sel.sum())
        if w.sum() > 0:
            ret = 100 * (1 - w[x > 0].sum() / w.sum())     # retained: not bound to the host (x = -inf counts)
            note_lines.append(f'{part} {n:,}, {ret:.0f}%')
        else:
            note_lines.append(f'{part} {n:,}, -')
        label = note_lines[-1] if in_legend else part
        if x.size == 0 or not np.isfinite(x).any():
            continue
        xs, y = curve(x, w)
        st = k[part]
        if st['fill'] > 0 and y.max() > 0:
            gradient_fill(ax, xs, y, st['fill'])
        if step:
            ax.stairs(y, xs, color=st['color'], lw=st['lw'], ls=st['ls'], zorder=3, label=label)
        else:
            ax.plot(xs, y, color=st['color'], lw=st['lw'], ls=st['ls'], zorder=3, label=label)
        y_top = max(y_top, y.max())

    ax.axvline(0.0, zorder=2, **k['zero'])
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(0.0, 1.1 * y_top if y_top > 0 else 1.0)
    ax.set_xlabel(r'$\log_{10}(E_H/E_S)$')
    unit = r'M$_\odot$' if k['weight'] == 'mass' else 'N'
    ax.set_ylabel(r'PDF [dex$^{-1}$]' if k['norm'] == 'density' else f'{unit} / dex')
    if k['notes']['show']:                               # stars and retention of each curve
        nt = k['notes']
        where = nt['where']
        if where in ('upper left', 'upper right', 'lower left', 'lower right'):
            left = where.endswith('left')
            upper = where.startswith('upper')
            ax.text(nt['inset'][0] if left else 1 - nt['inset'][0],
                    1 - nt['inset'][1] if upper else nt['inset'][1],
                    '\n'.join(note_lines), transform=ax.transAxes,
                    ha='left' if left else 'right', va='top' if upper else 'bottom',
                    fontsize=FONTS['notes'], color=S.INK2, zorder=4)
        elif where == 'title':
            ax.set_title('\n'.join(note_lines), fontsize=FONTS['notes'], color=S.INK2, pad=nt['title_pad'])
        elif where != 'legend':                          # 'legend': the curve labels
            raise ValueError(f"KDE['notes']['where'] must be 'upper left', 'upper right', 'lower left', "
                             f"'lower right', 'title' or 'legend', not {where!r}")
    if k['legend'] or in_legend:
        ax.legend(loc='upper right', fontsize=FONTS['notes'], handlelength=1.8)


# ---- common text sizes: axis labels and tick labels everywhere --------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=FONTS['ticks'])
    ax.xaxis.label.set_size(FONTS['label'])
    ax.yaxis.label.set_size(FONTS['label'])


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, R.stem(f'comp01_{ORBIT_SET}'), SAVE)

In [ ]:
# Plotting  ·  Comp01 figure, mirrored: map on the left, zooms on the right (orbits, map and zooms)

# ---- geometry: every box in inches from the figure's bottom-left corner -----
s, g, m = SIZE, GAP, MARGIN


ZOOM['zoom A'], ZOOM['zoom B'] = ZOOM['zoom B'], ZOOM['zoom A']


bottom_w = s['zoom_w'] + g['cols'] + s['map_size']
orbit_w  = s['orbit_w'] if s['orbit_w'] is not None else bottom_w
stack_h  = 2 * s['zoom_h'] + g['zooms']
bottom_h = max(s['map_size'], stack_h)

def place(h):
    # bottom edge of a block of height h inside the bottom row
    offset = {'bottom': 0.0, 'center': (bottom_h - h) / 2, 'top': bottom_h - h}[ALIGN]
    return m['bottom'] + offset

x0 = m['left']
boxes = {
    'orbit':  (x0, m['bottom'] + bottom_h + g['rows'], orbit_w, s['orbit_h']),
    'map':    (x0 + s['zoom_w'] + g['cols'], place(s['map_size']), s['map_size'], s['map_size']),
    'zoom A': (x0, place(stack_h) + s['zoom_h'] + g['zooms'], s['zoom_w'], s['zoom_h']),
    'zoom B': (x0, place(stack_h), s['zoom_w'], s['zoom_h']),
}

def cbar_box(panel, c):
    px, py, pw, ph = panel
    length = c['length'] * ph
    shift = {'bottom': 0.0, 'center': (ph - length) / 2, 'top': ph - length}[c['align']]
    return (px + pw + c['pad'], py + shift, c['width'], length)

# nudges, then the colourbars beside their (moved) panels
for name, (dx, dy) in OFFSET.items():
    x, y, w, h = boxes[name]
    boxes[name] = (x + dx, y + dy, w, h)

# mirror the bottom row left <-> right about its own centre (nudges included)
bottom_row = ['map', 'zoom A', 'zoom B']
row_lo = min(boxes[n][0] for n in bottom_row)
row_hi = max(boxes[n][0] + boxes[n][2] for n in bottom_row)
for name in bottom_row:
    x, y, w, h = boxes[name]
    boxes[name] = (row_lo + row_hi - (x + w), y, w, h)

boxes['cbar f_ret'] = cbar_box(boxes['orbit'], CBAR['orbit'])
boxes['cbar log X'] = cbar_box(boxes['map'], CBAR['map'])

# figure: the bounding box of every box, plus the margins
x_lo = min(x for x, y, w, h in boxes.values())
y_lo = min(y for x, y, w, h in boxes.values())
x_hi = max(x + w for x, y, w, h in boxes.values())
y_hi = max(y + h for x, y, w, h in boxes.values())

boxes = {name: (x - x_lo + m['left'], y - y_lo + m['bottom'], w, h) for name, (x, y, w, h) in boxes.items()}
W = m['left'] + (x_hi - x_lo) + m['right']
H = m['bottom'] + (y_hi - y_lo) + m['top']


# ---- figure -----------------------------------------------------------------
S.use()
fig = plt.figure(figsize=(W, H))

DRAWN = ('orbit', 'map', 'zoom A', 'zoom B', 'cbar f_ret', 'cbar log X')     # every panel is filled below

axes = {}
for name, (x, y, w, h) in boxes.items():
    axes[name] = ax = fig.add_axes([x / W, y / H, w / W, h / H])
    if name in DRAWN:
        continue
    if SHOW_NAMES:
        ax.text(0.5, 0.5, name, transform=ax.transAxes, ha='center', va='center', fontsize=7,
                color=S.INK2, rotation=90 if name.startswith('cbar') else 0)

print(f'figure {W:.2f} x {H:.2f} in')
for name, (x, y, w, h) in boxes.items():
    print(f'  {name:<11s} x {x:5.2f}  y {y:5.2f}  w {w:5.2f}  h {h:5.2f}')


# ---- helpers: colourbar end notes -------------------------------------------
def cbar_notes(cax, top_text, bottom_text):
    # notes just beyond both ends of a colourbar, NOTE_GAP inches away
    gap = NOTE_GAP / boxes[[n for n, a_ in axes.items() if a_ is cax][0]][3]
    style = dict(transform=cax.transAxes, ha='center', fontsize=FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, top_text, va='bottom', **style)
    cax.text(0.5, -gap, bottom_text, va='top', **style)


# ---- orbit panel: the axes the marks live in --------------------------------
from matplotlib.patches import ConnectionPatch, Rectangle
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator, MaxNLocator, NullFormatter

ax_orb = axes['orbit']
t_hi = T_RANGE[1] if T_RANGE[1] is not None else T_NOW
ax_orb.set_xlim(T_RANGE[0], t_hi)
if R_LOG:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(R_MIN, R_MAX)
    ax_orb.yaxis.set_major_locator(FixedLocator(R_TICKS))
    ax_orb.set_yticklabels([f'{r:g}' for r in R_TICKS])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, R_MAX)
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- orbits, coloured by each satellite's f_ret -----------------------------
ORBITS = SD.orbits(R, ORBIT_IDS)                   # in memory after the first time
cmap_f = SD.colormap(ORBIT_STYLE['cmap'])          # a name, or 'name-T' (cut at T)
fret_log = ORBIT_STYLE.get('fret_log', False)
if fret_log:
    norm_f = LogNorm(ORBIT_STYLE['fret_min'], 1.0, clip=True)   # below fret_min (0 included): the lowest colour
else:
    norm_f = Normalize(0.0, 1.0)


def draw_selected(ax, norm):
    # the selected satellites' orbits, with their drawing order (ORBIT_ZORDER), and the R_vir line
    for g in ORBIT_IDS:
        if g in ORBITS:
            draw_orbit(ax, *ORBITS[g], D032['sats'].get(g), ORBIT_STYLE, cmap_f, norm, np.asarray)
            if g in ORBIT_ZORDER:                               # this satellite's own drawing order
                ax.collections[-1].set_zorder(ORBIT_ZORDER[g])
    ax.axhline(1.0, **ORBIT_STYLE['rvir_line'])


# ---- background: every other satellite's orbit, faint (square line ends: no darker joints) ----
def draw_background(ax, norm):
    bg = BACKGROUND
    bg_ids = SD.background_ids(R, bg['which'], bg['min_npart'], exclude=ORBIT_IDS)
    bg_orbits = SD.orbits(R, bg_ids)                    # in memory after the first time
    for g in bg_ids:
        if g not in bg_orbits:
            continue
        tt, rr = bg_orbits[g]
        pts  = np.stack([tt, rr], axis=1)
        seg  = np.stack([pts[:-1], pts[1:]], axis=1)
        good = np.all(np.isfinite(seg), axis=(1, 2))

        lw = np.full(tt.size - 1, bg['lw'])
        t_sink = SD.sinking_time(tt, rr, ORBIT_STYLE['sink_r'])
        if t_sink is not None:
            lw[tt[:-1] >= t_sink] = bg['sunk_lw']

        if bg['colour'] == 'fret':
            f = SD.fret_along(D032['sats'].get(g), tt)
            f_seg = 0.5 * (f[:-1] + f[1:])
            defined = np.isfinite(f_seg)
            colours = cmap_f(norm(np.nan_to_num(f_seg)))
            colours[~defined] = to_rgba(ORBIT_STYLE['undefined_colour'])
            if ORBIT_STYLE['drop_undefined']:
                good &= defined
        else:
            colours = np.tile(to_rgba(bg['colour']), (tt.size - 1, 1))
        colours[:, 3] = bg['alpha']

        ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=lw[good], capstyle='butt',
                                         zorder=bg['zorder'], rasterized=bg['rasterized']))


# ---- star-formation beads: one per BEADS['dt'] bin with star formation (size ~ SFR, fill = newborn f_ret) ----
if BEADS['show']:
    b = BEADS
    beads = SD.sf_beads(R, ORBIT_IDS, b['dt'], since=b['since'], r_max=R_MAX if b['r_max'] is None else b['r_max'])     # in memory after the first time

    # bead size: the largest no wider than the spacing of neighbouring beads (fit_dt)
    t_axis = ax_orb.get_xlim()
    pt_per_gyr = boxes['orbit'][2] * 72 / (t_axis[1] - t_axis[0])
    d_max = min(b['d_max'], 0.95 * pt_per_gyr * b['dt']) if b['fit_dt'] else b['d_max']
    d_min = min(b['d_min'], d_max)

    def bead_diameter(sfr):
        sfr = np.atleast_1d(np.asarray(sfr, float))
        if b['sizes'] == 'classes':
            cl = np.asarray(b['classes'], float)
            k = np.clip(np.searchsorted(cl, sfr, side='right') - 1, 0, cl.size - 1)
            return d_min + (d_max - d_min) * k / max(cl.size - 1, 1)
        x = np.log10(sfr / b['sfr_min']) / np.log10(b['sfr_max'] / b['sfr_min'])
        return d_min + (d_max - d_min) * np.clip(x, 0, 1)

    # fill: the orbits' map and range, or BEADS' own
    cmap_b, norm_b, own_b = SD.bead_colours(cmap_f, norm_f, b['cmap'], b['vmin'], b['vmax'])


def draw_beads(ax, norm):
    # the selected satellites' beads, each just above its own orbit
    b = BEADS
    for g in ORBIT_IDS:
        if g not in beads or g not in ORBITS:
            continue
        e = beads[g]
        on = e['sfr'] >= b['show_min']
        if not on.any():
            continue
        tt, rr = ORBITS[g]
        ok = np.isfinite(tt) & np.isfinite(rr)
        t_b = e['t'][on]
        r_b = np.interp(t_b, tt[ok], rr[ok], left=np.nan, right=np.nan)    # on the orbit at the bin centre
        vis = np.isfinite(r_b)
        f_new = e['f_new'][on][vis]
        cm, nm = (cmap_b, norm_b) if own_b else (cmap_f, norm)
        colours = cm(nm(np.nan_to_num(f_new)))
        colours[~np.isfinite(f_new)] = to_rgba(b['undefined_colour'])
        ax.scatter(t_b[vis], r_b[vis], s=bead_diameter(e['sfr'][on][vis]) ** 2, c=colours,
                   edgecolors=b['edge']['color'], linewidths=b['edge']['lw'], zorder=ORBIT_ZORDER.get(g, 3) + b['zorder'] - 3)


# ---- the orbit panel: orbits, background, beads and their size key ---------
draw_selected(ax_orb, norm_f)
if BACKGROUND['show']:
    draw_background(ax_orb, norm_f)
if BEADS['show']:
    draw_beads(ax_orb, norm_f)
    if b['legend']['show']:
        classes = b['sizes'] == 'classes'
        values = b['classes'] if classes else b['legend']['values']
        handles = [ax_orb.scatter([], [], s=bead_diameter(v)[0] ** 2, c='#9a9a9a',
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in values]
        labels = [(r'' if classes else '') + f'{v:g}' for v in values]
        ax_orb.legend(handles, labels, title=r'SFR [M$_\odot$/yr]', loc=b['legend']['loc'], ncol=len(values),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
    ck = b['colour_key']
    if own_b and ck['show']:                       # colour key (own map only): middle-sized beads at these values
        x = (np.asarray(ck['values'], float) if ck['values'] is not None
             else np.asarray(norm_b.inverse(np.linspace(0, 1, ck['n'])), float))
        d_key = 0.5 * (d_min + d_max)
        handles = [ax_orb.scatter([], [], s=d_key ** 2, color=cmap_b(norm_b(v)),
                                  edgecolors=b['edge']['color'], linewidths=b['edge']['lw']) for v in x]
        previous = ax_orb.get_legend()
        ax_orb.legend(handles, [f'{v:.2g}' for v in x], title=ck['title'], loc=ck['loc'], ncol=len(x),
                      fontsize=FONTS['notes'], title_fontsize=FONTS['notes'], frameon=False,
                      handletextpad=0.3, columnspacing=1.0, scatterpoints=1)
        if previous is not None:
            ax_orb.add_artist(previous)

# redshift ticks on top
if ORBIT_STYLE['z_axis']:
    snaps   = sorted(SNAP_Z)
    zz      = np.array([SNAP_Z[s_] for s_ in snaps])
    tt      = np.array([SNAP_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in Z_TICKS if zz.min() <= z <= zz.max()]

    ax_orb.tick_params(axis='x', which='both', top=False)     # the redshift ticks replace them
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=FONTS['label'], labelpad=2)

# f_ret colourbar
cax_f = axes['cbar f_ret']
cb_f = S.colorbar(fig, ScalarMappable(norm=norm_f, cmap=cmap_f), cax=cax_f,
                  label=ORBIT_STYLE['cbar_label'], ends=False)
if not fret_log:
    cb_f.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar_notes(cax_f, *ORBIT_STYLE['cbar_ends'])


# ---- marks: a box per bottom panel (its snapshots x its radii), linked to that panel ----
targets = {'map': ((MAP_SNAP, MAP_SNAP), (0.0, MAP_BOX))}
for name, z in ZOOM.items():
    targets[name] = (SD.zoom_range(z), z['r'])

for name, ((s1, s2), (r_lo, r_hi)) in targets.items():
    if R_LOG:
        r_lo = max(r_lo, R_MIN)
    t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]
    if t_hi_ - t_lo < MARK['min_width']:                 # e.g. one snapshot: a box of the minimum width
        t_mid = 0.5 * (t_lo + t_hi_)
        t_lo, t_hi_ = t_mid - MARK['min_width'] / 2, t_mid + MARK['min_width'] / 2

    box_style  = {**MARK['box'], **MARK_PANEL.get(name, {}).get('box', {})}
    link_style = {**MARK['link'], **MARK_PANEL.get(name, {}).get('link', {})}

    ax_orb.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
    if MARK['label']:
        ax_orb.text(0.5 * (t_lo + t_hi_), r_hi, name, ha='center', va='bottom', fontsize=FONTS['marks'],
                    color=box_style.get('ec', S.INK2))

    # lower corners of the mark -> upper corners of the panel
    for t_corner, x_panel in ((t_lo, 0.0), (t_hi_, 1.0)):
        fig.add_artist(ConnectionPatch(xyA=(t_corner, r_lo), coordsA=ax_orb.transData,
                                       xyB=(x_panel, 1.0), coordsB=axes[name].transAxes, **link_style))


# ---- Comp01b's zoom region (INSET['comp01']): its mark only, 'look here in Comp01b', and a note under it ----
if INSET['comp01']:
    ins = INSET
    s1, s2 = ins['snaps']
    r_lo, r_hi = ins['r']
    t_lo, t_hi_ = SNAP_TIME[s1], SNAP_TIME[s2]
    box_style = {**MARK['box'], **MARK_PANEL.get('inset', {}).get('box', {})}     # line width etc.: MARK, MARK_PANEL['inset']
    ax_orb.add_patch(Rectangle((t_lo, r_lo), t_hi_ - t_lo, r_hi - r_lo, **box_style))
    if ins.get('comp01_text'):
        ax_orb.annotate(ins['comp01_text'], xy=(0.5 * (t_lo + t_hi_), r_lo), xycoords='data',
                        xytext=(0, -1), textcoords='offset points', ha='center', va='top',
                        fontsize=FONTS['notes'], color=box_style.get('ec', S.INK2), zorder=box_style.get('zorder', 5))


# ---- map: log X at MAP_SNAP -------------------------------------------------
ax_map = axes['map']
MAP_DATA = SD.square_map(R, MAP_SNAP, quantity='logratio', box=MAP_BOX, **MAP_OPTS)   # in memory after the first time
a_map  = SD.colour_args(MAP_DATA['args'], cmap=MAP_CMAP, zmin=MAP_ZRANGE[0], zmax=MAP_ZRANGE[1])
SD.colormap(MAP_CMAP)                              # registers a 'name-T' cut under its name
cmap_x, norm_x, clabel_x = SM.colours(a_map, MAP_DATA['img'])
e = MAP_DATA['edges']                              # pixel edges, R_vir

im = ax_map.imshow(
    np.ma.masked_invalid(MAP_DATA['img']),
    origin='lower',
    extent=(e[0], e[-1], e[0], e[-1]),
    cmap=cmap_x,
    norm=norm_x,
    interpolation='none' if MAP_OPTS['smooth'] else 'nearest',
)
ax_map.set_facecolor('white')

# R_vir circle and the host centre
ax_map.add_patch(Circle((0, 0), 1.0, fill=False, ls=(0, (3, 2)), lw=0.8, ec=S.INK2))
ax_map.plot([0], [0], marker='+', ms=5, mew=1.0, color=S.INK)

ax_map.set_xlim(e[0], e[-1])
ax_map.set_ylim(e[0], e[-1])
ax_map.set_aspect('equal')
_, _, xl, yl = SM.VIEWS[MAP_OPTS['view']]
ax_map.set_xlabel(rf'${xl}/R_{{\rm vir}}$')
ax_map.set_ylabel(rf'${yl}/R_{{\rm vir}}$', labelpad=1)
ax_map.text(0.5, 0.97, f'$t = {SNAP_TIME[MAP_SNAP]:.2f}$ Gyr', transform=ax_map.transAxes,
            ha='center', va='top', fontsize=FONTS['notes'], color=S.INK2)
ax_map.text(0.5, 0.93, f'$s = {int(MAP_SNAP)}$', transform=ax_map.transAxes,
            ha='center', va='top', fontsize=FONTS['notes'], color=S.INK2)

# log X colourbar: linear value axis (colours follow the two-leg norm), 'host' / 'owner' at its ends
cax_x = axes['cbar log X']
S.colorbar(fig, im, cax=cax_x, label=clabel_x, ends=False)
cax_x.set_yscale('linear')
cax_x.yaxis.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 2.5, 5, 10]))
cax_x.minorticks_off()
cbar_notes(cax_x, 'host', 'owner')


# ---- zooms: log10(E_H/E_S) of the satellites' stars in each box, all and newborn ----
k = KDE
x_lo, x_hi = k['xrange']
grid  = np.linspace(x_lo, x_hi, 801)
edges = np.linspace(x_lo, x_hi, k['bins'] + 1)
step  = k['histtype'] == 'step'

def curve(x, w):
    # per dex on the grid (kde) or per bin (step), normalised as KDE asks; x may hold -inf (not host-bound)
    finite = np.isfinite(x)
    if step:
        xs, y = edges, np.histogram(x[finite], bins=edges, weights=w[finite])[0] / np.diff(edges)
    else:
        xs, y = grid, SD.kde(x[finite], w[finite], grid, k['bw']) * w[finite].sum()
    if k['norm'] == 'density':
        y = y / max(w.sum() if k['per'] == 'population' else w[finite].sum(), 1e-300)
    return xs, y

def gradient_fill(ax, xs, y, alpha):
    # area under the curve filled with the map's log X colours
    px, py = (np.repeat(xs, 2)[1:-1], np.repeat(y, 2)) if step else (xs, y)
    poly = ax.fill_between(px, 0, py, facecolor='none', edgecolor='none')
    img = ax.imshow(grid[None, :], extent=(x_lo, x_hi, 0, py.max()), aspect='auto', origin='lower',
                    cmap=cmap_x, norm=norm_x, alpha=alpha, zorder=1)
    img.set_clip_path(poly.get_paths()[0], transform=ax.transData)

for name, z in ZOOM.items():
    ax = axes[name]
    Z = SD.zoom_sample(R, SD.zoom_range(z), z['r'])     # every satellite's stars in the box (in memory)

    # keep the stars of the chosen satellites: by default those drawn on the orbit panel
    sats = z.get('sats')
    if sats is None:
        sats = ORBIT_IDS
    if not (isinstance(sats, str) and sats == 'all'):
        keep = np.isin(Z['owner'], np.asarray(list(sats), np.int64))
        Z = {key: val[keep] for key, val in Z.items()}
    owners, counts = np.unique(Z['owner'], return_counts=True)
    print(f'{name}: {Z["m"].size:,} stars ({Z["new"].sum():,} newborn) from '
          + (', '.join(f'{o} ({c:,})' for o, c in zip(owners, counts)) if owners.size else 'none of the chosen satellites'))
    weights = Z['m'] if k['weight'] == 'mass' else np.ones(Z['m'].size)

    y_top, note_lines = 0.0, []
    in_legend = k['notes']['show'] and k['notes']['where'] == 'legend'
    for part, sel in (('all', np.ones(Z['m'].size, bool)), ('newborn', Z['new'])):
        x, w = Z['logx'][sel], weights[sel]
        n = int(sel.sum())
        if w.sum() > 0:
            ret = 100 * (1 - w[x > 0].sum() / w.sum())     # retained: not bound to the host (x = -inf counts)
            note_lines.append(f'{part} {n:,}, {ret:.0f}%')
        else:
            note_lines.append(f'{part} {n:,}, -')
        label = note_lines[-1] if in_legend else part
        if x.size == 0 or not np.isfinite(x).any():
            continue
        xs, y = curve(x, w)
        st = k[part]
        if st['fill'] > 0 and y.max() > 0:
            gradient_fill(ax, xs, y, st['fill'])
        if step:
            ax.stairs(y, xs, color=st['color'], lw=st['lw'], ls=st['ls'], zorder=3, label=label)
        else:
            ax.plot(xs, y, color=st['color'], lw=st['lw'], ls=st['ls'], zorder=3, label=label)
        y_top = max(y_top, y.max())

    ax.axvline(0.0, zorder=2, **k['zero'])
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(0.0, 1.1 * y_top if y_top > 0 else 1.0)
    ax.set_xlabel(r'$\log_{10}(E_H/E_S)$')
    unit = r'M$_\odot$' if k['weight'] == 'mass' else 'N'
    ax.set_ylabel(r'PDF [dex$^{-1}$]' if k['norm'] == 'density' else f'{unit} / dex')
    if k['notes']['show']:                               # stars and retention of each curve
        nt = k['notes']
        where = nt['where']
        if where in ('upper left', 'upper right', 'lower left', 'lower right'):
            left = where.endswith('left')
            upper = where.startswith('upper')
            ax.text(nt['inset'][0] if left else 1 - nt['inset'][0],
                    1 - nt['inset'][1] if upper else nt['inset'][1],
                    '\n'.join(note_lines), transform=ax.transAxes,
                    ha='left' if left else 'right', va='top' if upper else 'bottom',
                    fontsize=FONTS['notes'], color=S.INK2, zorder=4)
        elif where == 'title':
            ax.set_title('\n'.join(note_lines), fontsize=FONTS['notes'], color=S.INK2, pad=nt['title_pad'])
        elif where != 'legend':                          # 'legend': the curve labels
            raise ValueError(f"KDE['notes']['where'] must be 'upper left', 'upper right', 'lower left', "
                             f"'lower right', 'title' or 'legend', not {where!r}")
    if k['legend'] or in_legend:
        ax.legend(loc='upper right', fontsize=FONTS['notes'], handlelength=1.8)


# ---- common text sizes: axis labels and tick labels everywhere --------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=FONTS['ticks'])
    ax.xaxis.label.set_size(FONTS['label'])
    ax.yaxis.label.set_size(FONTS['label'])


# ---- show (and save if SAVE) ------------------------------------------------
SD.show(fig, R.stem(f'comp01_{ORBIT_SET}_mirror'), SAVE)

## Comp02: birth-set purity on the orbit panel (standalone)

Is a star, when it is born, where its birth galaxy is? Each galaxy is fitted, at every snapshot, with a 6D normal in position and velocity (mass-weighted mean and covariance of its members: the stars of its infall list that it binds; the newborns of that snapshot are left out). A star born in g (final birth map) is judged at its first snapshot, or `lag_myr` after its exact birth time: Λ = ln L_g − max over other galaxies h of ln L_h, with L = N(x | μ, Σ) (the host as two normals, inside and outside `host_split`). It is **contaminated** when another galaxy is more likely by a factor > K *and* it lies beyond g's `level` contour.

Line colour (`PURITY['mode']`): **total**, the purity of each satellite's whole birth set so far, 1 − M(contaminated) / M(scored) over the stars born in it by then (initial masses); or **differential**, the same over the stars born since the previous snapshot only (no scored births in that interval: the undefined colour, no interpolation across). Grey before its first scored birth.

**Standalone**: needs only the two Loading cells. Its own parameters (`PURITY` and the `C2_*` settings, which started as copies of the Comp01 values) never touch Comp01's or Comp01b's.

In [ ]:
# Plotting  ·  Comp02 parameters: birth-set purity, and everything the figure needs (standalone; started from the Comp01 values)

# ---- the purity test ----------------------------------------------------------
PURITY = dict(
    lag_myr    = 0,               # judge each star at its first snapshot (0), or this many Myr after its birth
                                  # (e.g. 100, 300: does it still look like its birth galaxy's?); ~3 min per new lag
    K          = 100,             # contaminated if another galaxy is more likely by more than this factor ...
    level      = 0.999,           # ... and the star lies beyond this contour of its birth galaxy (chi2, 6 dof)
    n_min      = 20,              # galaxies with fewer member stars get no fit (their newborns stay unscored)
    host_split = 15.0,            # kpc: the host as two normals, inside / outside this radius (None: one)
    fit        = 'refit',         # 'refit': normals fitted here to each galaxy's members (newborns left out);
                                  # 'rr': roadrunner's own stored normals (one per galaxy, host_split ignored; they
                                  #  include the newborns and drive roadrunner's own birth decisions: not independent)
    cmap       = 'cmyt.arbre-0.9',   # any matplotlib, cmr.* or cmyt.* name, octarine_tuned, or 'name-T'
    vmin       = 0.99,             # purity at the bottom of the colour scale (lower purity: the lowest colour)
    vmax       = 1.0,             # ... and at the top (1: pure)
    mode       = 'differential',         # 'total': purity of the whole birth set so far (cumulative);
                                  # 'differential': purity of the stars born since the previous snapshot only
                                  # (no star formation there: the undefined colour)
    cbar_label = dict(total=r'birth-set purity', differential=r'purity of the new stars'),
    cbar_ends  = ('pure', 'contaminated'),   # notes beyond the colourbar's ends, (top, bottom); None: none
    name       = 'comp02',        # file name <name>_lag<lag_myr>_<C2_SET_NAME>
)

# ---- which satellites ------------------------------------------------------------
C2_SET_NAME = 'final'
C2_SATS     = [27155, 26681, 3674, 18557, 10779, 3567, 20813, 11508, 34253]
C2_ZORDER   = {27155: 100}        # drawing order of single satellites (higher on top); unlisted: 3, later in C2_SATS on top

# ---- panel, colourbar, margins, inches ------------------------------------------------
C2_PANEL = dict(width=7.2, height=2.6)
C2_CBAR  = dict(width=0.2, pad=0.05, length=1.0, align='center')   # length: fraction of the panel height
C2_MARGIN = dict(left=0.45, right=0.55, top=0.35, bottom=0.40)

# ---- axes ---------------------------------------------------------------------------
C2_AXES = dict(
    t_range = (0.0, None),        # Gyr; None: the last snapshot
    r_max   = 6,                  # top of the r/R_vir axis (star tables reach r_max + 0.5 R_vir)
    r_log   = True,               # logarithmic r/R_vir axis ...
    r_min   = 1e-2,               # ... from here
    r_ticks = (1e-2, 1e-1, 1),    # labelled r/R_vir values (log axis)
    z_axis  = False,              # redshift ticks on top ...
    z_ticks = (10, 5, 4, 3, 2, 1, 0),   # ... at these redshifts
)

# ---- lines ------------------------------------------------------------------------------
C2_LINES = dict(
    lw               = 1.4,
    sunk_lw          = 0.2,       # after the sinking time ...
    sink_r           = 0.1,       # ... from which it never again rises above this r/R_vir
    undefined_colour = '#c8c8c8', # no scored births (before the first one; 'differential': none in that interval)
    drop_undefined   = False,     # True: leave those stretches out instead of grey
    rvir_line        = dict(color='#52514e', lw=1, ls=(0, (3, 2)), zorder=1000),
)

# ---- background: every other satellite's orbit, faint, under the selected ones (no beads) ----
C2_BACKGROUND = dict(
    show       = True,
    which      = 'candidates',    # 'candidates': entering R_vir with >= min_npart star particles at their peak; 'all'
    min_npart  = 50,
    colour     = 'purity',        # 'purity': their own birth-set purity, on the colour scale; or one colour, e.g. '#9a9a9a'
    alpha      = 0.55,
    lw         = 0.2,
    sunk_lw    = 0.0,
    zorder     = 2,
    rasterized = True,
)

# ---- text sizes, points -------------------------------------------------------------
C2_FONTS    = dict(label=14, ticks=9.5, notes=10)
C2_NOTE_GAP = 0.04                # inches between the colourbar's end and its note

In [ ]:
# Processing  ·  Comp02 (standalone: needs only the Loading cells)

from replay import birthpurity as BP

R.load()
R.set_r_keep(max(getattr(R, 'r_keep', 2.0), C2_AXES['r_max'] + 0.5))    # star tables reach beyond the panel's top

# ---- snapshot times ---------------------------------------------------------------------
_h = R.tree[R.tree['Sub_tree_id'] == R.host]
C2_TIME = dict(zip(_h['Snapshot'].astype(int).tolist(), _h['Time'].astype(float).tolist()))
C2_Z    = dict(zip(_h['Snapshot'].astype(int).tolist(), _h['Redshift'].astype(float).tolist()))
C2_T_NOW = C2_TIME[max(s for s in C2_TIME if s <= R.last)]

# ---- purity of every star with a birth galaxy (~3 min the first time per lag, n_min, host_split, fit) ----
PURITY_DATA = SD.birth_purity(R, PURITY['lag_myr'], PURITY['n_min'], PURITY['host_split'], PURITY['fit'])

# ---- orbits: the selected satellites and the background -----------------------------------
C2_ORBITS = SD.orbits(R, C2_SATS)
C2_BG_IDS = (SD.background_ids(R, C2_BACKGROUND['which'], C2_BACKGROUND['min_npart'], exclude=C2_SATS)
             if C2_BACKGROUND['show'] else [])
SD.orbits(R, C2_BG_IDS)

# ---- summary (K and level only change the flags: instant) ---------------------------------
_scored, _bad = BP.flags(PURITY_DATA, PURITY['K'], PURITY['level'])
print(f"fit {PURITY['fit']!r}, lag {PURITY['lag_myr']} Myr: {_scored.sum():,} of {_scored.size:,} births scored; "
      f"contaminated {100 * (PURITY_DATA['m0'] * _bad).sum() / (PURITY_DATA['m0'] * _scored).sum():.2f}% of the scored mass")
for g in C2_SATS:
    k = PURITY_DATA['g'] == g
    m_sc = (PURITY_DATA['m0'] * (k & _scored)).sum()
    m_bad = (PURITY_DATA['m0'] * (k & _bad)).sum()
    print(f'{g:>8}: {int((k & _scored).sum()):6,} scored births, {int((k & _bad).sum()):5,} contaminated'
          + (f'  (purity of the whole birth set {1 - m_bad / m_sc:.3f})' if m_sc > 0 else ''))

In [ ]:
# Plotting  ·  Comp02 figure: the orbit panel coloured by birth-set purity, total or differential (standalone: PURITY and the C2_* settings)

from matplotlib.ticker import FixedLocator, NullFormatter

# ---- geometry, inches ----------------------------------------------------------------
pw, ph = C2_PANEL['width'], C2_PANEL['height']
cb, mg = C2_CBAR, C2_MARGIN
cb_len = cb['length'] * ph
cb_dy  = {'bottom': 0.0, 'center': (ph - cb_len) / 2, 'top': ph - cb_len}[cb['align']]
boxes = {
    'orbit':       (mg['left'], mg['bottom'], pw, ph),
    'cbar purity': (mg['left'] + pw + cb['pad'], mg['bottom'] + cb_dy, cb['width'], cb_len),
}
W = mg['left'] + pw + cb['pad'] + cb['width'] + mg['right']
H = mg['bottom'] + ph + mg['top']

S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}


# ---- colours: birth-set purity, total or differential (PURITY['mode']) ------------------
pu = PURITY
P_DATA = SD.birth_purity(R, pu['lag_myr'], pu['n_min'], pu['host_split'], pu['fit'])     # in memory after the first time
cmap_p = SD.colormap(pu['cmap'])
norm_p = Normalize(pu['vmin'], pu['vmax'], clip=True)
ORBITS2 = SD.orbits(R, C2_SATS)
PURITY_LINES = SD.purity_series(R, P_DATA, list(C2_SATS) + list(C2_BG_IDS), pu['K'], pu['level'], pu['mode'])


def c2_line(ax, tt, rr, values, lw, sunk_lw, alpha=1.0, zorder=3, capstyle='round', rasterized=False):
    # one orbit r/R_vir(t), coloured by its purity ('total': interpolated in time; 'differential': each
    # segment takes the value of the snapshot interval it lies in), thinner after its sinking time
    if values is not None and values.get('step'):
        t_mid = 0.5 * (tt[:-1] + tt[1:])
        k = np.searchsorted(values['time'], t_mid, side='left')       # t_mid in (time[k-1], time[k]]
        v_seg = np.where(k < values['time'].size, values['fret'][np.minimum(k, values['time'].size - 1)], np.nan)
    else:
        v = SD.fret_along(values, tt)
        v_seg = 0.5 * (v[:-1] + v[1:])
    defined = np.isfinite(v_seg)
    pts  = np.stack([tt, rr], axis=1)
    seg  = np.stack([pts[:-1], pts[1:]], axis=1)
    good = np.all(np.isfinite(seg), axis=(1, 2))
    if C2_LINES['drop_undefined']:
        good &= defined
    widths = np.full(tt.size - 1, lw)
    t_sink = SD.sinking_time(tt, rr, C2_LINES['sink_r'])
    if t_sink is not None:
        widths[tt[:-1] >= t_sink] = sunk_lw
    colours = cmap_p(norm_p(np.nan_to_num(v_seg)))
    colours[~defined] = to_rgba(C2_LINES['undefined_colour'])
    colours[:, 3] *= alpha
    ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=widths[good],
                                     capstyle=capstyle, zorder=zorder, rasterized=rasterized))


# ---- orbit panel: axes ------------------------------------------------------------------
ax_orb = axes['orbit']
t_hi = C2_AXES['t_range'][1] if C2_AXES['t_range'][1] is not None else C2_T_NOW
ax_orb.set_xlim(C2_AXES['t_range'][0], t_hi)
if C2_AXES['r_log']:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(C2_AXES['r_min'], C2_AXES['r_max'])
    ax_orb.yaxis.set_major_locator(FixedLocator(C2_AXES['r_ticks']))
    ax_orb.set_yticklabels([f'{r:g}' for r in C2_AXES['r_ticks']])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, C2_AXES['r_max'])
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- the selected satellites, the background and the R_vir line ------------------------
for g in C2_SATS:
    if g in ORBITS2:
        c2_line(ax_orb, *ORBITS2[g], PURITY_LINES.get(g), C2_LINES['lw'], C2_LINES['sunk_lw'],
                zorder=C2_ZORDER.get(g, 3))

if C2_BACKGROUND['show']:
    bg = C2_BACKGROUND
    bg_orbits = SD.orbits(R, C2_BG_IDS)
    for g in C2_BG_IDS:
        if g not in bg_orbits:
            continue
        if bg['colour'] == 'purity':
            c2_line(ax_orb, *bg_orbits[g], PURITY_LINES.get(g), bg['lw'], bg['sunk_lw'], alpha=bg['alpha'],
                    zorder=bg['zorder'], capstyle='butt', rasterized=bg['rasterized'])
        else:
            tt, rr = bg_orbits[g]
            pts = np.stack([tt, rr], axis=1)
            seg = np.stack([pts[:-1], pts[1:]], axis=1)
            good = np.all(np.isfinite(seg), axis=(1, 2))
            ax_orb.add_collection(LineCollection(seg[good], colors=[to_rgba(bg['colour'], bg['alpha'])],
                                                 linewidths=bg['lw'], capstyle='butt', zorder=bg['zorder'],
                                                 rasterized=bg['rasterized']))

ax_orb.axhline(1.0, **C2_LINES['rvir_line'])


# ---- redshift ticks on top -----------------------------------------------------------------
if C2_AXES['z_axis']:
    snaps   = sorted(C2_Z)
    zz      = np.array([C2_Z[s_] for s_ in snaps])
    tt      = np.array([C2_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in C2_AXES['z_ticks'] if zz.min() <= z <= zz.max()]
    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=C2_FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=C2_FONTS['label'], labelpad=2)


# ---- colourbar ---------------------------------------------------------------------------
cax = axes['cbar purity']
S.colorbar(fig, ScalarMappable(norm=norm_p, cmap=cmap_p), cax=cax, label=pu['cbar_label'][pu['mode']], ends=False)
if pu['cbar_ends']:
    gap = C2_NOTE_GAP / boxes['cbar purity'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=C2_FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, pu['cbar_ends'][0], va='bottom', **end_style)
    cax.text(0.5, -gap, pu['cbar_ends'][1], va='top', **end_style)


# ---- text sizes ------------------------------------------------------------------------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=C2_FONTS['ticks'])
    ax.xaxis.label.set_size(C2_FONTS['label'])
    ax.yaxis.label.set_size(C2_FONTS['label'])


# ---- show (and save if SAVE) -----------------------------------------------------------------
SD.show(fig, R.stem(f"{pu['name']}_{pu['mode']}_lag{pu['lag_myr']:g}{'_rr' if pu['fit'] == 'rr' else ''}_{C2_SET_NAME}"), SAVE)

## Comp02b: greedy against roadrunner (birth sets)

The Comp02 orbit panel, now comparing greedy's birth assignment with roadrunner's. Greedy's birth galaxy of a star: the galaxy that binds it most at its first snapshot (the host only where its energies are stored, inside R_vir). Two comparisons (`C2B['compare']`), each `total` or `differential` as `PURITY['mode']`:

- **reverse**: of the stars greedy gives to a satellite at birth, the share roadrunner also gives to it (initial masses): 1 = greedy adds nothing foreign to the satellite's birth set.
- **ratio**: purity of greedy's birth set / purity of roadrunner's, both by Comp02's likelihood test (same `PURITY` lag, K, level, n_min, host_split, fit): > 1 greedy's set looks purer, < 1 roadrunner's does.

Everything else (satellites, panel, axes, lines, background, fonts) comes from the Comp02 parameters; needs the Comp02 cells above to have run.

In [ ]:
# Plotting  ·  Comp02b parameters: what is compared and its colours (all the rest: PURITY and the C2_* settings, Comp02)
PURITY["mode"] = "total"

C2B = dict(
    compare    = 'ratio',       # 'reverse': of the stars greedy gives to a satellite at birth, the share roadrunner
                                  #   also gives to it; 'ratio': purity of greedy's birth set / of roadrunner's (same
                                  #   test and settings as Comp02); both 'total' or 'differential' as PURITY['mode']
    cmap       = dict(reverse='cmyt.arbre-0.9', ratio='iceburn-custom'),   # per comparison: any matplotlib, cmr.*, cmyt.*
                                  # name ('ratio': diverging, dark at 1 so equal stays visible on white)
    vrange     = dict(reverse=(0.9, 1.0), ratio=(0.98, 1.02)),   # per comparison: colour range (values beyond: end colours)
    cbar_label = dict(reverse="greedy's birth set: share also roadrunner's",
                      ratio=r'purity$_{\rm greedy}$ / purity$_{\rm rr}$'),
    cbar_ends  = dict(reverse=('agree', 'greedy adds'), ratio=('greedy purer', 'rr purer')),   # (top, bottom); None: none
    name       = 'comp02b',       # file name <name>_<compare>_<mode>_lag<lag_myr>_<C2_SET_NAME>
)

In [ ]:
# Processing  ·  Comp02b (needs the Comp02 cells above)

# greedy's birth galaxies come with the newborn fates (~2 min the first time); the purity pass scores them too
PURITY_DATA = SD.birth_purity(R, PURITY['lag_myr'], PURITY['n_min'], PURITY['host_split'], PURITY['fit'])

# a summary over each drawn satellite's whole life (initial masses)
_m = PURITY_DATA['m0']
_sr, _br = BP.flags(PURITY_DATA, PURITY['K'], PURITY['level'], 'rr')
_sg, _bg = BP.flags(PURITY_DATA, PURITY['K'], PURITY['level'], 'greedy')
print(f"fit {PURITY['fit']!r}, lag {PURITY['lag_myr']} Myr")
for g in C2_SATS:
    k_rr, k_gr = PURITY_DATA['g'] == g, PURITY_DATA['g_greedy'] == g
    m_gr = _m[k_gr].sum()
    share = _m[k_gr & k_rr].sum() / m_gr if m_gr > 0 else np.nan
    p_rr = 1 - _m[k_rr & _br].sum() / _m[k_rr & _sr].sum() if (_m[k_rr & _sr].sum() > 0) else np.nan
    p_gr = 1 - _m[k_gr & _bg].sum() / _m[k_gr & _sg].sum() if (_m[k_gr & _sg].sum() > 0) else np.nan
    print(f'{g:>8}: births rr {_m[k_rr].sum():.3g}, greedy {m_gr:.3g} Msun | greedy set also rr: {share:.3f} | '
          f'purity rr {p_rr:.3f}, greedy {p_gr:.3f}')

In [ ]:
# Plotting  ·  Comp02b figure: greedy against roadrunner on the orbit panel (C2B['compare']; the rest: PURITY and the C2_* settings)

from matplotlib.ticker import FixedLocator, NullFormatter

# ---- geometry, inches ----------------------------------------------------------------
pw, ph = C2_PANEL['width'], C2_PANEL['height']
cb, mg = C2_CBAR, C2_MARGIN
cb_len = cb['length'] * ph
cb_dy  = {'bottom': 0.0, 'center': (ph - cb_len) / 2, 'top': ph - cb_len}[cb['align']]
boxes = {
    'orbit':       (mg['left'], mg['bottom'], pw, ph),
    'cbar purity': (mg['left'] + pw + cb['pad'], mg['bottom'] + cb_dy, cb['width'], cb_len),
}
W = mg['left'] + pw + cb['pad'] + cb['width'] + mg['right']
H = mg['bottom'] + ph + mg['top']

S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}


# ---- colours: the comparison (C2B['compare']), total or differential (PURITY['mode']) ----
pu = PURITY
P_DATA = SD.birth_purity(R, pu['lag_myr'], pu['n_min'], pu['host_split'], pu['fit'])     # in memory after the first time
cmp = C2B['compare']
cmap_p = SD.colormap(C2B['cmap'][cmp])
norm_p = Normalize(*C2B['vrange'][cmp], clip=True)
ORBITS2 = SD.orbits(R, C2_SATS)
PURITY_LINES = SD.compare_series(R, P_DATA, list(C2_SATS) + list(C2_BG_IDS), cmp, pu['K'], pu['level'], pu['mode'])


def c2_line(ax, tt, rr, values, lw, sunk_lw, alpha=1.0, zorder=3, capstyle='round', rasterized=False):
    # one orbit r/R_vir(t), coloured by its purity ('total': interpolated in time; 'differential': each
    # segment takes the value of the snapshot interval it lies in), thinner after its sinking time
    if values is not None and values.get('step'):
        t_mid = 0.5 * (tt[:-1] + tt[1:])
        k = np.searchsorted(values['time'], t_mid, side='left')       # t_mid in (time[k-1], time[k]]
        v_seg = np.where(k < values['time'].size, values['fret'][np.minimum(k, values['time'].size - 1)], np.nan)
    else:
        v = SD.fret_along(values, tt)
        v_seg = 0.5 * (v[:-1] + v[1:])
    defined = np.isfinite(v_seg)
    pts  = np.stack([tt, rr], axis=1)
    seg  = np.stack([pts[:-1], pts[1:]], axis=1)
    good = np.all(np.isfinite(seg), axis=(1, 2))
    if C2_LINES['drop_undefined']:
        good &= defined
    widths = np.full(tt.size - 1, lw)
    t_sink = SD.sinking_time(tt, rr, C2_LINES['sink_r'])
    if t_sink is not None:
        widths[tt[:-1] >= t_sink] = sunk_lw
    colours = cmap_p(norm_p(np.nan_to_num(v_seg)))
    colours[~defined] = to_rgba(C2_LINES['undefined_colour'])
    colours[:, 3] *= alpha
    ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=widths[good],
                                     capstyle=capstyle, zorder=zorder, rasterized=rasterized))


# ---- orbit panel: axes ------------------------------------------------------------------
ax_orb = axes['orbit']
t_hi = C2_AXES['t_range'][1] if C2_AXES['t_range'][1] is not None else C2_T_NOW
ax_orb.set_xlim(C2_AXES['t_range'][0], t_hi)
if C2_AXES['r_log']:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(C2_AXES['r_min'], C2_AXES['r_max'])
    ax_orb.yaxis.set_major_locator(FixedLocator(C2_AXES['r_ticks']))
    ax_orb.set_yticklabels([f'{r:g}' for r in C2_AXES['r_ticks']])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, C2_AXES['r_max'])
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- the selected satellites, the background and the R_vir line ------------------------
for g in C2_SATS:
    if g in ORBITS2:
        c2_line(ax_orb, *ORBITS2[g], PURITY_LINES.get(g), C2_LINES['lw'], C2_LINES['sunk_lw'],
                zorder=C2_ZORDER.get(g, 3))

if C2_BACKGROUND['show']:
    bg = C2_BACKGROUND
    bg_orbits = SD.orbits(R, C2_BG_IDS)
    for g in C2_BG_IDS:
        if g not in bg_orbits:
            continue
        if bg['colour'] == 'purity':
            c2_line(ax_orb, *bg_orbits[g], PURITY_LINES.get(g), bg['lw'], bg['sunk_lw'], alpha=bg['alpha'],
                    zorder=bg['zorder'], capstyle='butt', rasterized=bg['rasterized'])
        else:
            tt, rr = bg_orbits[g]
            pts = np.stack([tt, rr], axis=1)
            seg = np.stack([pts[:-1], pts[1:]], axis=1)
            good = np.all(np.isfinite(seg), axis=(1, 2))
            ax_orb.add_collection(LineCollection(seg[good], colors=[to_rgba(bg['colour'], bg['alpha'])],
                                                 linewidths=bg['lw'], capstyle='butt', zorder=bg['zorder'],
                                                 rasterized=bg['rasterized']))

ax_orb.axhline(1.0, **C2_LINES['rvir_line'])


# ---- redshift ticks on top -----------------------------------------------------------------
if C2_AXES['z_axis']:
    snaps   = sorted(C2_Z)
    zz      = np.array([C2_Z[s_] for s_ in snaps])
    tt      = np.array([C2_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in C2_AXES['z_ticks'] if zz.min() <= z <= zz.max()]
    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=C2_FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=C2_FONTS['label'], labelpad=2)


# ---- colourbar ---------------------------------------------------------------------------
cax = axes['cbar purity']
S.colorbar(fig, ScalarMappable(norm=norm_p, cmap=cmap_p), cax=cax, label=C2B['cbar_label'][cmp], ends=False)
ends_b = C2B['cbar_ends'][cmp] if C2B['cbar_ends'] else None
if ends_b:
    gap = C2_NOTE_GAP / boxes['cbar purity'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=C2_FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, ends_b[0], va='bottom', **end_style)
    cax.text(0.5, -gap, ends_b[1], va='top', **end_style)


# ---- text sizes ------------------------------------------------------------------------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=C2_FONTS['ticks'])
    ax.xaxis.label.set_size(C2_FONTS['label'])
    ax.yaxis.label.set_size(C2_FONTS['label'])


# ---- show (and save if SAVE) -----------------------------------------------------------------
SD.show(fig, R.stem(f"{C2B['name']}_{cmp}_{pu['mode']}_lag{pu['lag_myr']:g}{'_rr' if pu['fit'] == 'rr' else ''}_{C2_SET_NAME}"), SAVE)

## Comp03: birth-set completeness on the orbit panel (standalone)

Purity read the other way round. Purity: of the stars born with a galaxy's label, how many sit astray from its blob in position and velocity. Completeness: of the stars born *in* a galaxy's blob, how many carry its label. Same fits as Comp02 (6D normals per galaxy and snapshot, the host as two). For every new star, its likeliest galaxy whatever its labels: it **belongs** to that galaxy when it is more likely there than in any other by a factor > K *and* lies within that galaxy's `level` contour (tidal-tail stars, fitting no blob well, belong to none and are not counted). A star that belongs to g but whose birth galaxy (`COMPLETENESS['label']`: roadrunner's, or greedy's most-bound one) is another is **missing** from g.

Line colour (`COMPLETENESS['mode']`): **total**, 1 − M(missing) / M(belonging) over the stars born in g's blob by then (initial masses); or **differential**, the same over the stars born since the previous snapshot only (none belonging in that interval: the undefined colour, no interpolation across). Grey before the first one.

**Standalone**: needs only the two Loading cells. Its own parameters (`COMPLETENESS` and the `C3_*` settings, copies of the Comp02 ones) never touch the other figures'. `C3_UNIFORM`: every satellite drawn alike (none highlighted), so all can be compared.

In [ ]:
# Plotting  ·  Comp03 parameters: birth-set completeness, and everything the figure needs (standalone; copies of the Comp02 values)

# ---- the completeness test (the same fits and settings as Comp02's purity) ----------------
COMPLETENESS = dict(
    lag_myr    = 0,               # judge each star at its first snapshot (0), or this many Myr after its birth; ~3 min per new lag
    K          = 100,             # a star belongs to its likeliest galaxy if more likely there than anywhere else by more
    level      = 0.999,           # than this factor, and lies within this contour of it (chi2, 6 dof)
    n_min      = 20,              # galaxies with fewer member stars get no fit
    host_split = 15.0,            # kpc: the host as two normals, inside / outside this radius (None: one)
    fit        = 'refit',         # 'refit': normals fitted here to each galaxy's members (newborns left out);
                                  # 'rr': roadrunner's own stored normals (not independent: see Comp02)
    label      = 'rr',            # whose birth galaxies are judged: 'rr' (roadrunner's birth map) or 'greedy' (the
                                  # galaxy binding the star most at its first snapshot)
    cmap       = 'cmyt.arbre-0.9',   # any matplotlib, cmr.* or cmyt.* name, octarine_tuned, or 'name-T'
    vmin       = 0.99,            # completeness at the bottom of the colour scale (lower: the lowest colour)
    vmax       = 1.0,             # ... and at the top (1: complete)
    mode       = 'differential',  # 'total': completeness of the whole birth set so far (cumulative);
                                  # 'differential': of the stars born since the previous snapshot only
                                  # (none belonging there: the undefined colour)
    cbar_label = dict(total=r'birth-set completeness', differential=r'completeness of the new stars'),
    cbar_ends  = ('complete', 'missing'),   # notes beyond the colourbar's ends, (top, bottom); None: none
    name       = 'comp03',        # file name <name>_<label>_<mode>_lag<lag_myr>_<C3_SET_NAME>
)

# ---- which satellites ------------------------------------------------------------
C3_SET_NAME = 'final'
C3_SATS     = [27155, 26681, 3674, 18557, 10779, 3567, 20813, 11508, 34253]
C3_ZORDER   = {27155: 100}        # drawing order of single satellites (higher on top); unlisted: 3, later in C3_SATS on top

# ---- panel, colourbar, margins, inches ------------------------------------------------
C3_PANEL = dict(width=7.2, height=2.6)
C3_CBAR  = dict(width=0.2, pad=0.05, length=1.0, align='center')   # length: fraction of the panel height
C3_MARGIN = dict(left=0.45, right=0.55, top=0.35, bottom=0.40)

# ---- axes ---------------------------------------------------------------------------
C3_AXES = dict(
    t_range = (0.0, None),        # Gyr; None: the last snapshot
    r_max   = 6,                  # top of the r/R_vir axis (star tables reach r_max + 0.5 R_vir)
    r_log   = True,               # logarithmic r/R_vir axis ...
    r_min   = 1e-2,               # ... from here
    r_ticks = (1e-2, 1e-1, 1),    # labelled r/R_vir values (log axis)
    z_axis  = False,              # redshift ticks on top ...
    z_ticks = (10, 5, 4, 3, 2, 1, 0),   # ... at these redshifts
)

# ---- lines ------------------------------------------------------------------------------
C3_LINES = dict(
    lw               = 1.4,
    sunk_lw          = 0.2,       # after the sinking time ...
    sink_r           = 0.1,       # ... from which it never again rises above this r/R_vir
    undefined_colour = '#c8c8c8', # no scored births (before the first one; 'differential': none in that interval)
    drop_undefined   = False,     # True: leave those stretches out instead of grey
    rvir_line        = dict(color='#52514e', lw=1, ls=(0, (3, 2)), zorder=1000),
)

# ---- background: every other satellite's orbit, faint, under the selected ones (no beads) ----
C3_BACKGROUND = dict(
    show       = True,
    which      = 'candidates',    # 'candidates': entering R_vir with >= min_npart star particles at their peak; 'all'
    min_npart  = 50,
    colour     = 'completeness',  # 'completeness': their own, on the colour scale; or one colour, e.g. '#9a9a9a'
    alpha      = 0.55,
    lw         = 0.2,
    sunk_lw    = 0.0,
    zorder     = 2,
    rasterized = True,
)

# ---- every satellite alike (C3_SATS and the background, none highlighted): all comparable -------
C3_UNIFORM = dict(
    show       = True,            # False: C3_SATS thick (C3_LINES, C3_ZORDER), the background faint (C3_BACKGROUND)
    lw         = 0.8,
    sunk_lw    = 0.2,             # after the sinking time (C3_LINES['sink_r'])
    alpha      = 1.0,
    rasterized = False,
)

# ---- text sizes, points -------------------------------------------------------------
C3_FONTS    = dict(label=14, ticks=9.5, notes=10)
C3_NOTE_GAP = 0.04                # inches between the colourbar's end and its note

In [ ]:
# Processing  ·  Comp03 (standalone: needs only the Loading cells)

from replay import birthpurity as BP

R.load()
R.set_r_keep(max(getattr(R, 'r_keep', 2.0), C3_AXES['r_max'] + 0.5))    # star tables reach beyond the panel's top

# ---- snapshot times ---------------------------------------------------------------------
_h = R.tree[R.tree['Sub_tree_id'] == R.host]
C3_TIME = dict(zip(_h['Snapshot'].astype(int).tolist(), _h['Time'].astype(float).tolist()))
C3_Z    = dict(zip(_h['Snapshot'].astype(int).tolist(), _h['Redshift'].astype(float).tolist()))
C3_T_NOW = C3_TIME[max(s for s in C3_TIME if s <= R.last)]

# ---- the likelihood test of every star with a birth galaxy (shared with Comp02: ~3 min the first time per lag,
#      n_min, host_split, fit) ----
C3_DATA = SD.birth_purity(R, COMPLETENESS['lag_myr'], COMPLETENESS['n_min'], COMPLETENESS['host_split'], COMPLETENESS['fit'])

# ---- orbits: the selected satellites and the background -----------------------------------
C3_ORBITS = SD.orbits(R, C3_SATS)
C3_BG_IDS = (SD.background_ids(R, C3_BACKGROUND['which'], C3_BACKGROUND['min_npart'], exclude=C3_SATS)
             if C3_BACKGROUND['show'] else [])
SD.orbits(R, C3_BG_IDS)

# ---- summary (K and level only change who belongs where: instant) ---------------------------
_scored, _member = BP.members(C3_DATA, COMPLETENESS['K'], COMPLETENESS['level'])
_m = C3_DATA['m0']
_lab = C3_DATA['g'] if COMPLETENESS['label'] == 'rr' else C3_DATA['g_greedy']
_miss = _member & (_lab != C3_DATA['top'])
print(f"fit {COMPLETENESS['fit']!r}, lag {COMPLETENESS['lag_myr']} Myr, label {COMPLETENESS['label']!r}: "
      f"{_member.sum():,} of {_scored.size:,} births belong to a galaxy; missing {100 * (_m * _miss).sum() / (_m * _member).sum():.2f}% "
      f"of that mass")
for g in C3_SATS:
    k = _member & (C3_DATA['top'] == g)
    m_in, m_miss = _m[k].sum(), _m[k & _miss].sum()
    print(f'{g:>8}: {int(k.sum()):6,} births belong to it, {int((k & _miss).sum()):5,} missing'
          + (f'  (completeness of the whole birth set {1 - m_miss / m_in:.3f})' if m_in > 0 else ''))

In [ ]:
# Plotting  ·  Comp03 figure: the orbit panel coloured by birth-set completeness, total or differential (standalone: COMPLETENESS and the C3_* settings)

from matplotlib.ticker import FixedLocator, NullFormatter

# ---- geometry, inches ----------------------------------------------------------------
pw, ph = C3_PANEL['width'], C3_PANEL['height']
cb, mg = C3_CBAR, C3_MARGIN
cb_len = cb['length'] * ph
cb_dy  = {'bottom': 0.0, 'center': (ph - cb_len) / 2, 'top': ph - cb_len}[cb['align']]
boxes = {
    'orbit':       (mg['left'], mg['bottom'], pw, ph),
    'cbar completeness': (mg['left'] + pw + cb['pad'], mg['bottom'] + cb_dy, cb['width'], cb_len),
}
W = mg['left'] + pw + cb['pad'] + cb['width'] + mg['right']
H = mg['bottom'] + ph + mg['top']

S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}


# ---- colours: birth-set completeness, total or differential (COMPLETENESS['mode']) --------
pu = COMPLETENESS
C3_P = SD.birth_purity(R, pu['lag_myr'], pu['n_min'], pu['host_split'], pu['fit'])     # in memory after the first time
cmap_p = SD.colormap(pu['cmap'])
norm_p = Normalize(pu['vmin'], pu['vmax'], clip=True)
ORBITS3 = SD.orbits(R, C3_SATS)
COMPL_LINES = SD.completeness_series(R, C3_P, list(C3_SATS) + list(C3_BG_IDS), pu['K'], pu['level'], pu['mode'],
                                     pu['label'])


def c3_line(ax, tt, rr, values, lw, sunk_lw, alpha=1.0, zorder=3, capstyle='round', rasterized=False):
    # one orbit r/R_vir(t), coloured by its completeness ('total': interpolated in time; 'differential': each
    # segment takes the value of the snapshot interval it lies in), thinner after its sinking time
    if values is not None and values.get('step'):
        t_mid = 0.5 * (tt[:-1] + tt[1:])
        k = np.searchsorted(values['time'], t_mid, side='left')       # t_mid in (time[k-1], time[k]]
        v_seg = np.where(k < values['time'].size, values['fret'][np.minimum(k, values['time'].size - 1)], np.nan)
    else:
        v = SD.fret_along(values, tt)
        v_seg = 0.5 * (v[:-1] + v[1:])
    defined = np.isfinite(v_seg)
    pts  = np.stack([tt, rr], axis=1)
    seg  = np.stack([pts[:-1], pts[1:]], axis=1)
    good = np.all(np.isfinite(seg), axis=(1, 2))
    if C3_LINES['drop_undefined']:
        good &= defined
    widths = np.full(tt.size - 1, lw)
    t_sink = SD.sinking_time(tt, rr, C3_LINES['sink_r'])
    if t_sink is not None:
        widths[tt[:-1] >= t_sink] = sunk_lw
    colours = cmap_p(norm_p(np.nan_to_num(v_seg)))
    colours[~defined] = to_rgba(C3_LINES['undefined_colour'])
    colours[:, 3] *= alpha
    ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=widths[good],
                                     capstyle=capstyle, zorder=zorder, rasterized=rasterized))


# ---- orbit panel: axes ------------------------------------------------------------------
ax_orb = axes['orbit']
t_hi = C3_AXES['t_range'][1] if C3_AXES['t_range'][1] is not None else C3_T_NOW
ax_orb.set_xlim(C3_AXES['t_range'][0], t_hi)
if C3_AXES['r_log']:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(C3_AXES['r_min'], C3_AXES['r_max'])
    ax_orb.yaxis.set_major_locator(FixedLocator(C3_AXES['r_ticks']))
    ax_orb.set_yticklabels([f'{r:g}' for r in C3_AXES['r_ticks']])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, C3_AXES['r_max'])
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- the satellites (all alike: C3_UNIFORM; or the selected ones over a faint background) and the R_vir line ----
if C3_UNIFORM['show']:
    un = C3_UNIFORM
    all_ids = list(C3_SATS) + [g for g in C3_BG_IDS if g not in C3_SATS]
    all_orbits = SD.orbits(R, all_ids)
    for g in all_ids:
        if g in all_orbits:
            c3_line(ax_orb, *all_orbits[g], COMPL_LINES.get(g), un['lw'], un['sunk_lw'], alpha=un['alpha'],
                    rasterized=un['rasterized'])
else:
    for g in C3_SATS:
        if g in ORBITS3:
            c3_line(ax_orb, *ORBITS3[g], COMPL_LINES.get(g), C3_LINES['lw'], C3_LINES['sunk_lw'],
                    zorder=C3_ZORDER.get(g, 3))

    if C3_BACKGROUND['show']:
        bg = C3_BACKGROUND
        bg_orbits = SD.orbits(R, C3_BG_IDS)
        for g in C3_BG_IDS:
            if g not in bg_orbits:
                continue
            if bg['colour'] == 'completeness':
                c3_line(ax_orb, *bg_orbits[g], COMPL_LINES.get(g), bg['lw'], bg['sunk_lw'], alpha=bg['alpha'],
                        zorder=bg['zorder'], capstyle='butt', rasterized=bg['rasterized'])
            else:
                tt, rr = bg_orbits[g]
                pts = np.stack([tt, rr], axis=1)
                seg = np.stack([pts[:-1], pts[1:]], axis=1)
                good = np.all(np.isfinite(seg), axis=(1, 2))
                ax_orb.add_collection(LineCollection(seg[good], colors=[to_rgba(bg['colour'], bg['alpha'])],
                                                     linewidths=bg['lw'], capstyle='butt', zorder=bg['zorder'],
                                                     rasterized=bg['rasterized']))

ax_orb.axhline(1.0, **C3_LINES['rvir_line'])


# ---- redshift ticks on top -----------------------------------------------------------------
if C3_AXES['z_axis']:
    snaps   = sorted(C3_Z)
    zz      = np.array([C3_Z[s_] for s_ in snaps])
    tt      = np.array([C3_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in C3_AXES['z_ticks'] if zz.min() <= z <= zz.max()]
    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=C3_FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=C3_FONTS['label'], labelpad=2)


# ---- colourbar ---------------------------------------------------------------------------
cax = axes['cbar completeness']
S.colorbar(fig, ScalarMappable(norm=norm_p, cmap=cmap_p), cax=cax, label=pu['cbar_label'][pu['mode']], ends=False)
if pu['cbar_ends']:
    gap = C3_NOTE_GAP / boxes['cbar completeness'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=C3_FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, pu['cbar_ends'][0], va='bottom', **end_style)
    cax.text(0.5, -gap, pu['cbar_ends'][1], va='top', **end_style)


# ---- text sizes ------------------------------------------------------------------------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=C3_FONTS['ticks'])
    ax.xaxis.label.set_size(C3_FONTS['label'])
    ax.yaxis.label.set_size(C3_FONTS['label'])


# ---- show (and save if SAVE) -----------------------------------------------------------------
SD.show(fig, R.stem(f"{pu['name']}_{pu['label']}_{pu['mode']}_lag{pu['lag_myr']:g}{'_rr' if pu['fit'] == 'rr' else ''}_{C3_SET_NAME}"), SAVE)

## Comp03b: greedy against roadrunner, completeness side (birth sets)

The Comp03 orbit panel, comparing greedy's birth assignment with roadrunner's (greedy's birth galaxy: the one binding the star most at its first snapshot; the host only inside R_vir, where its energies are stored). Two comparisons (`C3B['compare']`), each `total` or `differential` as `COMPLETENESS['mode']`:

- **forward**: of the stars roadrunner gives to a satellite at birth, the share greedy also gives to it (initial masses): 1 = greedy loses none of them (Comp02b's *reverse*, read the other way; labels only, no fits).
- **ratio**: completeness of greedy's birth set / of roadrunner's, both by Comp03's likelihood test (same `COMPLETENESS` lag, K, level, n_min, host_split, fit): < 1 greedy misses more of the stars that belong to the satellite.

Everything else (satellites, panel, axes, lines, background, C3_UNIFORM, fonts) comes from the Comp03 parameters; needs the Comp03 cells above to have run.

In [ ]:
# Plotting  ·  Comp03b parameters: what is compared and its colours (all the rest: COMPLETENESS and the C3_* settings, Comp03)

C3B = dict(
    compare    = 'forward',       # 'forward': of the stars roadrunner gives to a satellite at birth, the share greedy also
                                  #   gives to it; 'ratio': completeness of greedy's birth set / of roadrunner's (same
                                  #   test and settings as Comp03); both 'total' or 'differential' as COMPLETENESS['mode']
    cmap       = dict(forward='cmyt.arbre-0.9', ratio='iceburn-custom'),   # per comparison: any matplotlib, cmr.*, cmyt.*
                                  # name ('ratio': diverging, dark at 1 so equal stays visible on white)
    vrange     = dict(forward=(0.9, 1.0), ratio=(0.98, 1.02)),   # per comparison: colour range (values beyond: end colours)
    cbar_label = dict(forward="roadrunner's birth set: share also greedy's",
                      ratio=r'completeness$_{\rm greedy}$ / completeness$_{\rm rr}$'),
    cbar_ends  = dict(forward=('agree', 'greedy loses'), ratio=('greedy more complete', 'rr more complete')),   # (top, bottom); None: none
    name       = 'comp03b',       # file name <name>_<compare>_<mode>_lag<lag_myr>_<C3_SET_NAME>
)

In [ ]:
# Processing  ·  Comp03b (needs the Comp03 cells above)

# greedy's birth galaxies come with the newborn fates (~2 min the first time); the likelihood pass scores them too
C3_DATA = SD.birth_purity(R, COMPLETENESS['lag_myr'], COMPLETENESS['n_min'], COMPLETENESS['host_split'], COMPLETENESS['fit'])

# a summary over each drawn satellite's whole life (initial masses)
_m = C3_DATA['m0']
_sc, _member = BP.members(C3_DATA, COMPLETENESS['K'], COMPLETENESS['level'])
print(f"fit {COMPLETENESS['fit']!r}, lag {COMPLETENESS['lag_myr']} Myr")
for g in C3_SATS:
    k_rr, k_gr = C3_DATA['g'] == g, C3_DATA['g_greedy'] == g
    k_in = _member & (C3_DATA['top'] == g)
    m_rr, m_in = _m[k_rr].sum(), _m[k_in].sum()
    share = _m[k_rr & k_gr].sum() / m_rr if m_rr > 0 else np.nan
    c_rr = _m[k_in & k_rr].sum() / m_in if m_in > 0 else np.nan
    c_gr = _m[k_in & k_gr].sum() / m_in if m_in > 0 else np.nan
    print(f'{g:>8}: births rr {m_rr:.3g}, greedy {_m[k_gr].sum():.3g} Msun | rr set also greedy: {share:.3f} | '
          f'completeness rr {c_rr:.3f}, greedy {c_gr:.3f}')

In [ ]:
# Plotting  ·  Comp03b figure: greedy against roadrunner on the orbit panel, completeness side (C3B['compare']; the rest: COMPLETENESS and the C3_* settings)

from matplotlib.ticker import FixedLocator, NullFormatter

# ---- geometry, inches ----------------------------------------------------------------
pw, ph = C3_PANEL['width'], C3_PANEL['height']
cb, mg = C3_CBAR, C3_MARGIN
cb_len = cb['length'] * ph
cb_dy  = {'bottom': 0.0, 'center': (ph - cb_len) / 2, 'top': ph - cb_len}[cb['align']]
boxes = {
    'orbit':       (mg['left'], mg['bottom'], pw, ph),
    'cbar completeness': (mg['left'] + pw + cb['pad'], mg['bottom'] + cb_dy, cb['width'], cb_len),
}
W = mg['left'] + pw + cb['pad'] + cb['width'] + mg['right']
H = mg['bottom'] + ph + mg['top']

S.use()
fig = plt.figure(figsize=(W, H))
axes = {name: fig.add_axes([x / W, y / H, w / W, h / H]) for name, (x, y, w, h) in boxes.items()}


# ---- colours: the comparison (C3B['compare']), total or differential (COMPLETENESS['mode']) ----
pu = COMPLETENESS
C3_P = SD.birth_purity(R, pu['lag_myr'], pu['n_min'], pu['host_split'], pu['fit'])     # in memory after the first time
cmp = C3B['compare']
cmap_p = SD.colormap(C3B['cmap'][cmp])
norm_p = Normalize(*C3B['vrange'][cmp], clip=True)
ORBITS3 = SD.orbits(R, C3_SATS)
COMPL_LINES = SD.compare_completeness(R, C3_P, list(C3_SATS) + list(C3_BG_IDS), cmp, pu['K'], pu['level'], pu['mode'])


def c3_line(ax, tt, rr, values, lw, sunk_lw, alpha=1.0, zorder=3, capstyle='round', rasterized=False):
    # one orbit r/R_vir(t), coloured by the comparison ('total': interpolated in time; 'differential': each
    # segment takes the value of the snapshot interval it lies in), thinner after its sinking time
    if values is not None and values.get('step'):
        t_mid = 0.5 * (tt[:-1] + tt[1:])
        k = np.searchsorted(values['time'], t_mid, side='left')       # t_mid in (time[k-1], time[k]]
        v_seg = np.where(k < values['time'].size, values['fret'][np.minimum(k, values['time'].size - 1)], np.nan)
    else:
        v = SD.fret_along(values, tt)
        v_seg = 0.5 * (v[:-1] + v[1:])
    defined = np.isfinite(v_seg)
    pts  = np.stack([tt, rr], axis=1)
    seg  = np.stack([pts[:-1], pts[1:]], axis=1)
    good = np.all(np.isfinite(seg), axis=(1, 2))
    if C3_LINES['drop_undefined']:
        good &= defined
    widths = np.full(tt.size - 1, lw)
    t_sink = SD.sinking_time(tt, rr, C3_LINES['sink_r'])
    if t_sink is not None:
        widths[tt[:-1] >= t_sink] = sunk_lw
    colours = cmap_p(norm_p(np.nan_to_num(v_seg)))
    colours[~defined] = to_rgba(C3_LINES['undefined_colour'])
    colours[:, 3] *= alpha
    ax.add_collection(LineCollection(seg[good], colors=colours[good], linewidths=widths[good],
                                     capstyle=capstyle, zorder=zorder, rasterized=rasterized))


# ---- orbit panel: axes ------------------------------------------------------------------
ax_orb = axes['orbit']
t_hi = C3_AXES['t_range'][1] if C3_AXES['t_range'][1] is not None else C3_T_NOW
ax_orb.set_xlim(C3_AXES['t_range'][0], t_hi)
if C3_AXES['r_log']:
    ax_orb.set_yscale('log')
    ax_orb.set_ylim(C3_AXES['r_min'], C3_AXES['r_max'])
    ax_orb.yaxis.set_major_locator(FixedLocator(C3_AXES['r_ticks']))
    ax_orb.set_yticklabels([f'{r:g}' for r in C3_AXES['r_ticks']])
    ax_orb.yaxis.set_minor_formatter(NullFormatter())
else:
    ax_orb.set_ylim(0.0, C3_AXES['r_max'])
ax_orb.set_xlabel('cosmic time [Gyr]')
ax_orb.set_ylabel(r'$r/R_{\rm vir}$')


# ---- the satellites (all alike: C3_UNIFORM; or the selected ones over a faint background) and the R_vir line ----
if C3_UNIFORM['show']:
    un = C3_UNIFORM
    all_ids = list(C3_SATS) + [g for g in C3_BG_IDS if g not in C3_SATS]
    all_orbits = SD.orbits(R, all_ids)
    for g in all_ids:
        if g in all_orbits:
            c3_line(ax_orb, *all_orbits[g], COMPL_LINES.get(g), un['lw'], un['sunk_lw'], alpha=un['alpha'],
                    rasterized=un['rasterized'])
else:
    for g in C3_SATS:
        if g in ORBITS3:
            c3_line(ax_orb, *ORBITS3[g], COMPL_LINES.get(g), C3_LINES['lw'], C3_LINES['sunk_lw'],
                    zorder=C3_ZORDER.get(g, 3))

    if C3_BACKGROUND['show']:
        bg = C3_BACKGROUND
        bg_orbits = SD.orbits(R, C3_BG_IDS)
        for g in C3_BG_IDS:
            if g not in bg_orbits:
                continue
            if bg['colour'] == 'completeness':
                c3_line(ax_orb, *bg_orbits[g], COMPL_LINES.get(g), bg['lw'], bg['sunk_lw'], alpha=bg['alpha'],
                        zorder=bg['zorder'], capstyle='butt', rasterized=bg['rasterized'])
            else:
                tt, rr = bg_orbits[g]
                pts = np.stack([tt, rr], axis=1)
                seg = np.stack([pts[:-1], pts[1:]], axis=1)
                good = np.all(np.isfinite(seg), axis=(1, 2))
                ax_orb.add_collection(LineCollection(seg[good], colors=[to_rgba(bg['colour'], bg['alpha'])],
                                                     linewidths=bg['lw'], capstyle='butt', zorder=bg['zorder'],
                                                     rasterized=bg['rasterized']))

ax_orb.axhline(1.0, **C3_LINES['rvir_line'])


# ---- redshift ticks on top -----------------------------------------------------------------
if C3_AXES['z_axis']:
    snaps   = sorted(C3_Z)
    zz      = np.array([C3_Z[s_] for s_ in snaps])
    tt      = np.array([C3_TIME[s_] for s_ in snaps])
    o       = np.argsort(zz)
    z_ticks = [z for z in C3_AXES['z_ticks'] if zz.min() <= z <= zz.max()]
    ax_orb.tick_params(axis='x', which='both', top=False)
    top = ax_orb.secondary_xaxis('top')
    top.set_xticks(np.interp(z_ticks, zz[o], tt[o]))
    top.set_xticklabels([f'{z:g}' for z in z_ticks])
    top.minorticks_off()
    top.tick_params(labelsize=C3_FONTS['ticks'])
    top.set_xlabel(r'$z$', fontsize=C3_FONTS['label'], labelpad=2)


# ---- colourbar ---------------------------------------------------------------------------
cax = axes['cbar completeness']
S.colorbar(fig, ScalarMappable(norm=norm_p, cmap=cmap_p), cax=cax, label=C3B['cbar_label'][cmp], ends=False)
ends_b = C3B['cbar_ends'][cmp] if C3B['cbar_ends'] else None
if ends_b:
    gap = C3_NOTE_GAP / boxes['cbar completeness'][3]
    end_style = dict(transform=cax.transAxes, ha='center', fontsize=C3_FONTS['notes'], color=S.INK2)
    cax.text(0.5, 1 + gap, ends_b[0], va='bottom', **end_style)
    cax.text(0.5, -gap, ends_b[1], va='top', **end_style)


# ---- text sizes ------------------------------------------------------------------------------
for ax in fig.axes:
    ax.tick_params(which='both', labelsize=C3_FONTS['ticks'])
    ax.xaxis.label.set_size(C3_FONTS['label'])
    ax.yaxis.label.set_size(C3_FONTS['label'])


# ---- show (and save if SAVE) -----------------------------------------------------------------
SD.show(fig, R.stem(f"{C3B['name']}_{cmp}_{pu['mode']}_lag{pu['lag_myr']:g}{'_rr' if pu['fit'] == 'rr' else ''}_{C3_SET_NAME}"), SAVE)